# ST1504 Deep Learning - CA2 Part A

# Background Research

### Conditional GANs and the DCGAN backbone

Generative Adversarial Networks work by training a Generator and a Discriminator against each other — the Generator tries to produce convincing fake images, the Discriminator tries to catch them, and this competition is what drives learning, rather than a direct pixel-matching loss.
The original formulation has no way to control *what* the Generator produces, so we use a **conditional** GAN, where both networks also receive a class label. This is the specific design choice that lets us answer the brief's requirement to generate images of a *specific* class — without it, the Generator would have no signal telling it which class to produce.

We build this on a standard convolutional Generator/Discriminator design (no fully-connected layers beyond the initial projection, BatchNormalization, LeakyReLU), which is well known to train more stably than earlier, fully-connected GAN designs. We also apply one-sided label smoothing to the Discriminator's real-image target (0.9 instead of 1.0) from the start, based on
the well-documented finding that this prevents the Discriminator from becoming overconfident too
early and starving the Generator of useful gradient signal — a decision we made proactively in
our baseline, before observing any problem, precisely because this is such a well-established
risk for this kind of model.

### TTUR (Two Time-Scale Update Rule)

A common GAN failure mode is one network overpowering the other during training. TTUR addresses
this by giving the Discriminator a slower learning rate than the Generator, so it updates more
conservatively.We tested this specifically because our own baseline's loss curve showed
exactly this kind of imbalance where Generator loss rising, Discriminator loss falling, over the
full 100 epochs therefore this wasn't a technique we tried because it's known to help GANs in
general, it was a targeted response to something we directly observed in our own results.

### Hyperparameter sensitivity (latent dimension, batch size)

GAN performance is known to be sensitive to batch size and latent dimensionality, but there is no reliable, universal "correct" value therefore the right choice depends on the specific dataset and
architecture, which is why a systematic search is the standard, expected approach rather than guessing. This is why we ran a manual grid search across both hyperparameters rather than assuming our initial values were already well-chosen.

Latent dimension is the size of the random noise vector fed into the Generator. A larger latent dimension gives the Generator more degrees of freedom to encode variation which potentially is useful for a dataset like CIFAR-10, where our EDA confirmed substantial intra-class visual variability across classes. However, a latent space that is too large can be harder to learn a meaningful structure across within a fixed training budget, since the Generator must organise a higher-dimensional space of inputs into coherent class-specific outputs. Too small, and it maylack the capacity to represent the full range of  visual variation present across ten very different categories.

**Batch size** controls how many images are processed in each gradient update step. Smaller batches mean more gradient update steps per epoch for the same amount of data (`steps_per_epoch = dataset_size / batch_size`), which can help the model converge more quickly, but each individual update is noisier since it is based on fewer examples. Larger batches give smoother, more stable gradient estimates at the cost of fewer total updates per epoch, potentially requiring proportionally more training time. Neither direction is universally
better and the optimal value depends on the specific model, dataset, and training budget, making empirical search the appropriate approach.

### UpSampling2D + Conv2D (checkerboard artifacts)

A well-documented GAN artifact is a regular, grid-like "checkerboard" pattern in generated images, caused by uneven overlap in transposed convolution layers which is the standard way GANs upsample feature maps from low to high resolution during generation.This uneven overlap occurs because, depending on the relationship between kernel size and stride, certain output pixels
receive contributions from more input positions than their neighbours, creating a systematic pattern in the output.this was identified as an inherent structural limitation of the transposed convolution operation itself, not a failure of training.

The proposed fix is to separate the upsampling and learning steps: first resize the feature map to the target resolution using a simple, non-learned operation (such as nearest-neighbour interpolation), then apply a standard convolution to learn the desired transformations at that resolution. Because the resize step distributes information evenly before any learned convolution is applied, the systematic overlap unevenness that causes checkerboard patterns is avoided by
construction. For a dataset like CIFAR-10 — where fine structural detail matters, and where our EDA identified that vehicle and animal classes have quite different structural properties therefore this kind of artifact-free upsampling could plausibly improve the Generator's ability to render clean, coherent edges and backgrounds, particularly for the simpler, more geometrically consistent classes such as vehicles and ships.

### Self-attention (SAGAN)

Standard convolutions only look at a small local neighbourhood of an image at a time, which can make it hard for a network to keep distant parts of an object consistent with each other.For example,ensuring that both sides of a generated animal's body are anatomically coherent, or that an object's parts are positioned plausibly relative to one another across the full image.
This is a fundamental limitation of local receptive fields: no matter how many convolutional layers are stacked, each layer only directly processes a bounded spatial region, meaning that long-range spatial relationships must be built up gradually and indirectly across many layers rather than modelled explicitly.

The Self-Attention GAN (SAGAN) was proposed to directly address this gap by
incorporating a self-attention mechanism into both the Generator and Discriminator. Self-attention allows every spatial location in a feature map to attend to every other location simultaneously, giving the network an explicit mechanism to model long-range dependencies rather than relying on depth alone to propagate spatial information across the image. This is particularly relevant for class-conditional generation on a dataset like CIFAR-10, which contains classes (such as animals) with high pose variability and complex, non-rigid spatial structure which is exactly the cases where local convolutions are most likely to produce spatially incoherent output. By adding self-attention at the 16×16 resolution stage, we give the network the ability to coordinate structure across the whole image at a scale where meaningful spatial relationships are already established but computational cost remains manageable.

### Model capacity (wider/deeper architecture)

A fundamental consideration in any deep learning model is whether the network has sufficient representational capacity to capture the complexity of the target data distribution. In the context of GANs, capacity refers primarily to the number of learnable parameters which is determined by the width (number of filters per layer) and depth (number of layers) of both the Generator
and Discriminator. A network with insufficient capacity may be unable to represent the full range of variation present in the training data, regardless of how well the training procedure itself is designed.

Brock, Donahue, and Simonyan (2018) demonstrated this directly in their BigGAN work on large-scale ImageNet generation: simply scaling up the number of parameters and batch size in an otherwise standard conditional GAN produced substantial, consistent improvements in both image quality (FID) and diversity (Inception Score), without requiring any new architectural mechanism or loss function. Their finding suggests that capacity itself is an important,
sometimes under-exploited variable in GAN development and that targeted mechanism-level changes should not be assumed to be more impactful than simply giving the model more room to learn.

For CIFAR-10 specifically, this matters because our EDA (Section 3A.9) identified substantial intra-class variability and feature-space overlap between certain classes — particularly among animal categories. A Generator with limited capacity may default to modelling the most common, average-looking representatives of each class, rather than capturing the full diversity
of poses, backgrounds, and appearances that each class genuinely contains. Increasing width and depth gives the model more representational room to capture this variability, which could plausibly benefit the harder, higher-variability classes (dog, cat, bird) more than the simpler,
more geometrically consistent ones (truck, ship) — a testable hypothesis that our per-class evaluation framework is well-positioned to verify.

### EMA (Exponential Moving Average) weight-averaging

GAN training is inherently noisy on a step-by-step basis, even when the overall training process is converging successfully. Because the Generator and Discriminator are continuousl  reacting to each other's updates, the Generator's weights at any single training step may reflect a temporary, noisy fluctuation rather than a stable, well-generalised state. This oscillatory behaviour is a well-documented property of adversarial training, distinct from the more severe divergence-type instabilities addressed by techniques like WGAN-GP.

Exponential Moving Average (EMA) weight-averaging addresses this by maintaining a second, "shadow" copy of the Generator's weights that is updated after every training step as a slow-moving weighted average of the live weights, following the rule:

`shadow_weight = decay × shadow_weight + (1 − decay) × live_weight`

With a high decay value (typically 0.999), the shadow changes only gradually and smoothly, filtering out step-to-step noise while still tracking the overall direction of improvement over training. Yazici et al. (2019) provided the first theoretical argument specifically in support of EMA for GANs, showing that while standard gradient descent in adversarial games frequently
fails to converge which instead cycling around the optimal solution — EMA exponentially shrinks the amplitude of these limit cycles, confining the shadow generator to the neighbourhood of the equilibrium point. Karras et al. (2018) and Brock et al. (2018) both incorporated EMA as a standard component of their state-of-the-art GAN training pipelines, reporting consistent improvements in final image quality from using the averaged weights for generation rather than the raw, live weights.

### Conditional WGAN-GP

The standard GAN training objective minimises the Jensen-Shannon divergence between the real and generated data distributions. A well-known limitation of this approach is that when the two distributions have limited overlap which is common early in training ,the gradient signal provided to the Generator can become weak or uninformative, contributing to training instability and mode collapse (where the Generator learns to produce only a limited subset of the possible outputs rather than the full diversity of the target distribution).

Arjovsky, Chintala, and Bottou (2017) proposed the Wasserstein GAN (WGAN) as an alternative, replacing the Jensen-Shannon divergence with the Wasserstein distance, which provides a smoother, more informative gradient signal even when the real and generated distributions are far apart. This requires the Discriminator (renamed the Critic in this formulation) to satisfy
a Lipschitz continuity constraint. Gulrajani et al. (2017) improved on the original weight-clipping approach to enforcing this constraint by introducing a gradient penalty term (WGAN-GP), which directly penalises the Critic's gradient norm deviating from 1 at interpolated points between real and generated samples, reporting substantially more stable training as a result.

For class-conditional generation on a diverse dataset like CIFAR-10, where the Generator must learn ten distinct visual distributions simultaneously, mode collapse is a genuine and relevant risk — particularly if certain classes are harder to generate than others, the Generator may
find it easier to focus on simpler classes and progressively neglect harder ones. WGAN-GP's theoretical grounding in providing a more stable, informative training signal makes it a well-motivated approach to guard against exactly this failure mode.

### Projection discriminator

The standard approach to class conditioning in a GAN Discriminator is to concatenate the class label typically as a learned embedding projected to a spatial map directly with the input image as an additional channel. While simple and effective, Miyato and Koyama (2018) argued that this concatenation approach is not the most theoretically well-motivated way to incorporate
class information into the Discriminator's decision.

They proposed an alternative: the projection discriminator, in which the Discriminator computes a final output score as the sum of an unconditional realism score and an inner product between the class label embedding and a learned projection of the Discriminator's pooled image features.
This formulation is derived from an explicit probabilistic model of the optimal conditional discriminator, and more directly ties the Discriminator's real/fake judgment to whether the image's features are consistent with the claimed class label — rather than simply appending label information as an extra input channel. The authors reported improved class-conditional generation quality on ImageNet compared to concatenation-based conditioning, suggesting that the mechanism through which class information is injected into the Discriminator can itself be a meaningful variable in conditional GAN design.

### 1. Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"  # Ensure Keras uses TensorFlow as its backend
import numpy as np  # Numerical operations and array handling
import tensorflow as tf  # Deep-learning framework
from tensorflow.keras import layers  # Neural-network building blocks
import matplotlib.pyplot as plt  # Data and training visualisation
import time  # Measure training duration
import random  # Python random-number generation
from IPython import display  # Refresh training outputs in the notebook
from scipy.stats import entropy, pearsonr, spearmanr  # Statistical and distribution measures
from scipy.linalg import sqrtm  # Matrix square root used in FID calculation
import os

SEED = 42  # Fixed seed for reproducible experiments
random.seed(SEED)  # Seed Python's random generator
np.random.seed(SEED)  # Seed NumPy's random generator
tf.random.set_seed(SEED)  # Seed TensorFlow's random generator

gpus = tf.config.experimental.list_physical_devices('GPU')  # Detect available GPUs
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)  # Allocate GPU memory gradually as needed

print("TensorFlow version:", tf.__version__)
print("GPUs available:", gpus)
print("Random seed set to:", SEED)

TensorFlow version: 2.20.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Random seed set to: 42


In [ ]:
from google.colab import drive
drive.mount("/content/drive")  # Connect persistent Google Drive storage to Colab

import os  # File and directory path management
import json  # Save experiment results in JSON format

DRIVE_ROOT = "/content/drive/MyDrive/ST1504_CA2_PartA"  # Main persistent project folder
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")  # Intermediate training weights
OUTPUT_ROOT = os.path.join(
    DRIVE_ROOT,
    "cifar10_generative_models"
)  # Final model weights, metrics and artifacts
PROGRESS_DIR = os.path.join(
    DRIVE_ROOT,
    "progress_snapshots"
)  # Generated-image previews saved during training

for directory in (
    DRIVE_ROOT,
    CHECKPOINT_DIR,
    OUTPUT_ROOT,
    PROGRESS_DIR
):
    os.makedirs(directory, exist_ok=True)  # Create each folder safely if it is missing

print("Drive root:", DRIVE_ROOT)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Output directory:", OUTPUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive/ST1504_CA2_PartA
Checkpoint directory: /content/drive/MyDrive/ST1504_CA2_PartA/checkpoints
Output directory: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models


In [ ]:
def save_final_gan_weights(model_name):
    model_directory = os.path.join(
        OUTPUT_ROOT,
        model_name
    )  # Give each experiment its own output folder

    os.makedirs(model_directory, exist_ok=True)  # Create the folder if it does not exist

    generator_path = os.path.join(
        model_directory,
        f"{model_name}_generator_final.weights.h5"
    )  # Define the final Generator weight path

    discriminator_path = os.path.join(
        model_directory,
        f"{model_name}_discriminator_final.weights.h5"
    )  # Define the corresponding Discriminator weight path

    generator.save_weights(generator_path)  # Save the current Generator weights
    discriminator.save_weights(discriminator_path)  # Save the current Discriminator weights

    print("Generator saved to:", generator_path)
    print("Discriminator saved to:", discriminator_path)

    if "results_log" in globals():  # Back up metrics when a results log is available
        results_path = os.path.join(
            model_directory,
            f"{model_name}_results_log.json"
        )  # Store the results beside the model weights

        with open(results_path, "w") as file:
            json.dump(
                results_log,
                file,
                indent=2,
                default=str
            )  # Save logged metrics in a readable JSON format

        print("Results saved to:", results_path)

## 2. Results logging

In [ ]:
results_log = {}  # Central dictionary for storing every model's results


def log_results(model_name, overall_fid=None, overall_kl=None, overall_kid=None,
                precision=None, recall=None, density=None, coverage=None,
                per_class_fid=None, per_class_kl=None,
                inception_score=None, class_cond_accuracy=None, epochs_trained=None, notes=""):
    results_log[model_name] = {  # Store all metrics under the model's unique name
        "overall_fid": overall_fid, "overall_kl": overall_kl, "overall_kid": overall_kid,
        "precision": precision, "recall": recall, "density": density, "coverage": coverage,
        "per_class_fid": per_class_fid, "per_class_kl": per_class_kl,
        "inception_score": inception_score, "class_cond_accuracy": class_cond_accuracy,
        "epochs_trained": epochs_trained, "notes": notes,
    }
    print(f"Logged results for '{model_name}'.")  # Confirm that the entry was recorded


def show_results_table():
    print(f"{'Model':22s} {'Epochs':7s} {'FID':9s} {'KID':9s} {'Prec':7s} {'Rec':7s} "
          f"{'Dens':7s} {'Cov':7s} {'IS':16s} {'ClassAcc':9s} Notes")  # Print comparison headings
    print("-" * 150)

    for name, r in results_log.items():  # Format every experiment consistently
        epochs = r['epochs_trained'] if r['epochs_trained'] is not None else "-"
        fid = f"{r['overall_fid']:.2f}" if r['overall_fid'] is not None else "-"
        kid = f"{r['overall_kid']:.4f}" if r['overall_kid'] is not None else "-"
        prec = f"{r['precision']:.3f}" if r['precision'] is not None else "-"
        rec = f"{r['recall']:.3f}" if r['recall'] is not None else "-"
        dens = f"{r['density']:.3f}" if r.get('density') is not None else "-"
        cov = f"{r['coverage']:.3f}" if r.get('coverage') is not None else "-"
        is_str = f"{r['inception_score'][0]:.2f}+/-{r['inception_score'][1]:.2f}" if r['inception_score'] else "-"
        class_acc = f"{r['class_cond_accuracy']:.1%}" if r['class_cond_accuracy'] is not None else "-"

        print(f"{name:22s} {str(epochs):7s} {fid:9s} {kid:9s} {prec:7s} {rec:7s} "
              f"{dens:7s} {cov:7s} {is_str:16s} {class_acc:9s} {r['notes']}")  # Display one model per row

## 3. Load CIFAR-10

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()  # Load the official CIFAR-10 train/test split
y_train = y_train.reshape(-1).astype(np.int32)  # Convert labels to a one-dimensional integer array
y_test = y_test.reshape(-1).astype(np.int32)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']  # Map class IDs to readable names
NUM_CLASSES = 10  # CIFAR CIFAR-10 contains ten classes

print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)
print("Pixel range:", x_train.min(), x_train.max())
assert x_train.shape == (50000, 32, 32, 3)  # Confirm the expected training-set structure
assert x_test.shape == (10000, 32, 32, 3)  # Confirm the expected test-set structure

# Combine both official splits only for descriptive EDA, never for training or evaluation.
x_all = np.concatenate([x_train, x_test], axis=0)
y_all = np.concatenate([y_train, y_test], axis=0)
print("\nCombined (EDA only):", x_all.shape, y_all.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1650s 10us/step
Train: (50000, 32, 32, 3) (50000,)
Test : (10000, 32, 32, 3) (10000,)
Pixel range: 0 255

Combined (EDA only): (60000, 32, 32, 3) (60000,)


## 3A. Exploratory Data Analysis

Exploratory data analysis was conducted before model development to identify the properties of CIFAR-10 that are relevant to conditional image generation. The purpose of this section is not merely to describe the dataset, but to translate the observations into decisions about preprocessing, model architecture, conditioning and evaluation.

For descriptive EDA, the original training and test partitions are temporarily combined into
`x_all` and `y_all` to describe the complete CIFAR-10 collection. This combined dataset is
used only for descriptive analysis. All generative models and the auxiliary classifier are
trained exclusively on the official training partition, while the official test partition remains
unseen during training and is reserved for final evaluation.

The analysis examines:

- dataset dimensions, data type and pixel range;
- balance across the ten semantic classes;
- within-class variation in object appearance, pose and background;
- overall and class-specific colour characteristics;
- spatial patterns captured by class means and variation maps;
- similarity between average class representations;
- the effect of removing colour through an RGB-versus-grayscale comparison; and
- class separability in a lower-dimensional PCA representation.

These findings are subsequently used to justify the generator output shape, discriminator input shape, image normalisation, conditional label sampling, RGB output configuration, grayscale comparison experiment and class-specific evaluation.


## 3A.1 Dataset Overview

In [ ]:
unique_classes, class_counts = np.unique(y_all, return_counts=True)  # Count images belonging to each class
for class_id, count in zip(unique_classes, class_counts):
    percentage = count / len(y_all) * 100  # Calculate each class's dataset proportion
    print(f"{class_id}: {CLASS_NAMES[class_id]:12s} - {count:5d} images ({percentage:.2f}%)")

plt.figure(figsize=(11, 5))
bars = plt.bar(CLASS_NAMES, class_counts)  # Visualise the class distribution
plt.title("Number of CIFAR-10 Images per Class (combined, EDA only)")
plt.xlabel("Class"); plt.ylabel("Number of images"); plt.xticks(rotation=45)
for bar, count in zip(bars, class_counts):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40, str(count), ha="center", fontsize=9)  # Label each bar with its count
plt.ylim(0, class_counts.max() * 1.1)  # Leave space above the bars for labels
plt.tight_layout()
plt.show()

0: airplane     -  6000 images (10.00%)
1: automobile   -  6000 images (10.00%)
2: bird         -  6000 images (10.00%)
3: cat          -  6000 images (10.00%)
4: deer         -  6000 images (10.00%)
5: dog          -  6000 images (10.00%)
6: frog         -  6000 images (10.00%)
7: horse        -  6000 images (10.00%)
8: ship         -  6000 images (10.00%)
9: truck        -  6000 images (10.00%)


<Figure size 1100x500 with 1 Axes>

#### What this result means

The bar chart and printed counts show exactly 6,000 images per class in the combined 60,000-image
set (5,000 from the training partition, 1,000 from the test partition, per class). This confirms
CIFAR-10 is a perfectly balanced dataset by construction - there is no class with more or fewer
examples than any other, unlike many real-world datasets where rare classes are underrepresented.
This also means any imbalance related failure mode like for example a model ignoring a rare class because it saw
too few examples of it simply cannot explain anything we observe later in this notebook.

#### Modelling decision implication

Because classes are exactly balanced, we do not need class-weighting, oversampling, or a weighted loss function anywhere in our GAN/VAE training. This is an important baseline assumption: it means that if we later observe one class (e.g. dog) performing substantially worse than others across our quantitative evaluation metrics, the difference is more likely to reflect the intrinsic characteristics of that class rather than dataset imbalance

### 3A.2 Random per-class sample grid

In [ ]:
SAMPLES_PER_CLASS = 8  # Display eight examples from each CIFAR-10 class
rng = np.random.default_rng(SEED)  # Make image selection reproducible

fig, axes = plt.subplots(NUM_CLASSES, SAMPLES_PER_CLASS, figsize=(14, 17))  # Arrange one class per row

for class_id in range(NUM_CLASSES):
    class_indices = np.where(y_all == class_id)[0]  # Locate all images from the current class
    selected = rng.choice(class_indices, size=SAMPLES_PER_CLASS, replace=False)  # Sample distinct images

    for column, image_index in enumerate(selected):
        axes[class_id, column].imshow(x_all[image_index])
        axes[class_id, column].set_xlabel(CLASS_NAMES[class_id], fontsize=8)  # Display the class below each image
        axes[class_id, column].set_xticks([])
        axes[class_id, column].set_yticks([])

    axes[class_id, 0].set_ylabel(
        CLASS_NAMES[class_id],
        fontsize=10,
        rotation=0,
        labelpad=45,
        va="center"
    )  # Label each row with its class name

plt.suptitle("Random CIFAR-10 Samples from Each Class", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

<Figure size 1400x1700 with 80 Axes>

#### What this result means

Visually scanning the grid, several patterns emerge. Vehicle classes (automobile,truck,ship,airplane) show relatively consistent silhouettes and viewing angles across samples — most images present the object as a clear, centred shape against a comparatively simple background. Animal classes (cat,dog,bird,horse,deer) show far more variation: the animal may be facing any direction,occupy a small or large fraction of the frame, and sit
against cluttered natural backgrounds sich as grass,foliage, indoor scenes. cat and dog in particular look visually similar to a human observer at this resolution — both are medium-sized, furred, similarly posed animals photographed at similar distances. At only 32x32 pixels, fine details such as (fur texture, facial features) are barely resolvable, which compresses the visual difference between already-similar classes even further.

#### Modelling decision implication

This qualitative observation motivates two concrete choices later in this notebook. First, our per-class evaluation strategy: we compute FID, KL-divergence, and class-conditioning accuracy separately for every class, because an overall/averaged score would completely hide a class-specific weakness like the cat/dog confusion risk we can already see by eye here. Second, it sets up a specific, falsifiable expectation we can test once we have real results: if visual similarity genuinely causes generation difficulty, we would expect cat and dog to both show elevated per-class FID and possibly cross-confusion in the class-conditioning accuracy breakdown, compared to visually distinct classes like automobile or ship.

### 3A.3 Overall RGB pixel distribution

In [ ]:
rng = np.random.default_rng(SEED)  # Make image sampling reproducible
sample_indices = rng.choice(len(x_all), size=10000, replace=False)  # Select a manageable EDA subset
pixel_sample = x_all[sample_indices]

red_pixels = pixel_sample[:, :, :, 0].reshape(-1)  # Flatten all red-channel intensities
green_pixels = pixel_sample[:, :, :, 1].reshape(-1)  # Flatten all green-channel intensities
blue_pixels = pixel_sample[:, :, :, 2].reshape(-1)  # Flatten all blue-channel intensities

plt.figure(figsize=(11, 5))
plt.hist(red_pixels, bins=50, alpha=0.45, label="Red", density=True)  # Plot normalised red-channel density
plt.hist(green_pixels, bins=50, alpha=0.45, label="Green", density=True)
plt.hist(blue_pixels, bins=50, alpha=0.45, label="Blue", density=True)
plt.title("Distribution of RGB Pixel Intensities")
plt.xlabel("Pixel intensity"); plt.ylabel("Density"); plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean red intensity   : {red_pixels.mean():.2f}")
print(f"Mean green intensity : {green_pixels.mean():.2f}")
print(f"Mean blue intensity  : {blue_pixels.mean():.2f}")  # Summarise the average colour balance

<Figure size 1100x500 with 1 Axes>

Mean red intensity   : 125.25
Mean green intensity : 123.09
Mean blue intensity  : 114.09


#### What this result means

The overlapping histograms show that red, green, and blue channel intensities follow broadly similar distributions across the whole dataset - no channel is dramatically brighter,darker,or more concentrated than the others.The printed mean intensities for R/G/B are all in a similar range (roughly 120-130 out of 255) which indicates the dataset as a whole is not colour-biased towards any particular hue at the aggregate level.This is an expected result for a dataset covering 10 very different real-world object categories (vehicles, animals, etc.) — any single class's colour bias (e.g. `frog` skewing green) gets averaged out once all classes are pooled together.
#### Modelling decision implication

This confirms that a single, dataset wide preprocessing choice — scaling all pixels to [-1, 1] via (x - 127.5) / 127.5 is appropriate,since there is no global colour-channel skew that would require separate per-channel normalisation or colour-correction before training. Had we found, for example, that blue pixels were systematically much dimmer than red/green across the whole dataset, we might have considered per-channel standardisation instead of a single shared scaling factor. This result is a necessary sanity check rather than a novel finding, but it directly justifies a specific line of code in our preprocessing pipeline.

### 3A.4 Per-class brightness, contrast and colourfulness analysis cell

In [ ]:
def calculate_colourfulness(images):
    images = images.astype(np.float32)  # Prevent overflow during channel calculations
    red, green, blue = images[:,:,:,0], images[:,:,:,1], images[:,:,:,2]  # Separate RGB channels
    rg = red - green  # Measure red–green colour variation
    yb = 0.5 * (red + green) - blue  # Measure yellow–blue colour variation
    rg_std, yb_std = np.std(rg), np.std(yb)  # Capture spread of colour differences
    rg_mean, yb_mean = np.mean(rg), np.mean(yb)  # Capture average colour differences
    return np.sqrt(rg_std**2 + yb_std**2) + 0.3 * np.sqrt(rg_mean**2 + yb_mean**2)  # Combine variation and average colour strength

class_statistics = {}  # Store descriptive statistics for every class
for class_id, class_name in enumerate(CLASS_NAMES):
    class_images = x_all[y_all == class_id].astype(np.float32)  # Select all images from the current class
    grayscale = 0.299*class_images[:,:,:,0] + 0.587*class_images[:,:,:,1] + 0.114*class_images[:,:,:,2]  # Convert RGB to perceived brightness
    brightness = grayscale.mean()  # Measure average image brightness
    contrast = grayscale.std()  # Measure variation between light and dark pixels
    pixel_variability = class_images.std()  # Measure overall RGB intensity variation
    colourfulness = calculate_colourfulness(class_images)  # Measure the strength and variety of colours
    class_statistics[class_name] = {"brightness": brightness, "contrast": contrast,
                                    "pixel_variability": pixel_variability, "colourfulness": colourfulness}  # Save class-level EDA results
    print(f"{class_name:12s} | Brightness={brightness:6.2f} | Contrast={contrast:6.2f} | "
          f"Pixel variability={pixel_variability:6.2f} | Colourfulness={colourfulness:6.2f}")

airplane     | Brightness=141.55 | Contrast= 60.60 | Pixel variability= 64.60 | Colourfulness= 51.36
automobile   | Brightness=116.96 | Contrast= 66.29 | Pixel variability= 68.84 | Colourfulness= 45.39
bird         | Brightness=123.31 | Contrast= 55.55 | Pixel variability= 59.38 | Colourfulness= 47.13
cat          | Brightness=118.12 | Contrast= 63.32 | Pixel variability= 65.62 | Colourfulness= 42.55
deer         | Brightness=116.32 | Contrast= 51.54 | Pixel variability= 54.90 | Colourfulness= 45.85
dog          | Brightness=120.18 | Contrast= 61.28 | Pixel variability= 63.83 | Colourfulness= 43.47
frog         | Brightness=111.66 | Contrast= 54.93 | Pixel variability= 58.41 | Colourfulness= 45.83
horse        | Brightness=122.39 | Contrast= 60.95 | Pixel variability= 63.53 | Colourfulness= 42.59
ship         | Brightness=132.17 | Contrast= 60.83 | Pixel variability= 63.37 | Colourfulness= 42.35
truck        | Brightness=124.81 | Contrast= 67.07 | Pixel variability= 69.55 | Colourfulne

In [ ]:
for metric_name, ylabel, title in [
    ("brightness", "Mean grayscale intensity", "Average Image Brightness by Class"),
    ("contrast", "Grayscale pixel standard deviation", "Average Pixel Contrast by Class"),
    ("colourfulness", "Colourfulness score", "Estimated Colourfulness by Class"),
]:
    values = [class_statistics[name][metric_name] for name in CLASS_NAMES]
    plt.figure(figsize=(11, 5))
    plt.bar(CLASS_NAMES, values)
    plt.title(title); plt.xlabel("Class"); plt.ylabel(ylabel); plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


<Figure size 1100x500 with 1 Axes>

<Figure size 1100x500 with 1 Axes>

<Figure size 1100x500 with 1 Axes>

#### What this result means

Brightness (mean grayscale intensity) varies across classes — for instance, classes photographed predominantly outdoors in daylight such as airplane or ship tend to skew brighter, while classes with darker fur/plumage or indoor settings such as dog or cat can skew darker. Contrast (grayscale standard deviation) captures how much light to dark variation exists within a typical image of that class - higher contrast suggests images with strong shadows, sharp object/background boundaries, or high-detail texture. Colourfulness (our red-green/yellow-blue channel-difference score) captures how far a class's images deviate from a neutral grey/brown palette - classes with consistently vivid, saturated colours (e.g. frog, with its green tones) score higher than classes dominated by muted, natural tones (e.g. deer against brown/green backgrounds).

#### Modelling decision implication

These statistics provide an interpretable description of each class's visual characteristics
before model training begins. Rather than being used as evaluation metrics themselves, they
serve as contextual information that may later help explain differences observed during the
final model evaluation. For example, classes with greater colour variation or stronger texture
contrast may present additional challenges for the generator when learning realistic image
distributions, potentially affecting image fidelity, diversity and class-conditioning
performance across individual classes.

### 3A.5 Per-class average images and spatial variation maps

In [ ]:
class_mean_images = {}  # Store the average image for each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))  # Arrange all ten classes in two rows

for class_id, class_name in enumerate(CLASS_NAMES):
    class_images = x_all[y_all == class_id].astype(np.float32)  # Select images from the current class
    mean_image = class_images.mean(axis=0)  # Average corresponding pixels across the class
    class_mean_images[class_name] = mean_image  # Preserve the result for later analysis
    axis = axes[class_id // 5, class_id % 5]  # Locate the correct subplot
    axis.imshow(mean_image.astype(np.uint8))
    axis.set_title(class_name); axis.axis("off")

plt.suptitle("Average CIFAR-10 Image for Each Class", fontsize=16)
plt.tight_layout()
plt.show()

<Figure size 1500x600 with 10 Axes>

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))  # Arrange variation maps for all ten classes

for class_id, class_name in enumerate(CLASS_NAMES):
    class_images = x_all[y_all == class_id].astype(np.float32)  # Select images from the current class
    variation_map = class_images.std(axis=0).mean(axis=2)  # Measure pixel variation across images and average over RGB channels
    axis = axes[class_id // 5, class_id % 5]  # Locate the correct subplot
    axis.imshow(variation_map, cmap="viridis")  # Brighter areas indicate greater spatial variation
    axis.set_title(class_name); axis.axis("off")

plt.suptitle("Spatial Pixel-Variation Maps by Class", fontsize=16)
plt.tight_layout()
plt.show()

<Figure size 1500x600 with 10 Axes>

#### What this result means

The per-class average images are noticeably blurry across all categories—an expected outcome rather than a data-quality issue. Because objects appear in varying positions, scales, and orientations, averaging many samples smears out fine details, leaving only coarse shared features like dominant color palettes (e.g., blue backgrounds for airplane and ship, earthy tones for animals). The spatial pixel-variation maps are far more informative: classes like automobile, truck, and ship exhibit structured, localized variance with consistent low-variance backgrounds and tight high-variance bands where the subject sits, whereas categories like dog, cat, bird, and horse display wide, diffuse variance spread across the frame due to diverse poses, scales, and backgrounds.

#### Modelling decision implication

Classes showing wide, diffuse spatial variation are expected to be harder for a fixed-capacity conditional GAN to model consistently, because the Generator must learn to map a single class label to a much broader range of spatial configurations. While these variation maps capture pixel-level positional footprint, our subsequent PCA feature distance analysis in Section 3A.9 evaluates high-level semantic diversity in feature space. Using spatial maps as a qualitative, visual precursor to that quantitative metric creates a complete picture—if the later, more rigorous PCA metric agrees with what we see visually here, it strengthens our confidence that spatial and structural diversity is a genuine driver of generation difficulty, not an artefact of this particular visualisation method.

### 3A.6 Per-class RGB composition analysis cell

In [ ]:
mean_rgb_by_class = np.zeros((NUM_CLASSES, 3))  # Store mean red, green and blue values per class

for class_id, class_name in enumerate(CLASS_NAMES):
    class_images = x_all[y_all == class_id].astype(np.float32)  # Select images from the current class
    mean_rgb = class_images.mean(axis=(0, 1, 2))  # Average each colour channel across all images and pixels
    mean_rgb_by_class[class_id] = mean_rgb
    print(f"{class_name:12s} | R={mean_rgb[0]:6.2f} | G={mean_rgb[1]:6.2f} | B={mean_rgb[2]:6.2f}")

positions = np.arange(NUM_CLASSES); bar_width = 0.25  # Define grouped-bar positions and width
plt.figure(figsize=(12, 5))
plt.bar(positions - bar_width, mean_rgb_by_class[:, 0], width=bar_width, label="Red")  # Compare red intensity
plt.bar(positions, mean_rgb_by_class[:, 1], width=bar_width, label="Green")  # Compare green intensity
plt.bar(positions + bar_width, mean_rgb_by_class[:, 2], width=bar_width, label="Blue")  # Compare blue intensity
plt.xticks(positions, CLASS_NAMES, rotation=45)
plt.xlabel("Class"); plt.ylabel("Mean pixel intensity")
plt.title("Average RGB Composition of Each CIFAR-10 Class")
plt.legend(); plt.tight_layout()
plt.show()

airplane     | R=134.39 | G=143.38 | B=150.90
automobile   | R=120.13 | G=115.91 | B=114.09
bird         | R=124.79 | G=125.47 | B=108.31
cat          | R=126.26 | G=116.34 | B=105.98
deer         | R=119.98 | G=118.38 | B= 96.12
dog          | R=127.88 | G=118.90 | B=106.58
frog         | R=120.07 | G=111.95 | B= 88.06
horse        | R=128.13 | G=122.52 | B=106.67
ship         | R=125.19 | G=133.94 | B=141.37
truck        | R=127.44 | G=123.97 | B=122.22


<Figure size 1200x500 with 1 Axes>

#### What this result means

The grouped bar chart shows each class's average red, green, and blue channel intensity.
`airplane` and `ship` are the only categories where blue is the dominant channel on average,
consistent with sky and sea backgrounds being common in those images. Most other classes show
red and green as the dominant, closely-matched channels, reflecting the prevalence of
brown/green natural tones (fur, foliage, earth) across animal and outdoor-scene classes. No
class shows an extreme, single-channel dominance — differences between classes are present but
modest in magnitude.

#### Modelling decision implication

Classes with a strongly consistent dominant channel (like `airplane`/`ship` skewing blue) give
the Generator a simpler, more predictable colour prior to learn — essentially, "this class is
usually blue-ish" is an easy statistical pattern to pick up. We would therefore expect our
grayscale-vs-colour discussion (Section 3A.8) to matter *less* for these specific classes, since
colour there is fairly predictable rather than carrying much unique distinguishing information.
For classes without a strong colour signature, colour is less likely to be doing heavy lifting in
class recognition, meaning shape/texture accuracy likely matters more for those classes'
generation quality.

### 3A.7 Similarity between class-average images analysis cell

In [ ]:
mean_image_vectors = np.array([class_mean_images[c].reshape(-1) for c in CLASS_NAMES])  # Flatten each class-average image
vector_norms = np.linalg.norm(mean_image_vectors, axis=1, keepdims=True)  # Calculate each vector's magnitude
normalised_vectors = mean_image_vectors / np.maximum(vector_norms, 1e-8)  # Normalise safely for cosine similarity
similarity_matrix = normalised_vectors @ normalised_vectors.T  # Compare average colour and spatial patterns between classes

plt.figure(figsize=(9, 8))
img = plt.imshow(similarity_matrix, cmap="viridis", vmin=0, vmax=1)  # Display pairwise class similarities
plt.colorbar(img, label="Cosine similarity")
plt.xticks(np.arange(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
plt.yticks(np.arange(NUM_CLASSES), CLASS_NAMES)
for row in range(NUM_CLASSES):
    for col in range(NUM_CLASSES):
        plt.text(col, row, f"{similarity_matrix[row, col]:.2f}", ha="center", va="center", fontsize=7)  # Annotate exact similarity values
plt.title("Similarity Between Class-Average Images")
plt.tight_layout()
plt.show()

class_pairs = []  # Store every unique pair of different classes
for i in range(NUM_CLASSES):
    for j in range(i + 1, NUM_CLASSES):  # Avoid duplicate and self-comparisons
        class_pairs.append((CLASS_NAMES[i], CLASS_NAMES[j], similarity_matrix[i, j]))
class_pairs = sorted(class_pairs, key=lambda x: x[2], reverse=True)  # Rank pairs from most to least similar

print("Most similar class-average image pairs:\n")
for a, b, sim in class_pairs[:10]:
    print(f"{a:12s} vs {b:12s}: {sim:.4f}")  # Display the ten most visually similar class averages

<Figure size 900x800 with 2 Axes>

Most similar class-average image pairs:

cat          vs dog         : 0.9981
bird         vs horse       : 0.9980
cat          vs deer        : 0.9980
deer         vs frog        : 0.9980
bird         vs deer        : 0.9979
bird         vs cat         : 0.9973
cat          vs frog        : 0.9972
deer         vs dog         : 0.9968
deer         vs horse       : 0.9966
cat          vs horse       : 0.9963


#### What this result means

The cosine-similarity matrix between class-average images is uniformly high (most values close
to 1.0) across nearly all class pairs — including pairs we would expect to look quite different,
like `automobile` and `frog`. This is because, as noted in Section 3A.5, every class-average
image is dominated by the same broad brown/green background tones once individual object detail
is blurred away by averaging. In other words, this particular method is measuring "do these two
classes share a similar overall colour cast," which turns out to be "yes, almost always," rather
than measuring anything about the objects themselves.

#### Modelling decision implication

Because this simple pixel-averaging approach produces an uninformative, uniformly-high
similarity matrix, we treat it as **inconclusive on its own** for predicting which classes will
be confused with each other or harder to generate. This negative/uninformative result is
important to report honestly rather than omit — it demonstrates that a naive analysis method can
fail to detect real class differences, and it directly motivates why we build a more
discriminative, per-image (not per-class-average) feature-space analysis in Section 3A.9, using
PCA and nearest-neighbour overlap instead of simple average-image comparison.

### 3A.8 RGB versus grayscale

In [ ]:
rng = np.random.default_rng(SEED)  # Make image selection reproducible
selected = [rng.choice(np.where(y_all == c)[0]) for c in range(NUM_CLASSES)]  # Select one image from each class
fig, axes = plt.subplots(NUM_CLASSES, 2, figsize=(6, 20))  # Place RGB and grayscale versions side by side

for row, idx in enumerate(selected):
    rgb_image = x_all[idx]  # Retrieve the selected RGB image
    gray = 0.299*rgb_image[:,:,0] + 0.587*rgb_image[:,:,1] + 0.114*rgb_image[:,:,2]  # Convert using perceptual channel weights
    axes[row, 0].imshow(rgb_image); axes[row, 0].set_title(f"{CLASS_NAMES[row]} -- RGB"); axes[row, 0].axis("off")
    axes[row, 1].imshow(gray, cmap="gray", vmin=0, vmax=255); axes[row, 1].set_title(f"{CLASS_NAMES[row]} -- Grayscale"); axes[row, 1].axis("off")

plt.suptitle("RGB and Grayscale CIFAR-10 Samples", fontsize=15)
plt.tight_layout()
plt.show()

<Figure size 600x2000 with 20 Axes>

In [ ]:
from sklearn.decomposition import PCA  # Reduce high-dimensional pixel features
from sklearn.metrics import silhouette_score  # Measure labelled class separation
from sklearn.preprocessing import StandardScaler  # Standardise features before PCA

rng = np.random.default_rng(SEED)  # Make sampling reproducible
SAMPLES_PER_CLASS_PCA = 300  # Use an equal, manageable sample from every class
selected_indices = np.concatenate([
    rng.choice(np.where(y_all == c)[0], size=SAMPLES_PER_CLASS_PCA, replace=False)
    for c in range(NUM_CLASSES)
])  # Create a class-balanced EDA subset

sample_images = x_all[selected_indices].astype(np.float32) / 255.0  # Scale sampled pixels to [0, 1]
sample_labels = y_all[selected_indices]
rgb_features = sample_images.reshape(len(sample_images), -1)  # Flatten RGB images into feature vectors
grayscale_images = 0.299*sample_images[:,:,:,0] + 0.587*sample_images[:,:,:,1] + 0.114*sample_images[:,:,:,2]  # Remove colour information
grayscale_features = grayscale_images.reshape(len(grayscale_images), -1)  # Flatten grayscale images

rgb_scaled = StandardScaler().fit_transform(rgb_features)  # Give RGB pixel features comparable scales
grayscale_scaled = StandardScaler().fit_transform(grayscale_features)
rgb_pca_features = PCA(n_components=50, random_state=SEED).fit_transform(rgb_scaled)  # Retain 50 principal RGB components
grayscale_pca_features = PCA(n_components=50, random_state=SEED).fit_transform(grayscale_scaled)

rgb_silhouette = silhouette_score(rgb_pca_features, sample_labels)  # Measure RGB class separation
grayscale_silhouette = silhouette_score(grayscale_pca_features, sample_labels)  # Measure separation without colour

print(f"RGB silhouette score       : {rgb_silhouette:.4f}")
print(f"Grayscale silhouette score : {grayscale_silhouette:.4f}")
print("\nHigher silhouette scores indicate stronger separation between labelled classes.")
print("Note: CIFAR-10 classes overlap heavily in raw pixel space, so low/negative scores are expected.")

RGB silhouette score       : -0.0557
Grayscale silhouette score : -0.0611

Higher silhouette scores indicate stronger separation between labelled classes.
Note: CIFAR-10 classes overlap heavily in raw pixel space, so low/negative scores are expected.


In [ ]:
rgb_pca_2d = PCA(
    n_components=2,
    random_state=42
).fit_transform(rgb_scaled)  # Reduce RGB features to two components for visualisation

plt.figure(figsize=(10, 8))

for class_id, class_name in enumerate(CLASS_NAMES):
    class_mask = sample_labels == class_id  # Select PCA points belonging to the current class

    plt.scatter(
        rgb_pca_2d[class_mask, 0],
        rgb_pca_2d[class_mask, 1],
        s=12,
        alpha=0.55,
        label=class_name
    )  # Plot each class separately to reveal overlap and clustering

plt.title("PCA Projection of RGB CIFAR-10 Images")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)  # Place the legend outside to avoid covering data points
plt.tight_layout()
plt.show()

<Figure size 1000x800 with 1 Axes>

In [ ]:
grayscale_pca_2d = PCA(
    n_components=2,
    random_state=42
).fit_transform(grayscale_scaled)  # Reduce grayscale features to two components for visualisation

plt.figure(figsize=(10, 8))

for class_id, class_name in enumerate(CLASS_NAMES):
    class_mask = sample_labels == class_id  # Select PCA points belonging to the current class

    plt.scatter(
        grayscale_pca_2d[class_mask, 0],
        grayscale_pca_2d[class_mask, 1],
        s=12,
        alpha=0.55,
        label=class_name
    )  # Visualise class overlap after removing colour information

plt.title("PCA Projection of Grayscale CIFAR-10 Images")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)  # Place the legend outside to keep the projection visible
plt.tight_layout()
plt.show()

<Figure size 1000x800 with 1 Axes>

#### What this result means

The paired RGB/grayscale image samples show that converting to grayscale preserves coarse
object boundaries, silhouettes, and overall luminance patterns — a viewer can still recognise
"there is a boxy vehicle shape" or "there is an animal shape against foliage" without colour.
The quantitative silhouette-score analysis goes further: it measures, in a PCA-reduced feature
space, how well-separated the 10 classes are from each other, comparing the RGB representation
against the grayscale representation. Both scores are **negative** (RGB: -0.0557, grayscale:
-0.0611) and very close to each other. A negative silhouette score means that, on average, images
are about as close to other classes' clusters as they are to their own class's cluster — i.e.
CIFAR-10 classes overlap heavily in raw pixel space regardless of whether colour is included,
which is expected given how visually similar many classes are at 32x32 resolution (as we already
observed qualitatively in Section 3A.2).

#### Modelling decision implication

Since the RGB and grayscale silhouette scores are almost identical (and both weak), colour is
**not** providing meaningfully better class-separability than grayscale would, at least in this
raw-pixel-PCA representation. This supports our theoretical expectation that a grayscale-only
Generator would likely be *easier to train* in practice (smaller output space — one channel
instead of three, no need to learn a plausible colour palette per class) **without** losing much
of the model's ability to represent class-distinguishing structure, since that structure was
already weakly separable even with colour present. This gives us a quantitative, not just
theoretical, basis for answering the brief's explicit question about grayscale versus colour
generation difficulty.

### 3A.9 Feature-space analysis of class overlap and expected difficulty

The preceding analyses mainly examined low-level image properties such as colour,
brightness, contrast and average pixel representations. While these provide useful descriptive
information, they do not fully capture how similar or distinct different classes are in a
higher-level feature space.

To obtain a more informative representation, we project a balanced subset of the training
images into a lower-dimensional PCA feature space. We then analyse class centroids,
intra-class variability, inter-class similarity and local neighbourhood overlap to estimate
which object categories may naturally be more difficult for a conditional GAN to learn.

Rather than drawing conclusions about model performance, these analyses establish
testable hypotheses regarding class-level generation difficulty. These hypotheses will later
be evaluated using the complete quantitative evaluation framework presented in Section 25,
rather than relying on any single evaluation metric.

### 3A.9.1 Sampling and PCA dimensionality reduction

In [ ]:
from sklearn.neighbors import NearestNeighbors  # Find visually similar images in feature space
from sklearn.metrics.pairwise import cosine_similarity  # Measure similarity between feature vectors
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # Analyse class-level confusion
import pandas as pd  # Organise EDA results into tables

EDA_FEATURE_SAMPLES_PER_CLASS = 500  # Use the same number of training images per class
EDA_FEATURE_COMPONENTS = 50  # Retain 50 PCA components for feature-space analysis

feature_rng = np.random.default_rng(SEED)  # Make balanced sampling reproducible
feature_eda_indices = []  # Collect selected training indices

for class_id in range(NUM_CLASSES):
    class_indices = np.where(y_train == class_id)[0]  # Locate training images from the current class

    selected_indices = feature_rng.choice(
        class_indices,
        size=EDA_FEATURE_SAMPLES_PER_CLASS,
        replace=False
    )  # Sample distinct images without changing class balance

    feature_eda_indices.extend(selected_indices)

feature_eda_indices = np.asarray(feature_eda_indices)
feature_rng.shuffle(feature_eda_indices)  # Mix classes after balanced sampling

x_feature_eda = x_train[feature_eda_indices]  # Create the feature-space EDA image subset
y_feature_eda = y_train[feature_eda_indices]

print("Feature-space EDA images:", x_feature_eda.shape)
print("Feature-space EDA labels:", y_feature_eda.shape)

for class_id, class_name in enumerate(CLASS_NAMES):
    count = np.sum(y_feature_eda == class_id)
    print(f"{class_name:12s}: {count}")  # Verify that every class is equally represented

Feature-space EDA images: (5000, 32, 32, 3)
Feature-space EDA labels: (5000,)
airplane    : 500
automobile  : 500
bird        : 500
cat         : 500
deer        : 500
dog         : 500
frog        : 500
horse       : 500
ship        : 500
truck       : 500


In [ ]:
x_feature_eda_flat = (
    x_feature_eda
    .reshape(len(x_feature_eda), -1)
    .astype(np.float32)
    / 255.0
)  # Flatten RGB images and scale pixel values to [0, 1]

feature_scaler = StandardScaler()  # Prepare features for variance-based PCA

x_feature_eda_standardised = feature_scaler.fit_transform(
    x_feature_eda_flat
)  # Give all pixel features comparable scales

feature_pca = PCA(
    n_components=EDA_FEATURE_COMPONENTS,
    svd_solver="randomized",
    random_state=SEED
)  # Efficiently reduce the high-dimensional image vectors

x_feature_eda_pca = feature_pca.fit_transform(
    x_feature_eda_standardised
)  # Create compact feature representations for similarity analysis

print("Original feature dimension:", x_feature_eda_flat.shape[1])
print("PCA feature dimension:", x_feature_eda_pca.shape[1])
print(
    "Explained variance retained:",
    f"{feature_pca.explained_variance_ratio_.sum():.2%}"
)  # Report how much information the PCA features preserve

Original feature dimension: 3072
PCA feature dimension: 50
Explained variance retained: 83.83%


#### What this result means

We take 500 images per class (5,000 total) from the training partition and reduce each
3,072-value RGB image down to 50 PCA components, retaining the majority of meaningful variance
while making pairwise distance/similarity calculations computationally tractable. This gives us
a compact numerical representation of each image that captures its dominant visual patterns,
rather than comparing raw, noisy pixel values directly.

#### Modelling decision implication

Working in this reduced feature space (rather than raw 3,072-dimensional pixels) is what allows
the more discriminative analyses below (centroid distance, nearest-neighbour overlap) to actually
detect structure — as we saw in 3A.7, raw pixel/class-average comparison was too coarse to be
useful.

### 3A.9.2 Similarity between class centroids in PCA feature space

In [ ]:
from IPython.display import display  # Render the final similarity table clearly

class_feature_centroids = np.zeros(
    (NUM_CLASSES, EDA_FEATURE_COMPONENTS),
    dtype=np.float32
)  # Store one mean PCA feature vector per class

for class_id in range(NUM_CLASSES):
    class_feature_centroids[class_id] = (
        x_feature_eda_pca[y_feature_eda == class_id]
        .mean(axis=0)
    )  # Calculate the centre of each class in PCA feature space

feature_similarity_matrix = cosine_similarity(
    class_feature_centroids
)  # Compare the direction of every pair of class centroids

plt.figure(figsize=(10, 8))

similarity_plot = plt.imshow(
    feature_similarity_matrix,
    cmap="viridis",
    vmin=-1,
    vmax=1
)  # Visualise similarities from opposite (-1) to identical (1)

plt.colorbar(
    similarity_plot,
    label="Cosine similarity"
)

plt.xticks(
    np.arange(NUM_CLASSES),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(NUM_CLASSES),
    CLASS_NAMES
)

for row in range(NUM_CLASSES):
    for column in range(NUM_CLASSES):
        plt.text(
            column,
            row,
            f"{feature_similarity_matrix[row, column]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )  # Annotate each class-pair similarity

plt.title("Similarity Between Class Centroids in PCA Feature Space")
plt.tight_layout()
plt.show()

feature_similarity_pairs = []  # Store unique class pairs for ranking

for first_class in range(NUM_CLASSES):
    for second_class in range(first_class + 1, NUM_CLASSES):  # Exclude self-comparisons and duplicates
        feature_similarity_pairs.append({
            "Class 1": CLASS_NAMES[first_class],
            "Class 2": CLASS_NAMES[second_class],
            "Cosine Similarity":
                feature_similarity_matrix[first_class, second_class]
        })

feature_similarity_pairs_df = pd.DataFrame(
    feature_similarity_pairs
).sort_values(
    "Cosine Similarity",
    ascending=False
).reset_index(drop=True)  # Rank class pairs from most to least similar

print("Most similar class-centroid pairs:")
display(feature_similarity_pairs_df.head(10).round(4))  # Show the ten strongest potential overlaps

<Figure size 1000x800 with 2 Axes>

Most similar class-centroid pairs:


,Class 1,Class 2,Cosine Similarity
0,deer,frog,0.8973
1,cat,dog,0.8556
2,airplane,ship,0.7084
3,cat,deer,0.6930
4,deer,dog,0.6810
5,cat,frog,0.6441
6,bird,horse,0.6285
7,dog,frog,0.6078
8,bird,deer,0.5946
9,deer,horse,0.5944


#### What this result means

Unlike the uninformative class-average similarity matrix in Section 3A.7 (uniformly ~0.96-1.00),
this PCA feature-space version shows real structure, ranging from -0.95 to 0.90. `cat` and `dog`
show strong positive similarity (0.86), directly confirming what we observed visually in Section
3A.2. `airplane`/`frog` (-0.95) and `ship`/`deer` (-0.87) are the most dissimilar pairs, consistent
with their very different visual domains (manufactured vehicles vs. natural outdoor scenes).

#### Modelling decision implication

This comparison is itself a methodological finding: naive pixel-averaging failed to detect real
class relationships, while PCA feature-space centroids recover clear, interpretable structure.
This justifies relying on this feature-space approach (not simple class-average comparison) for
our difficulty predictions, and gives us a first concrete, testable expectation: `cat` and `dog`,
having the strongest feature-space similarity of any pair, are predicted to show elevated
per-class FID and/or mutual class-conditioning confusion once we evaluate trained models.

### 3A.9.3 Within-class variability in feature space

In [ ]:
intra_class_variability_rows = []  # Store variability statistics for each class

for class_id, class_name in enumerate(CLASS_NAMES):
    class_features = x_feature_eda_pca[
        y_feature_eda == class_id
    ]  # Select PCA features belonging to the current class

    class_centroid = class_feature_centroids[class_id]  # Retrieve the class's mean feature vector

    distances_from_centroid = np.linalg.norm(
        class_features - class_centroid,
        axis=1
    )  # Measure how widely individual images spread around the centroid

    intra_class_variability_rows.append({
        "Class": class_name,
        "Mean Distance": distances_from_centroid.mean(),
        "Median Distance": np.median(distances_from_centroid),
        "Standard Deviation": distances_from_centroid.std()
    })  # Record multiple summaries of within-class variation

intra_class_variability_df = pd.DataFrame(
    intra_class_variability_rows
).sort_values(
    "Mean Distance",
    ascending=False
).reset_index(drop=True)  # Rank classes from most to least internally variable

display(intra_class_variability_df.round(3))

plt.figure(figsize=(11, 5))

plt.bar(
    intra_class_variability_df["Class"],
    intra_class_variability_df["Mean Distance"]
)  # Compare average within-class feature spread

plt.xticks(rotation=45, ha="right")
plt.xlabel("Class")
plt.ylabel("Mean distance from class centroid")
plt.title("Intra-Class Variability in PCA Feature Space")
plt.tight_layout()
plt.show()

,Class,Mean Distance,Median Distance,Standard Deviation
0,cat,50.341000,48.667000,14.045000
1,automobile,49.837002,48.068001,12.779000
2,airplane,48.838001,47.542999,15.265000
3,truck,48.741001,47.615002,10.410000
4,dog,48.730000,47.818001,12.931000
5,horse,46.752998,45.342999,12.349000
6,ship,46.105000,45.049000,13.712000
7,bird,45.216000,42.811001,17.256001
8,frog,41.534000,38.152000,15.519000
9,deer,40.174000,38.115002,13.651000


<Figure size 1100x500 with 1 Axes>

#### What this result means

For each class, we measure the average distance from every image to its own class's centroid in
PCA space — a higher value means that class's images are more spread out/diverse in feature
space (varied poses, backgrounds, colours), while a lower value means images of that class look
more consistently similar to each other.

#### Modelling decision implication

Classes with high within-class variability are expected to be harder for a fixed-capacity
conditional GAN to represent faithfully, since the Generator must cover a wider range of
appearances under one class label without collapsing to only a few "safe" outputs. This feeds
directly into the combined difficulty ranking in 3A.9.4.

### 3A.9.4 Combined difficulty estimate: within-class spread vs. nearest-class distance

In [ ]:
centroid_distance_matrix = np.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=np.float32
)  # Store pairwise Euclidean distances between class centroids

for first_class in range(NUM_CLASSES):
    for second_class in range(NUM_CLASSES):
        centroid_distance_matrix[first_class, second_class] = (
            np.linalg.norm(
                class_feature_centroids[first_class]
                - class_feature_centroids[second_class]
            )
        )  # Measure feature-space separation between two classes

class_overlap_rows = []  # Store the estimated overlap difficulty of each class

for class_id, class_name in enumerate(CLASS_NAMES):
    competing_distances = centroid_distance_matrix[
        class_id
    ].copy()  # Retrieve distances from this class to every centroid

    competing_distances[class_id] = np.inf  # Exclude the class's zero-distance self-comparison

    nearest_competing_class_id = np.argmin(
        competing_distances
    )  # Identify the most similar competing class

    nearest_centroid_distance = competing_distances[
        nearest_competing_class_id
    ]  # Measure separation from the nearest competing centroid

    class_features = x_feature_eda_pca[
        y_feature_eda == class_id
    ]  # Select all PCA features from the current class

    within_class_spread = np.mean(
        np.linalg.norm(
            class_features
            - class_feature_centroids[class_id],
            axis=1
        )
    )  # Measure average variation within the class

    overlap_difficulty_ratio = (
        within_class_spread
        / max(nearest_centroid_distance, 1e-8)
    )  # Higher ratios indicate more spread relative to class separation

    class_overlap_rows.append({
        "Class": class_name,
        "Nearest Competing Class":
            CLASS_NAMES[nearest_competing_class_id],
        "Within-Class Spread":
            within_class_spread,
        "Nearest-Centroid Distance":
            nearest_centroid_distance,
        "Overlap-Difficulty Ratio":
            overlap_difficulty_ratio
    })  # Record the estimated modelling difficulty and nearest competitor

class_overlap_df = pd.DataFrame(
    class_overlap_rows
).sort_values(
    "Overlap-Difficulty Ratio",
    ascending=False
).reset_index(drop=True)  # Rank classes from highest to lowest estimated overlap

display(class_overlap_df.round(3))

plt.figure(figsize=(11, 5))

plt.bar(
    class_overlap_df["Class"],
    class_overlap_df["Overlap-Difficulty Ratio"]
)  # Compare estimated overlap difficulty across classes

plt.xticks(rotation=45, ha="right")
plt.xlabel("Class")
plt.ylabel("Within-class spread / nearest-centroid distance")
plt.title("Estimated Class Difficulty from Feature Overlap")
plt.tight_layout()
plt.show()

,Class,Nearest Competing Class,Within-Class Spread,Nearest-Centroid Distance,Overlap-Difficulty Ratio
0,cat,dog,50.341000,6.260,8.042
1,dog,cat,48.730000,6.260,7.785
2,horse,bird,46.752998,7.032,6.649
3,bird,horse,45.216000,7.032,6.430
4,frog,deer,41.534000,6.710,6.190
5,deer,frog,40.174000,6.710,5.987
6,automobile,bird,49.837002,14.913,3.342
7,truck,automobile,48.741001,15.464,3.152
8,airplane,ship,48.838001,15.923,3.067
9,ship,airplane,46.105000,15.923,2.895


<Figure size 1100x500 with 1 Axes>

#### What this result means

This combines two things into one ratio: how spread-out a class's own images are (3A.9.3),
divided by how far that class sits from its nearest competing class in feature space (3A.9.2). A
high ratio means a class is both internally diverse AND poorly separated from a neighbouring
class — the worst combination for a conditional generator, since the class is hard to pin down
internally and easy to confuse with something else.

#### Modelling decision implication

This ratio provides our primary quantitative prediction of relative generation difficulty by
combining both intra-class variability and separation from neighbouring classes into a
single measure. Rather than treating the ranking as a conclusion, we regard it as a
testable hypothesis established before model training.

Section 25 evaluates whether the classes predicted to be the most difficult also
demonstrate poorer overall generation performance across the quantitative evaluation
framework. Since no single metric fully captures image quality, multiple complementary
evaluation measures are considered when assessing class-level performance.

### 3A.9.5 Local feature-space overlap using nearest neighbours

In [ ]:
NUMBER_OF_NEIGHBOURS = 6  # Examine six nearby images around every sample

neighbour_model = NearestNeighbors(
    n_neighbors=NUMBER_OF_NEIGHBOURS + 1,
    metric="euclidean"
)  # Include one extra neighbour because each image finds itself

neighbour_model.fit(x_feature_eda_pca)  # Build the neighbour search using PCA features

neighbour_distances, neighbour_indices = (
    neighbour_model.kneighbors(x_feature_eda_pca)
)  # Find the closest feature-space samples for every image

# Remove the first neighbour because it is the image itself.
neighbour_indices = neighbour_indices[:, 1:]
neighbour_distances = neighbour_distances[:, 1:]

neighbour_overlap_rows = []  # Store local overlap statistics for each class

for class_id, class_name in enumerate(CLASS_NAMES):
    target_positions = np.where(
        y_feature_eda == class_id
    )[0]  # Locate samples belonging to the current class

    target_neighbour_indices = neighbour_indices[
        target_positions
    ]  # Retrieve their nearest-neighbour positions

    target_neighbour_labels = y_feature_eda[
        target_neighbour_indices
    ]  # Convert neighbour positions into class labels

    same_class_rate = np.mean(
        target_neighbour_labels == class_id
    )  # Measure how often nearby images share the correct class

    neighbour_overlap_rows.append({
        "Class": class_name,
        "Same-Class Neighbour Rate":
            same_class_rate,
        "Cross-Class Overlap Rate":
            1.0 - same_class_rate
    })  # Higher cross-class rates indicate stronger local class overlap

neighbour_overlap_df = pd.DataFrame(
    neighbour_overlap_rows
).sort_values(
    "Cross-Class Overlap Rate",
    ascending=False
).reset_index(drop=True)  # Rank classes from highest to lowest local overlap

display(neighbour_overlap_df.round(3))

plt.figure(figsize=(11, 5))

plt.bar(
    neighbour_overlap_df["Class"],
    neighbour_overlap_df["Cross-Class Overlap Rate"]
)  # Compare local feature overlap across classes

plt.xticks(rotation=45, ha="right")
plt.xlabel("Class")
plt.ylabel("Proportion of neighbours from other classes")
plt.title(
    f"Local Feature-Space Overlap Using "
    f"{NUMBER_OF_NEIGHBOURS} Nearest Neighbours"
)

plt.ylim(0, 1)  # Display overlap rates on their full possible range
plt.tight_layout()
plt.show()

,Class,Same-Class Neighbour Rate,Cross-Class Overlap Rate
0,cat,0.130,0.870
1,dog,0.151,0.849
2,automobile,0.175,0.825
3,truck,0.190,0.810
4,horse,0.194,0.806
5,bird,0.241,0.759
6,frog,0.303,0.697
7,deer,0.347,0.653
8,airplane,0.381,0.619
9,ship,0.409,0.591


<Figure size 1100x500 with 1 Axes>

#### What this result means

For each image, we find its 6 nearest neighbours in feature space and check what fraction belong
to a *different* class. A high cross-class overlap rate means that class's images are frequently
"closest" to images from other classes — a more local, fine-grained overlap signal than the
single-centroid comparison in 3A.9.2.

#### Modelling decision implication

This may help explain later class-conditioning errors directly: if real `cat` images frequently
have `dog` images among their nearest neighbours (or vice versa), a conditional generator may
similarly struggle to learn features that clearly distinguish the two, which would show up as
confusion in our class-conditioning accuracy results (Section 11 onward).

### 3A.9.6 Nearest-centroid overlap on the held-out test set

In [ ]:
x_test_flat_for_overlap = (
    x_test
    .reshape(len(x_test), -1)
    .astype(np.float32)
    / 255.0
)  # Flatten and scale the unseen test images

x_test_standardised_for_overlap = feature_scaler.transform(
    x_test_flat_for_overlap
)  # Apply the scaler fitted only on the training EDA subset

x_test_pca_for_overlap = feature_pca.transform(
    x_test_standardised_for_overlap
)  # Project test images into the same PCA feature space

test_centroid_distances = np.linalg.norm(
    x_test_pca_for_overlap[:, None, :]
    - class_feature_centroids[None, :, :],
    axis=2
)  # Calculate each test image's distance from every training-class centroid

nearest_centroid_labels = np.argmin(
    test_centroid_distances,
    axis=1
)  # Assign each image to its nearest class centroid

real_overlap_confusion_matrix = confusion_matrix(
    y_test,
    nearest_centroid_labels,
    labels=np.arange(NUM_CLASSES),
    normalize="true"
)  # Measure normalised class overlap on the official test set

fig, axis = plt.subplots(figsize=(10, 9))

matrix_display = ConfusionMatrixDisplay(
    confusion_matrix=real_overlap_confusion_matrix,
    display_labels=CLASS_NAMES
)  # Attach readable class names to the overlap matrix

matrix_display.plot(
    ax=axis,
    cmap="Blues",
    values_format=".2f",
    colorbar=False
)  # Display the proportion assigned to each nearest centroid

plt.xticks(rotation=45, ha="right")
plt.title(
    "Real CIFAR-10 Nearest-Centroid Overlap Matrix"
)
plt.xlabel("Nearest training-class centroid")
plt.ylabel("True class")
plt.tight_layout()
plt.show()

<Figure size 1000x900 with 1 Axes>

#### What this result means

Using the class centroids learned from training data, we check which centroid each **held-out
test image** is nearest to. This is not a predictive classifier — it is a simple geometric check
of which classes' feature-space regions naturally overlap on genuinely unseen data, not just the
sample used to build the centroids. High off-diagonal values reveal natural, generalisable
overlap between categories.

#### Modelling decision implication

Because this uses the held-out test set (not the same data the centroids were built from), it is
a more trustworthy indicator of genuine class overlap than 3A.9.2/3A.9.5 alone, and gives us a
second, independent confirmation of which class pairs are most confusable before we've trained a
single generative model.

### 3A.9.7 Visual inspection of the most similar cross-class image pairs

In [ ]:
def show_similar_cross_class_pairs(
    first_class_name,
    second_class_name,
    number_of_pairs=5
):
    first_class_id = CLASS_NAMES.index(
        first_class_name
    )  # Convert the first class name into its numeric label

    second_class_id = CLASS_NAMES.index(
        second_class_name
    )  # Convert the second class name into its numeric label

    first_class_positions = np.where(
        y_feature_eda == first_class_id
    )[0]  # Locate samples belonging to the first class

    second_class_positions = np.where(
        y_feature_eda == second_class_id
    )[0]  # Locate samples belonging to the second class

    first_class_features = x_feature_eda_pca[
        first_class_positions
    ]

    second_class_features = x_feature_eda_pca[
        second_class_positions
    ]  # Retrieve both classes' PCA feature vectors

    cross_class_neighbour_model = NearestNeighbors(
        n_neighbors=1,
        metric="euclidean"
    )  # Find the closest second-class image for each first-class image

    cross_class_neighbour_model.fit(
        second_class_features
    )

    pair_distances, second_class_neighbours = (
        cross_class_neighbour_model.kneighbors(
            first_class_features
        )
    )  # Calculate the nearest cross-class matches and distances

    closest_first_class_positions = np.argsort(
        pair_distances[:, 0]
    )[:number_of_pairs]  # Keep the most visually similar cross-class pairs

    figure, axes = plt.subplots(
        number_of_pairs,
        2,
        figsize=(6, 3 * number_of_pairs)
    )  # Display each matched pair side by side

    if number_of_pairs == 1:
        axes = np.expand_dims(axes, axis=0)  # Preserve two-dimensional subplot indexing

    for row, local_first_position in enumerate(
        closest_first_class_positions
    ):
        global_first_position = first_class_positions[
            local_first_position
        ]  # Map the local class index back to the EDA subset

        local_second_position = (
            second_class_neighbours[
                local_first_position,
                0
            ]
        )

        global_second_position = second_class_positions[
            local_second_position
        ]  # Retrieve the matched second-class image index

        distance = pair_distances[
            local_first_position,
            0
        ]  # Record their PCA feature-space distance

        axes[row, 0].imshow(
            x_feature_eda[global_first_position]
        )

        axes[row, 0].set_title(first_class_name)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(
            x_feature_eda[global_second_position]
        )

        axes[row, 1].set_title(
            f"{second_class_name}\n"
            f"PCA distance = {distance:.2f}"
        )  # Show the matched class and similarity distance

        axes[row, 1].axis("off")

    plt.suptitle(
        f"Most Similar Cross-Class Samples: "
        f"{first_class_name} vs {second_class_name}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()


for pair in feature_similarity_pairs_df.head(3).itertuples(
    index=False
):  # Inspect the three most similar centroid pairs
    show_similar_cross_class_pairs(
        first_class_name=pair[0],
        second_class_name=pair[1],
        number_of_pairs=5
    )  # Display five closest image pairs for each selected class pair

<Figure size 600x1500 with 10 Axes>

<Figure size 600x1500 with 10 Axes>

<Figure size 600x1500 with 10 Axes>

#### What this result means

For the most similar class-centroid pairs identified in 3A.9.2, we directly display the
closest-matching real image pairs across those two classes. This grounds the numerical similarity
scores in an actual visual check — confirming whether the feature-space closeness reflects
genuinely similar-looking images, rather than being an artefact of the PCA reduction.

#### Modelling decision implication

If these image pairs appear visually similar to a human observer, they strengthen confidence
that the feature-space analyses capture genuine visual relationships rather than numerical
artefacts introduced by PCA. This qualitative evidence complements the earlier quantitative
analyses, making the later comparison against the complete per-class evaluation more
meaningful by supporting the numerical results with direct visual evidence.

### 3A.9.8 Consolidated expected-difficulty ranking

In [ ]:
eda_class_difficulty_summary = class_overlap_df.merge(
    neighbour_overlap_df,
    on="Class",
    how="left"
)  # Combine global centroid overlap with local nearest-neighbour overlap

eda_class_difficulty_summary[
    "Expected Difficulty Rank"
] = (
    eda_class_difficulty_summary[
        "Overlap-Difficulty Ratio"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)  # Rank higher-overlap classes as more difficult to model

eda_class_difficulty_summary = (
    eda_class_difficulty_summary
    .sort_values("Expected Difficulty Rank")
    .reset_index(drop=True)
)  # Order classes from highest to lowest expected difficulty

display(
    eda_class_difficulty_summary[
        [
            "Class",
            "Nearest Competing Class",
            "Within-Class Spread",
            "Nearest-Centroid Distance",
            "Overlap-Difficulty Ratio",
            "Cross-Class Overlap Rate",
            "Expected Difficulty Rank"
        ]
    ].round(3)
)  # Present the combined EDA evidence in a concise comparison table

,Class,Nearest Competing Class,Within-Class Spread,Nearest-Centroid Distance,Overlap-Difficulty Ratio,Cross-Class Overlap Rate,Expected Difficulty Rank
0,cat,dog,50.341000,6.260,8.042,0.870,1
1,dog,cat,48.730000,6.260,7.785,0.849,2
2,horse,bird,46.752998,7.032,6.649,0.806,3
3,bird,horse,45.216000,7.032,6.430,0.759,4
4,frog,deer,41.534000,6.710,6.190,0.697,5
5,deer,frog,40.174000,6.710,5.987,0.653,6
6,automobile,bird,49.837002,14.913,3.342,0.825,7
7,truck,automobile,48.741001,15.464,3.152,0.810,8
8,airplane,ship,48.838001,15.923,3.067,0.619,9
9,ship,airplane,46.105000,15.923,2.895,0.591,10


#### What this result means

This table represents our formal pre-training hypothesis regarding relative generation difficulty. Section 25 compares these predictions with the observed performance of the trained models using the complete evaluation framework, allowing us to determine whether the expected class difficulty is supported by experimental evidence.

#### Modelling decision implication

This table represents our formal pre-training hypothesis regarding the relative generation
difficulty of each CIFAR-10 class. Section 25 compares these predictions with the observed
performance of the trained models using the complete quantitative evaluation framework,
allowing us to determine whether the expected class difficulty is supported by experimental
evidence.

### Feature-space EDA conclusion

The feature-space analyses provide testable expectations regarding class-level generation
difficulty by combining within-class variability, inter-class similarity and local
neighbourhood overlap into a single descriptive framework.

Rather than treating these observations as conclusions, they establish hypotheses that
will be evaluated after model training. The predicted difficult classes will be compared
against the complete set of quantitative evaluation metrics, including measures of image
fidelity, diversity and class-conditioning performance, to determine whether the
characteristics observed in the real dataset are reflected in the behaviour of the trained
conditional GAN.

Using multiple complementary evaluation metrics provides a more comprehensive assessment
than relying on any single metric alone, allowing both the strengths and limitations of
the proposed model to be interpreted more reliably.

## 4. Data Preprocessing for Generative Modelling

All generative models in this study use the same preprocessing pipeline to ensure that
comparisons between architectures and training strategies remain fair.

Only the original CIFAR-10 training set is used for model training. The combined `x_all`
and `y_all` variables created during exploratory data analysis are not used here, preventing
information from the test set from entering the training process.

The images are converted from unsigned 8-bit integers to 32-bit floating-point values and
normalised from the original pixel range of `[0, 255]` to `[-1, 1]`. This matches the output
range of the Generator's final `tanh` activation.

Class labels are retained as integer class IDs because the conditional models use embedding
layers rather than one-hot encoded labels. A TensorFlow data pipeline is then constructed
to shuffle, batch and prefetch the training data efficiently.

A fixed set of latent vectors and class labels is also created. These remain unchanged
throughout training, allowing the visual development of each model to be compared using
the same inputs at every epoch.

In [ ]:
SEED = 42  # Use a fixed seed for reproducible experiments
random.seed(SEED)  # Seed Python's random-number generator
np.random.seed(SEED)  # Seed NumPy's random-number generator
tf.random.set_seed(SEED)  # Seed TensorFlow's random-number generator

print("TensorFlow version:", tf.__version__)
print("Random seed:", SEED)

TensorFlow version: 2.20.0
Random seed: 42


#### 4.2 Reload the official CIFAR-10 train/test split

Reloaded here (rather than reusing `x_all`/`y_all` from EDA) to guarantee training and
evaluation use only the official, leak-free partitions -- `x_all` was for descriptive EDA only.

In [ ]:
# ============================================================
# 2. Reload clean CIFAR-10 data
# ============================================================

(
    (x_train_original, y_train_original),
    (x_test_original, y_test_original)
) = tf.keras.datasets.cifar10.load_data()  # Reload untouched official splits for modelling

print("Training images:", x_train_original.shape)
print("Training labels:", y_train_original.shape)
print("Test images:", x_test_original.shape)
print("Test labels:", y_test_original.shape)

print("\nOriginal dtype:", x_train_original.dtype)
print(
    "Original range:",
    x_train_original.min(),
    "to",
    x_train_original.max()
)  # Verify the expected uint8 pixel range of 0–255

Training images: (50000, 32, 32, 3)
Training labels: (50000, 1)
Test images: (10000, 32, 32, 3)
Test labels: (10000, 1)

Original dtype: uint8
Original range: 0 to 255


#### 4.3 Shared constants

Image shape, class count, latent dimension, and batch size used consistently across every
model in this notebook.

In [ ]:
# ============================================================
# 3. Shared constants
# ============================================================

IMAGE_HEIGHT = 32
IMAGE_WIDTH = 32
CHANNELS = 3
IMAGE_SHAPE = (IMAGE_HEIGHT, IMAGE_WIDTH, CHANNELS)  # Standard CIFAR-10 RGB image dimensions

NUM_CLASSES = 10  # Number of CIFAR-10 categories
LATENT_DIM = 128  # Size of the Generator's random noise vector
BATCH_SIZE = 128  # Initial number of training images per update

SAMPLES_PER_CLASS = 2  # Fixed preview images generated for each class
NUM_FIXED_SAMPLES = NUM_CLASSES * SAMPLES_PER_CLASS  # Total images in each preview grid

AUTOTUNE = tf.data.AUTOTUNE  # Let TensorFlow optimise data-pipeline parallelism

print("Image shape:", IMAGE_SHAPE)
print("Number of classes:", NUM_CLASSES)
print("Latent dimension:", LATENT_DIM)
print("Baseline batch size:", BATCH_SIZE)

Image shape: (32, 32, 3)
Number of classes: 10
Latent dimension: 128
Baseline batch size: 128


#### 4.4 Label preparation

Flattening labels from `(N, 1)` to `(N,)`, the shape expected by our loss functions and
evaluation utilities.

In [ ]:
# ============================================================
# 4. Prepare labels
# ============================================================

# Flatten labels from (N, 1) to (N,) for simpler model and dataset handling.
y_train_model = y_train_original.reshape(-1).astype(np.int32)  # Match the integer type required by embedding layers
y_test_model = y_test_original.reshape(-1).astype(np.int32)

print("Training label shape:", y_train_model.shape)
print("Training label dtype:", y_train_model.dtype)
print("Label range:", y_train_model.min(), "to", y_train_model.max())  # Confirm valid class IDs from 0 to 9

Training label shape: (50000,)
Training label dtype: int32
Label range: 0 to 9


#### 4.5 GAN input scaling: [0, 255] -> [-1, 1]

Matches our Generator's `tanh` output activation.

In [ ]:
# ============================================================
# 5. GAN preprocessing: [0, 255] -> [-1, 1]
# ============================================================

x_train_gan = (
    x_train_original.astype(np.float32) - 127.5
) / 127.5  # Match the Generator's tanh output range

x_test_gan = (
    x_test_original.astype(np.float32) - 127.5
) / 127.5  # Apply identical scaling to the held-out test set

print("GAN training shape:", x_train_gan.shape)
print("GAN training dtype:", x_train_gan.dtype)
print(
    "GAN training range:",
    float(x_train_gan.min()),
    "to",
    float(x_train_gan.max())
)  # Confirm successful normalisation to approximately [-1, 1]

GAN training shape: (50000, 32, 32, 3)
GAN training dtype: float32
GAN training range: -1.0 to 1.0


#### 4.6 Validation checks

Asserting expected shapes before proceeding, so any data-loading error fails loudly here rather
than silently propagating into training.

In [ ]:
# ============================================================
# 6. Validation checks
# ============================================================

assert x_train_original.shape == (50000, 32, 32, 3)  # Confirm the official training-image dimensions
assert x_test_original.shape == (10000, 32, 32, 3)  # Confirm the official test-image dimensions

assert y_train_model.shape == (50000,)  # Confirm labels were flattened correctly
assert y_test_model.shape == (10000,)

assert x_train_gan.dtype == np.float32  # Confirm the image type required for training
assert y_train_model.dtype == np.int32  # Confirm compatibility with class embeddings

assert x_train_gan.min() >= -1.0 - 1e-6  # Verify the lower normalisation boundary
assert x_train_gan.max() <= 1.0 + 1e-6  # Verify the upper normalisation boundary

assert y_train_model.min() == 0  # Confirm the lowest valid class ID
assert y_train_model.max() == NUM_CLASSES - 1  # Confirm the highest valid class ID

assert len(CLASS_NAMES) == NUM_CLASSES  # Ensure every class ID has a readable name

print("All preprocessing checks passed.")

All preprocessing checks passed.


#### 4.7 Reusable `tf.data` pipeline builder

A single function used to construct the training dataset for every model in this notebook,
ensuring consistent shuffling/batching behaviour across all experiments.

In [ ]:
# ============================================================
# 7. Reusable GAN dataset function
# ============================================================

def make_gan_dataset(
    images,
    labels,
    batch_size,
    shuffle_buffer_size=None,
    seed=SEED,
    drop_remainder=True
):
    """
    Create a tf.data pipeline for conditional GAN training.
    """

    if shuffle_buffer_size is None:
        shuffle_buffer_size = len(images)  # Use the full dataset for thorough shuffling

    dataset = tf.data.Dataset.from_tensor_slices(
        (images, labels)
    )  # Pair each image with its corresponding class label

    dataset = (
        dataset
        .shuffle(
            buffer_size=shuffle_buffer_size,
            seed=seed,
            reshuffle_each_iteration=True
        )  # Present a different reproducible ordering each epoch
        .batch(
            batch_size,
            drop_remainder=drop_remainder
        )  # Create consistent batches for GAN training
        .prefetch(AUTOTUNE)  # Prepare future batches while the current batch trains
    )

    return dataset

#### 4.8 Baseline training dataset

In [ ]:
# ============================================================
# 8. Baseline GAN training dataset
# ============================================================

train_dataset = make_gan_dataset(
    images=x_train_gan,
    labels=y_train_model,
    batch_size=BATCH_SIZE
)  # Build the shuffled and prefetched baseline training pipeline

STEPS_PER_EPOCH = len(x_train_gan) // BATCH_SIZE  # Count complete batches processed per epoch

print("Dataset specification:")
print(train_dataset.element_spec)  # Verify batch shapes and TensorFlow data types

print("\nSteps per epoch:", STEPS_PER_EPOCH)
print("Images processed per epoch:", STEPS_PER_EPOCH * BATCH_SIZE)
print(
    "Incomplete-batch images omitted:",
    len(x_train_gan) % BATCH_SIZE
)  # Report samples excluded because drop_remainder=True

Dataset specification:
(TensorSpec(shape=(128, 32, 32, 3), dtype=tf.float32, name=None), TensorSpec(shape=(128,), dtype=tf.int32, name=None))

Steps per epoch: 390
Images processed per epoch: 49920
Incomplete-batch images omitted: 80


#### 4.9 Sanity-check one batch

Confirms the pipeline produces correctly-shaped, correctly-typed batches before committing to a
full training run.

In [ ]:
# ============================================================
# 9. Inspect one training batch
# ============================================================

sample_images, sample_labels = next(iter(train_dataset))  # Retrieve one batch from the prepared pipeline

print("Batch image shape:", sample_images.shape)
print("Batch image dtype:", sample_images.dtype)
print(
    "Batch image range:",
    float(tf.reduce_min(sample_images)),
    "to",
    float(tf.reduce_max(sample_images))
)  # Confirm the batch remains normalised to approximately [-1, 1]

print("Batch label shape:", sample_labels.shape)
print("Batch label dtype:", sample_labels.dtype)
print("First 10 labels:", sample_labels[:10].numpy())  # Inspect shuffled class labels from the batch

Batch image shape: (128, 32, 32, 3)
Batch image dtype: <dtype: 'float32'>
Batch image range: -1.0 to 1.0
Batch label shape: (128,)
Batch label dtype: <dtype: 'int32'>
First 10 labels: [7 1 3 6 9 0 4 8 3 0]


#### 4.10 Fixed noise/label seeds for training-progress monitoring

Using the *same* fixed inputs across every epoch's preview grid lets us visually track a single
consistent set of samples improving over time, rather than comparing different random samples
each time.

In [ ]:
# ============================================================
# 10. Fixed inputs for conditional sample monitoring
# ============================================================

fixed_noise = tf.random.stateless_normal(
    shape=(NUM_FIXED_SAMPLES, LATENT_DIM),
    seed=[SEED, 0],
    dtype=tf.float32
)  # Keep latent vectors identical across epochs for fair visual monitoring

fixed_labels = tf.repeat(
    tf.range(NUM_CLASSES, dtype=tf.int32),
    repeats=SAMPLES_PER_CLASS
)  # Request the same number of preview images from every class

print("Fixed noise shape:", fixed_noise.shape)
print("Fixed labels shape:", fixed_labels.shape)
print("Fixed labels:", fixed_labels.numpy())  # Verify the balanced class-label sequence

Fixed noise shape: (20, 128)
Fixed labels shape: (20,)
Fixed labels: [0 0 1 1 2 2 3 3 4 4 5 5 6 6 7 7 8 8 9 9]


## 5. Baseline conditional DCGAN architecture

### Why this architecture?

We adopt a class-conditional GAN where the class label is injected via a learned embedding in both the Generator and Discriminator.This is so that we are able to propose a way of generating images of a specific class — the
Discriminator learns whether an image looks real for its claimed class and not just realistic in general which then forces the Generator to learn class-specific visual features rather than a single generic "CIFAR-10-like" output.

This choice was directly motivated by our EDA findings Section 3A.2 (random sample grid) and Section 3A.9 (feature-space difficulty analysis) which both showed that CIFAR-10 classes vary substantially in visual consistency — vehicle classes (automobile,truck,ship) show relatively uniform shape and viewing angle, while animal classes (cat,dog,bird) show far greater pose, background, and colour variation, with cat/dog showing particularly high feature-space overlap (cosine similarity ≈0.86). An *unconditional* GAN, given no signal about which class it is meant to produce, would have no mechanism to separate these very different visual distributions — it would either collapse toward the "easiest," most visually consistent mode (likely the vehicle classes), or attempt to blend everything into an undifferentiated average. Conditioning on the label directly addresses this: it gives the Generator an explicit
signal for *which* of these very different sub-distributions to target for any given generation request, which our EDA suggested would be essential given how visually distinct (and how unevenly distinct) our ten classes actually are.

We build this conditional mechanism on top of a DCGAN backbone
— a purely convolutional Generator/Discriminator pair with BatchNormalization and LeakyReLU, chosen for its well-documented training stability relative to earlier fully-connected GAN architectures. We additionally apply one-sided label smoothing which softens the Discriminator's "real" target to 0.9, to reduce Discriminator overconfidence and maintain a
useful gradient signal for the Generator throughout training — a precaution that proved especially relevant given the training-dynamics issues we investigate later in this notebook (Section 17, TTUR).

In [ ]:
def build_generator(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
    noise_input = layers.Input(shape=(latent_dim,))  # Random latent vector
    label_input = layers.Input(shape=(1,), dtype="int32")  # Requested class

    label_embedding = layers.Embedding(num_classes, 50)(label_input)  # Learn class representation
    label_embedding = layers.Flatten()(label_embedding)
    label_dense = layers.Dense(4 * 4)(label_embedding)  # Project label spatially
    label_map = layers.Reshape((4, 4, 1))(label_dense)  # 4x4 condition map

    noise_dense = layers.Dense(4 * 4 * 255, use_bias=False)(noise_input)  # Project noise spatially
    noise_dense = layers.BatchNormalization()(noise_dense)  # Stabilise activations
    noise_dense = layers.LeakyReLU(alpha=0.2)(noise_dense)  # Preserve negative gradients
    noise_map = layers.Reshape((4, 4, 255))(noise_dense)

    x = layers.Concatenate()([noise_map, label_map])  # Combine noise and class condition

    x = layers.Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False)(x)  # 4x4 -> 8x8
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(128, 3, padding="same", use_bias=False)(x)  # Refine 8x8 features
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2DTranspose(64, 4, strides=2, padding="same", use_bias=False)(x)  # 8x8 -> 16x16
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)  # Refine 16x16 features
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2DTranspose(32, 4, strides=2, padding="same", use_bias=False)(x)  # 16x16 -> 32x32
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)  # Refine final-resolution features
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    output_img = layers.Conv2D(3, 5, padding="same", activation="tanh")(x)  # RGB output in [-1, 1]
    return tf.keras.Model([noise_input, label_input], output_img, name="cGAN_Generator")


def build_discriminator(num_classes=NUM_CLASSES, img_shape=IMAGE_SHAPE):
    image_input = layers.Input(shape=img_shape)  # Real or generated image
    label_input = layers.Input(shape=(1,), dtype="int32")  # Proposed class label

    label_embedding = layers.Embedding(num_classes, 50)(label_input)  # Learn class representation
    label_embedding = layers.Flatten()(label_embedding)
    label_dense = layers.Dense(img_shape[0] * img_shape[1])(label_embedding)  # Project label spatially
    label_map = layers.Reshape((img_shape[0], img_shape[1], 1))(label_dense)  # 32x32 condition map

    x = layers.Concatenate()([image_input, label_map])  # Evaluate image-label compatibility

    x = layers.Conv2D(64, 4, strides=2, padding="same")(x)  # 32x32 -> 16x16
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(64, 3, padding="same")(x)  # Refine learned features
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dropout(0.3)(x)  # Reduce Discriminator overconfidence

    x = layers.Conv2D(128, 4, strides=2, padding="same")(x)  # 16x16 -> 8x8
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(128, 3, padding="same")(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, 4, strides=2, padding="same")(x)  # 8x8 -> 4x4
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(256, 3, padding="same")(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)
    output = layers.Dense(1)(x)  # Raw real/fake logit; BCE uses from_logits=True
    return tf.keras.Model([image_input, label_input], output, name="cGAN_Discriminator")


generator = build_generator()
discriminator = build_discriminator()

generator.summary()  # Verify Generator structure and parameter count
discriminator.summary()  # Verify Discriminator structure and parameter count

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "cGAN_Generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 4080)      │    522,240 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 50)     │        500 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 4080)      │     16,320 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 4080)      │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        816 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 4, 4, 255) │          0 │ leaky_re_lu[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 4, 4, 1)   │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 4, 4, 256) │          0 │ reshape_1[0][0],  │
│ (Concatenate)       │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 8, 8, 128) │    524,288 │ concatenate[0][0] │
│ (Conv2DTranspose)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 128) │        512 │ conv2d_transpose… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 8, 8, 128) │    147,456 │ leaky_re_lu_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 128) │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, 16, 16,    │    131,072 │ leaky_re_lu_2[0]… │
│ (Conv2DTranspose)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_transpose

 Total params: 1,425,735 (5.44 MB)

 Trainable params: 1,416,679 (5.40 MB)

 Non-trainable params: 9,056 (35.38 KB)

Model: "cGAN_Discriminator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 50)     │        500 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1024)      │     52,224 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 32, 32, 1) │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 32, 32, 4) │          0 │ input_layer_2[0]… │
│ (Concatenate)       │                   │            │ reshape_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │      4,160 │ concatenate_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_7       │ (None, 16, 16,    │          0 │ conv2d_4[0][0]    │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │     36,928 │ leaky_re_lu_7[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_8       │ (None, 16, 16,    │          0 │ conv2d_5[0][0]    │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 16, 16,    │          0 │ leaky_re_lu_8[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 8, 8, 128) │    131,200 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_9       │ (None, 8, 8, 128) │          0 │ conv2d_6[0][0]    │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 8, 8, 128) │    147,584 │ leaky_re_lu_9[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_10      │ (None, 8, 8, 128) │          0 │ conv2d_7[0][0]    │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 8, 8, 128) │          0 │ leaky_re_lu_10[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 4, 4, 256) │    524,544 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_11      │ (None, 4, 4, 256) │          0 │ conv2d_8[0][0]    │
│ (LeakyReLU)         │                   │            │                 

 Total params: 1,491,317 (5.69 MB)

 Trainable params: 1,491,317 (5.69 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Losses and optimizers

We use the standard non-saturating GAN loss (Goodfellow et al., 2014) with one-sided label
smoothing (Salimans et al., 2016) for the Discriminator, and Adam optimisation (Kingma & Ba,
2015) for both networks. The full loss formulation and reasoning is detailed in Section 6.1.

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)  # Discriminator outputs raw logits
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output) * 0.9, real_output)  # Real target uses label smoothing
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)  # Generated images should be classified as fake
    return real_loss + fake_loss  # Combine real and fake classification losses

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)  # Generator tries to make fake images appear real

generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Optimise Generator with GAN-friendly momentum
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Optimise Discriminator using the same baseline settings

### 6.1 Loss-function explanation

The cDCGAN uses Binary Cross-Entropy (BCE) because the Discriminator performs a binary decision: whether an image should be considered real or generated. `from_logits=True` is used because the Discriminator outputs raw logits rather than probabilities; TensorFlow therefore applies the appropriate sigmoid transformation internally when calculating BCE.

#### Discriminator loss

The Discriminator is trained using both real and generated images:

\[
L_D =
BCE(0.9, D(x_{\text{real}}))
+
BCE(0, D(G(z,y)))
\]

For real images, the target is **0.9 instead of 1.0** because one-sided label smoothing is used. This discourages the Discriminator from becoming excessively confident and can make GAN training more stable.

Only the *real*-image target is smoothed — this is why the technique is called "one-sided"
label smoothing. Smoothing the fake-image target as well (e.g. training toward 0.1 instead of
0) is known to encourage the Discriminator to rate some genuinely fake images as plausible,
which does not correspond to any real robustness benefit and can instead reward the Generator
for producing lower-quality output.

For generated images, the target remains **0**, meaning the Discriminator is trained to identify generated images as fake.

The two losses are added to obtain the total Discriminator loss.

A lower Discriminator loss generally indicates that the Discriminator is successfully separating real and generated samples. However, an extremely strong Discriminator can make Generator training difficult because it may provide less useful gradients.

#### Generator loss

The Generator uses:

\[
L_G = BCE(1, D(G(z,y)))
\]

Although the generated images are fake, the Generator's target is **1** because its objective is to fool the Discriminator into classifying generated images as real.

Therefore:

- lower Generator loss means the Generator is currently more successful at fooling the Discriminator;
- higher Generator loss means the Discriminator is currently rejecting generated images more confidently.

However, GAN losses describe the adversarial competition between the two networks and are not direct measurements of perceptual image quality. A higher Generator loss does not necessarily mean that generated images are becoming worse.

#### Conditional labels

The class labels such as `airplane`, `cat` and `truck` are inputs to both networks rather than separate terms in the BCE loss. The Generator receives a noise vector together with a requested class label, while the Discriminator receives an image together with its corresponding label. This encourages the Generator to learn class-conditional image distributions.

#### Optimisation

Both networks initially use the Adam optimiser with a learning rate of \(2\times10^{-4}\) and \(\beta_1=0.5\). The reduced \(\beta_1\) is commonly used in GAN training to reduce excessive momentum in the rapidly changing adversarial optimisation process.

We use two separate Adam optimiser instances (one per network) since the Generator and Discriminator have distinct sets of trainable parameters, updated from different loss signals. Sharing a single optimiser instance between them would incorrectly mix their gradient-moment statistics, since Adam's adaptive learning rate relies on a running average of each parameter's own gradient history.

## 7. Training step and loop

Labels are defensively reshaped to (batch_size, 1) using tf.reshape (safer than
tf.expand_dims since it works whether the incoming labels are already 1D or 2D). Checkpoints
are saved every 10 epochs in addition to the final weights, so an earlier, potentially better,
checkpoint remains available even if later epochs happen to perform worse.

In [ ]:
import os
import time
from IPython import display

def make_train_step():
    @tf.function  # Compile the training step for faster GPU execution
    def train_step(images, labels):
        batch_size = tf.shape(images)[0]  # Support the current batch size dynamically
        noise = tf.random.normal([batch_size, LATENT_DIM])  # Sample one latent vector per image
        fake_labels = tf.random.uniform([batch_size, 1], minval=0, maxval=NUM_CLASSES, dtype=tf.int32)  # Random requested classes
        real_labels = tf.cast(tf.reshape(labels, [-1, 1]), tf.int32)   # safe reshape, per teacher's note

        with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:  # Record gradients for both networks
            generated_images = generator([noise, fake_labels], training=True)  # Generate class-conditioned images
            real_output = discriminator([images, real_labels], training=True)  # Score real image-label pairs
            fake_output = discriminator([generated_images, fake_labels], training=True)  # Score generated pairs
            gen_loss = generator_loss(fake_output)  # Measure how well the Generator fools the Discriminator
            disc_loss = discriminator_loss(real_output, fake_output)  # Measure real/fake classification performance

        grads_g = gen_tape.gradient(gen_loss, generator.trainable_variables)  # Calculate Generator gradients
        grads_d = disc_tape.gradient(disc_loss, discriminator.trainable_variables)  # Calculate Discriminator gradients
        generator_optimizer.apply_gradients(zip(grads_g, generator.trainable_variables))  # Update Generator weights
        discriminator_optimizer.apply_gradients(zip(grads_d, discriminator.trainable_variables))  # Update Discriminator weights
        return gen_loss, disc_loss  # Return losses for epoch-level monitoring
    return train_step

train_step = make_train_step()  # Create the compiled training function

NUM_EXAMPLES = NUM_CLASSES  # Generate one monitoring image per class
seed_noise = tf.random.normal([NUM_EXAMPLES, LATENT_DIM])  # Fixed monitoring noise
seed_labels = tf.constant(np.arange(NUM_CLASSES).reshape(-1, 1), dtype=tf.int32)  # One label per class

# CHECKPOINT_DIR was defined in the Drive setup cell
os.makedirs(CHECKPOINT_DIR, exist_ok=True)  # Ensure the checkpoint folder exists

print("Checkpoints will be saved to:", CHECKPOINT_DIR)

def generate_and_plot(model, epoch, noise, labels, tag="baseline"):
    predictions = model([noise, labels], training=False)  # Generate previews without training updates
    predictions = (predictions.numpy() * 127.5 + 127.5).astype('uint8')  # Convert [-1, 1] to displayable pixels
    fig, axes = plt.subplots(1, NUM_EXAMPLES, figsize=(20, 3))
    for i in range(NUM_EXAMPLES):
        axes[i].imshow(predictions[i])
        axes[i].set_title(CLASS_NAMES[labels[i, 0].numpy()], fontsize=9)  # Display the requested class
        axes[i].axis('off')
    plt.suptitle(f"[{tag}] Epoch {epoch}")
    plt.tight_layout()
    plt.savefig(f"progress_{tag}_epoch_{epoch:04d}.png")  # Preserve a visual record of training progress
    plt.show()

gen_loss_history = []  # Store mean Generator loss for each epoch
disc_loss_history = []  # Store mean Discriminator loss for each epoch

def train(dataset, epochs, display_every=10, checkpoint_every=10, tag="baseline"):
    training_start_time = time.time()  # Track total training duration
    for epoch in range(epochs):
        epoch_start = time.time()  # Track the duration of the current epoch
        epoch_gen_loss, epoch_disc_loss = [], []
        for image_batch, label_batch in dataset:
            g_loss, d_loss = train_step(image_batch, label_batch)  # Train both networks on one batch
            epoch_gen_loss.append(float(g_loss))
            epoch_disc_loss.append(float(d_loss))

        gen_loss_history.append(np.mean(epoch_gen_loss))  # Record mean Generator loss
        disc_loss_history.append(np.mean(epoch_disc_loss))  # Record mean Discriminator loss
        current_epoch_num = epoch + 1

        if current_epoch_num % display_every == 0 or current_epoch_num == epochs:
            display.clear_output(wait=True)  # Replace the previous preview for cleaner output
            generate_and_plot(generator, current_epoch_num, seed_noise, seed_labels, tag=tag)

        if current_epoch_num % checkpoint_every == 0:
            generator.save_weights(os.path.join(CHECKPOINT_DIR, f"{tag}_generator_epoch_{current_epoch_num:03d}.weights.h5"))  # Save Generator checkpoint
            discriminator.save_weights(os.path.join(CHECKPOINT_DIR, f"{tag}_discriminator_epoch_{current_epoch_num:03d}.weights.h5"))  # Save Discriminator checkpoint

        print(f"Epoch {current_epoch_num}/{epochs}  gen_loss={gen_loss_history[-1]:.4f}  "
              f"disc_loss={disc_loss_history[-1]:.4f}  epoch_time={time.time()-epoch_start:.1f}s  "
              f"elapsed={((time.time()-training_start_time)/60):.1f}min")  # Report progress and timing

Checkpoints will be saved to: /content/drive/MyDrive/ST1504_CA2_PartA/checkpoints


## 8. Train the baseline (50,000 official training images only)

In [ ]:
EPOCHS_TRAINED = 100
train(train_dataset, epochs=EPOCHS_TRAINED, display_every=10, checkpoint_every=10, tag="baseline")

<Figure size 2000x300 with 10 Axes>

Epoch 100/100  gen_loss=2.2108  disc_loss=0.8657  epoch_time=8.5s  elapsed=11.9min


In [ ]:
save_final_gan_weights("baseline")

Generator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline/baseline_generator_final.weights.h5
Discriminator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline/baseline_discriminator_final.weights.h5
Results saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline/baseline_results_log.json


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gen_loss_history, label='Generator loss')
plt.plot(disc_loss_history, label='Discriminator loss')
plt.legend()
plt.title('Baseline cDCGAN losses')
plt.show()

<Figure size 800x500 with 1 Axes>

### Baseline loss-curve interpretation

The baseline cDCGAN does not show the conventional decreasing-loss pattern expected in a standard supervised learning model. Generator loss initially fluctuates before gradually increasing from approximately 1.6–1.8 to above 2.2, while Discriminator loss gradually decreases from around 1.0 to approximately 0.87.

This suggests that the Discriminator becomes progressively more successful at distinguishing real from generated images, making it increasingly difficult for the Generator to fool it. Consequently, the Generator's Binary Cross-Entropy loss increases while the Discriminator loss decreases.

This behaviour is not inherently unusual for a GAN because the Generator and Discriminator are trained adversarially rather than minimising a single shared objective. As one network improves, the task faced by the opposing network can become harder. Therefore, GAN losses are not expected to decrease together, converge to zero, or necessarily become equal.

The curves also remain relatively smooth without explosive oscillations, NaN values or sudden divergence. However, the increasing Generator loss indicates a growing advantage for the Discriminator and should still be treated as a potential sign of adversarial imbalance rather than automatically being considered desirable.

For this reason, the loss curves are used primarily to understand training dynamics. Final model quality is assessed together with the generated images and quantitative metrics such as FID, KID, precision, recall, coverage and class-conditioning accuracy, since adversarial loss alone does not directly measure perceptual image quality or output diversity.

## 9. Evaluation toolkit

We deliberately use **multiple complementary metrics** rather than relying on FID as the primary
or sole indicator, since FID alone cannot distinguish between different failure modes (unrealistic
vs. undiverse output) and is sensitive to sample size and implementation details. FID is included
as one part of the overall picture, not the deciding factor.

| Metric | What it tells us |
|---|---|
| FID / KID | Overall distributional similarity to real images (structure/texture) |
| Precision / Density | Realism -- do generated images look plausible? |
| Recall / Coverage | Diversity -- does the Generator cover the full range of real image variety? |
| Per-class FID/KL | Which specific classes are strong/weak (an overall score hides this) |
| Inception Score | Confidence + diversity, from an independently trained classifier |
| Class-conditioning accuracy | Does the requested class match what the classifier sees? |
| Eye-power (manual) | Human judgement -- grounds all the above in visual reality |

A fixed sample of 5,000 images from the official, held-out test partition is used as the "real"
reference for every metric and every model, ensuring fair, consistent comparisons.

In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from scipy.stats import entropy
from scipy.linalg import sqrtm

FID_KL_SAMPLE_SIZE = 5000  # Use the same evaluation sample size for every model
eval_rng = np.random.default_rng(SEED)  # Make reference sampling reproducible
eval_indices = eval_rng.choice(len(x_test_original), size=FID_KL_SAMPLE_SIZE, replace=False)  # Sample without duplicates
REAL_EVAL_IMAGES = x_test_original[eval_indices]  # Fixed held-out reference images
REAL_EVAL_LABELS = y_test_model[eval_indices]  # Corresponding reference labels

inception_model = InceptionV3(include_top=False, weights="imagenet", pooling="avg", input_shape=(75, 75, 3))  # Extract semantic image features
inception_model.trainable = False  # Use Inception only as a fixed feature extractor


def preprocess_for_inception(images):
    images = tf.convert_to_tensor(images, dtype=tf.float32)  # Ensure the required tensor dtype
    images = tf.image.resize(images, size=(75, 75), method="bilinear")  # Meet Inception's minimum input size
    return preprocess_input(images)  # Apply Inception-specific pixel scaling


def get_activations(images, batch_size=64):
    activations = []
    for start in range(0, len(images), batch_size):  # Process in batches to control memory usage
        batch = preprocess_for_inception(images[start:start + batch_size])
        activations.append(inception_model(batch, training=False).numpy())  # Extract fixed feature vectors
    return np.concatenate(activations, axis=0)


import warnings
from scipy.linalg import sqrtm, LinAlgWarning


def calculate_fid(real_images, fake_images, epsilon=1e-6):
    act_real = get_activations(real_images).astype(np.float64)  # Real-image feature representations
    act_fake = get_activations(fake_images).astype(np.float64)  # Generated-image feature representations

    mu_real, mu_fake = act_real.mean(axis=0), act_fake.mean(axis=0)  # Feature-distribution means
    sigma_real = np.cov(act_real, rowvar=False)  # Real-feature covariance
    sigma_fake = np.cov(act_fake, rowvar=False)  # Generated-feature covariance
    diff = mu_real - mu_fake  # Difference between feature means

    # Attempt the standard FID calculation, watching for a singular-matrix warning
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", LinAlgWarning)
        covmean = sqrtm(sigma_real @ sigma_fake)  # Covariance interaction term

    is_singular = any(issubclass(w.category, LinAlgWarning) for w in caught)

    # Fall back to epsilon-regularized version only if the first attempt was unstable
    if is_singular or not np.isfinite(covmean).all():
        offset = np.eye(sigma_real.shape[0], dtype=np.float64) * epsilon  # Stabilise singular covariance matrices
        covmean = sqrtm((sigma_real + offset) @ (sigma_fake + offset))

    if np.iscomplexobj(covmean):
        if np.max(np.abs(covmean.imag)) > 1e-3:
            raise ValueError("FID covariance square root contains large imaginary components.")
        covmean = covmean.real  # Remove negligible numerical imaginary components

    fid = diff @ diff + np.trace(sigma_real) + np.trace(sigma_fake) - 2.0 * np.trace(covmean)  # Lower is better
    return float(fid)


def calculate_kid(real_images, fake_images, degree=3, gamma=None, coef0=1.0, n_subsets=10, subset_size=100):
    act_real = get_activations(real_images)  # Extract real-image features
    act_fake = get_activations(fake_images)  # Extract generated-image features
    n_features = act_real.shape[1]
    if gamma is None:
        gamma = 1.0 / n_features  # Scale the kernel by feature dimensionality

    def polynomial_kernel(X, Y):
        return (gamma * X.dot(Y.T) + coef0) ** degree  # Measure similarity in feature space

    estimates = []
    n = min(len(act_real), len(act_fake), subset_size)
    for _ in range(n_subsets):  # Average several subsets for a more stable KID estimate
        idx_r = np.random.choice(len(act_real), n, replace=False)
        idx_f = np.random.choice(len(act_fake), n, replace=False)
        X, Y = act_real[idx_r], act_fake[idx_f]
        Kxx, Kyy, Kxy = polynomial_kernel(X, X), polynomial_kernel(Y, Y), polynomial_kernel(X, Y)
        sum_xx = (Kxx.sum() - np.trace(Kxx)) / (n * (n - 1))  # Real-to-real similarity
        sum_yy = (Kyy.sum() - np.trace(Kyy)) / (n * (n - 1))  # Fake-to-fake similarity
        sum_xy = Kxy.mean()  # Real-to-fake similarity
        estimates.append(sum_xx + sum_yy - 2 * sum_xy)
    return float(np.mean(estimates))  # Lower is better


def calculate_pairwise_distances(A, B):
    A_sq = np.sum(A**2, axis=1, keepdims=True)  # Squared magnitudes of A
    B_sq = np.sum(B**2, axis=1, keepdims=True)  # Squared magnitudes of B
    return np.sqrt(np.maximum(A_sq - 2 * A.dot(B.T) + B_sq.T, 0))  # Euclidean distances


def calculate_kth_neighbour_radius(features, k):
    distances = calculate_pairwise_distances(features, features)  # Distances within one feature set
    np.fill_diagonal(distances, np.inf)  # Exclude each sample's distance to itself
    return np.partition(distances, kth=k - 1, axis=1)[:, k - 1]  # Radius to the kth neighbour


def calculate_precision_recall(real_images, fake_images, k=3):
    act_real = get_activations(real_images)  # Real-data feature manifold
    act_fake = get_activations(fake_images)  # Generated-data feature manifold
    real_radii = calculate_kth_neighbour_radius(act_real, k)
    fake_radii = calculate_kth_neighbour_radius(act_fake, k)
    fake_to_real = calculate_pairwise_distances(act_fake, act_real)
    precision = np.mean(np.any(fake_to_real <= real_radii[np.newaxis, :], axis=1))  # Generated-sample fidelity
    real_to_fake = fake_to_real.T
    recall = np.mean(np.any(real_to_fake <= fake_radii[np.newaxis, :], axis=1))  # Real-distribution coverage
    return float(precision), float(recall)


def calculate_density_coverage(real_images, fake_images, k=5):
    act_real = get_activations(real_images)  # Extract real-image features
    act_fake = get_activations(fake_images)  # Extract generated-image features
    real_radii = calculate_kth_neighbour_radius(act_real, k)
    fake_to_real = calculate_pairwise_distances(act_fake, act_real)
    density = np.mean(np.sum(fake_to_real <= real_radii[np.newaxis, :], axis=1) / k)  # Local sample concentration
    min_fake_dist_per_real = np.min(fake_to_real, axis=0)
    coverage = np.mean(min_fake_dist_per_real <= real_radii)  # Proportion of real manifold represented
    return float(density), float(coverage)


def pixel_histogram(images, bins=50):
    hist, _ = np.histogram(np.asarray(images).reshape(-1), bins=bins, range=(0, 255), density=False)  # Overall pixel distribution
    hist = hist.astype(np.float64) + 1e-8  # Prevent zero probabilities in KL
    return hist / hist.sum()  # Convert counts into probabilities


def calculate_pixel_kl(real_images, fake_images):
    return float(entropy(pixel_histogram(real_images), pixel_histogram(fake_images)))  # Compare low-level pixel distributions


def generate_conditional_images(generator, latent_dim, number_of_images, labels=None, seed=SEED, batch_size=128):
    rng = np.random.default_rng(seed)  # Reproduce labels and generated samples
    if labels is None:
        labels = rng.integers(0, NUM_CLASSES, size=number_of_images, dtype=np.int32)  # Sample requested classes
    else:
        labels = np.asarray(labels, dtype=np.int32).reshape(-1)  # Standardise supplied labels

    batches = []
    for start in range(0, number_of_images, batch_size):  # Generate in batches to control memory usage
        end = min(start + batch_size, number_of_images)
        n = end - start
        noise = tf.random.stateless_normal(shape=(n, latent_dim), seed=[seed, start], dtype=tf.float32)  # Reproducible latent noise
        batch_labels = tf.reshape(tf.convert_to_tensor(labels[start:end], dtype=tf.int32), [-1, 1])  # Match model input shape
        generated = generator([noise, batch_labels], training=False)  # Generate without updating weights
        generated = tf.clip_by_value((generated + 1.0) / 2.0, 0.0, 1.0)  # Convert [-1, 1] to [0, 1]
        generated = tf.cast(tf.round(generated * 255.0), tf.uint8)  # Convert to displayable RGB pixels
        batches.append(generated.numpy())
    return np.concatenate(batches, axis=0), labels


def per_class_evaluation(generator, latent_dim, real_test_images, real_test_labels, n_samples_per_class=200, seed=SEED):
    per_class_fid, per_class_kl = {}, {}  # Store class-level results separately
    rng = np.random.default_rng(seed)
    for class_id in range(NUM_CLASSES):
        real_class_images = real_test_images[real_test_labels == class_id]  # Select real images from one class
        n_use = min(n_samples_per_class, len(real_class_images))  # Avoid requesting more real samples than available
        real_sample = real_class_images[rng.choice(len(real_class_images), n_use, replace=False)]  # Fixed real-class sample
        requested_labels = np.full(n_use, class_id, dtype=np.int32)  # Request only the current class
        gen_images, _ = generate_conditional_images(generator, latent_dim, n_use, labels=requested_labels, seed=seed + class_id)

        class_fid = calculate_fid(real_sample, gen_images)  # Measure class-specific feature similarity
        class_kl = calculate_pixel_kl(real_sample, gen_images)  # Measure class-specific pixel similarity
        per_class_fid[CLASS_NAMES[class_id]] = class_fid
        per_class_kl[CLASS_NAMES[class_id]] = class_kl
        print(f"{CLASS_NAMES[class_id]:12s} | FID: {class_fid:9.2f} | Pixel KL: {class_kl:.4f}")
    return per_class_fid, per_class_kl

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


## 10. Auxiliary classifier (for Inception Score and class-conditioning accuracy)

Trained only on the official training partition; its own test accuracy is reported so that a
low conditioning score can be correctly attributed to Generator weakness rather than classifier
weakness.

In [ ]:
def build_cifar_classifier():
    model = tf.keras.Sequential([
        layers.Input(shape=IMAGE_SHAPE),  # Accept preprocessed 32x32 RGB images
        layers.Conv2D(32, 3, activation='relu', padding='same'),  # Learn low-level visual features
        layers.BatchNormalization(),  # Stabilise activations during training
        layers.Conv2D(32, 3, activation='relu', padding='same'),  # Refine low-level features
        layers.MaxPooling2D(),  # Reduce spatial size while retaining strong features
        layers.Dropout(0.25),  # Reduce overfitting

        layers.Conv2D(64, 3, activation='relu', padding='same'),  # Learn more complex features
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),  # Refine intermediate features
        layers.MaxPooling2D(),  # Further reduce spatial dimensions
        layers.Dropout(0.30),

        layers.Conv2D(128, 3, activation='relu', padding='same'),  # Learn higher-level class features
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),  # Summarise each feature map with fewer parameters
        layers.Dense(128, activation='relu'),  # Combine extracted features for classification
        layers.Dropout(0.30),
        layers.Dense(NUM_CLASSES, activation='softmax')  # Output probabilities for all CIFAR-10 classes
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),  # Optimise classifier parameters
        loss='sparse_categorical_crossentropy',  # Suitable for integer class labels
        metrics=['accuracy']
    )

    return model


cifar_classifier = build_cifar_classifier()  # Build an independent classifier for GAN evaluation

classifier_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",  # Monitor performance on unseen validation images
    patience=4,  # Stop after four epochs without improvement
    mode="max",
    restore_best_weights=True  # Retain the best validation-performing weights
)


classifier_history = cifar_classifier.fit(
    x_train_gan, y_train_model, epochs=20, batch_size=128,
    validation_split=0.1, shuffle=True, callbacks=[classifier_early_stopping], verbose=1
)  # Train only on the official CIFAR-10 training partition


classifier_test_loss, classifier_test_accuracy = cifar_classifier.evaluate(
    x_test_gan,
    y_test_model,
    verbose=0
)  # Verify classifier reliability on the held-out test set

print(f"\nClassifier test accuracy (held-out official test set): {classifier_test_accuracy:.4f}")


def class_conditioning_accuracy(generator, latent_dim, classifier, n_samples_per_class=100, seed=SEED):
    per_class_accuracy = {}  # Store conditional accuracy for each class
    total_correct, total_samples = 0, 0

    for class_id in range(NUM_CLASSES):
        requested_labels = np.full(n_samples_per_class, class_id, dtype=np.int32)  # Request one class at a time

        gen_images, _ = generate_conditional_images(
            generator,
            latent_dim,
            n_samples_per_class,
            labels=requested_labels,
            seed=seed + class_id
        )  # Generate an equal number of images for each class

        gen_gan_range = gen_images.astype(np.float32) / 127.5 - 1.0  # Match the classifier's training range
        predictions = classifier.predict(gen_gan_range, verbose=0)  # Predict generated-image classes
        predicted_classes = np.argmax(predictions, axis=1)  # Select the highest-probability class
        correct = np.sum(predicted_classes == class_id)  # Compare predictions with requested labels
        per_class_accuracy[CLASS_NAMES[class_id]] = float(correct / n_samples_per_class)
        total_correct += correct
        total_samples += n_samples_per_class

    return float(total_correct / total_samples), per_class_accuracy  # Return overall and per-class accuracy


def classifier_based_inception_score(generator, latent_dim, classifier, n_samples=1000, splits=10, seed=SEED):
    gen_images, _ = generate_conditional_images(generator, latent_dim, n_samples, seed=seed)  # Generate evaluation images
    gen_gan_range = gen_images.astype(np.float32) / 127.5 - 1.0  # Match the classifier's expected range
    predictions = np.clip(classifier.predict(gen_gan_range, verbose=0), 1e-8, 1.0)  # Avoid log(0)
    split_scores = []
    split_size = n_samples // splits  # Divide samples to estimate score variability

    for i in range(splits):
        part = predictions[i * split_size:(i + 1) * split_size]  # Select one evaluation split
        marginal = np.mean(part, axis=0)  # Estimate the overall class distribution
        kl_divs = np.sum(part * (np.log(part) - np.log(marginal)), axis=1)  # Combine class clarity and diversity
        split_scores.append(np.exp(np.mean(kl_divs)))

    return float(np.mean(split_scores)), float(np.std(split_scores))  # Return mean score and variation

Epoch 1/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 25s 38ms/step - accuracy: 0.4602 - loss: 1.4685 - val_accuracy: 0.1676 - val_loss: 2.6844
Epoch 2/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6138 - loss: 1.0911 - val_accuracy: 0.6760 - val_loss: 0.9392
Epoch 3/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6612 - loss: 0.9533 - val_accuracy: 0.6848 - val_loss: 0.9003
Epoch 4/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6964 - loss: 0.8610 - val_accuracy: 0.7094 - val_loss: 0.8202
Epoch 5/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7241 - loss: 0.7867 - val_accuracy: 0.7486 - val_loss: 0.7161
Epoch 6/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7436 - loss: 0.7259 - val_accuracy: 0.7186 - val_loss: 0.8493
Epoch 7/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7628 - loss: 0.6811 - val_accuracy: 0.7694 - val_loss: 0.6695
Epoch 8/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7756 - loss: 0.6370 - val_accuracy: 

## 11. Results logging

In [ ]:
def log_results(model_name, overall_fid=None, overall_kl=None, overall_kid=None,
                precision=None, recall=None, density=None, coverage=None,
                per_class_fid=None, per_class_kl=None,
                inception_score=None, class_cond_accuracy=None, epochs_trained=None, notes=""):
    results_log[model_name] = {  # Store all metrics under the model's unique name
        "overall_fid": overall_fid, "overall_kl": overall_kl, "overall_kid": overall_kid,
        "precision": precision, "recall": recall, "density": density, "coverage": coverage,
        "per_class_fid": per_class_fid, "per_class_kl": per_class_kl,
        "inception_score": inception_score, "class_cond_accuracy": class_cond_accuracy,
        "epochs_trained": epochs_trained, "notes": notes,
    }
    print(f"Logged results for '{model_name}'.")  # Confirm that the entry was recorded


def show_results_table():
    print(f"{'Model':22s} {'Epochs':7s} {'FID':9s} {'KID':9s} {'Prec':7s} {'Rec':7s} "
          f"{'Dens':7s} {'Cov':7s} {'IS':16s} {'ClassAcc':9s} Notes")  # Print table headings
    print("-" * 150)

    for name, r in results_log.items():  # Format every logged experiment consistently
        epochs = r['epochs_trained'] if r['epochs_trained'] is not None else "-"
        fid = f"{r['overall_fid']:.2f}" if r['overall_fid'] is not None else "-"
        kid = f"{r['overall_kid']:.4f}" if r['overall_kid'] is not None else "-"
        prec = f"{r['precision']:.3f}" if r['precision'] is not None else "-"
        rec = f"{r['recall']:.3f}" if r['recall'] is not None else "-"
        dens = f"{r['density']:.3f}" if r.get('density') is not None else "-"
        cov = f"{r['coverage']:.3f}" if r.get('coverage') is not None else "-"
        is_str = f"{r['inception_score'][0]:.2f}+/-{r['inception_score'][1]:.2f}" if r['inception_score'] else "-"
        class_acc = f"{r['class_cond_accuracy']:.1%}" if r['class_cond_accuracy'] is not None else "-"

        print(f"{name:22s} {str(epochs):7s} {fid:9s} {kid:9s} {prec:7s} {rec:7s} "
              f"{dens:7s} {cov:7s} {is_str:16s} {class_acc:9s} {r['notes']}")  # Display one comparison row

## 12. Full evaluation for baseline cDCGAN

FID is reported here as **one metric among several**, not the primary or deciding indicator, per
guidance that FID alone should not be treated as the main criterion for judging model quality.

### 12.1 Overall distribution metrics

In [ ]:
gen_images, _ = generate_conditional_images(generator, LATENT_DIM, FID_KL_SAMPLE_SIZE, seed=SEED)  # Generate a reproducible baseline evaluation set

baseline_fid = calculate_fid(REAL_EVAL_IMAGES, gen_images)  # Measure real–generated feature distance
baseline_kid = calculate_kid(REAL_EVAL_IMAGES, gen_images)  # Provide a complementary kernel-based estimate
baseline_precision, baseline_recall = calculate_precision_recall(REAL_EVAL_IMAGES, gen_images)  # Measure fidelity and diversity
baseline_density, baseline_coverage = calculate_density_coverage(REAL_EVAL_IMAGES, gen_images)  # Assess quality and real-data coverage
baseline_pixel_kl = calculate_pixel_kl(REAL_EVAL_IMAGES, gen_images)  # Compare overall pixel distributions

print(f"FID:       {baseline_fid:.2f}")
print(f"KID:       {baseline_kid:.4f}")
print(f"Precision: {baseline_precision:.3f}")
print(f"Recall:    {baseline_recall:.3f}")
print(f"Density:   {baseline_density:.3f}")
print(f"Coverage:  {baseline_coverage:.3f}")
print(f"Pixel KL:  {baseline_pixel_kl:.4f}")  # Display the complete baseline evaluation

FID:       170.29
KID:       0.0893
Precision: 0.683
Recall:    0.389
Density:   1.378
Coverage:  0.953
Pixel KL:  0.0033


### 12.2 Class-conditioning accuracy


In [ ]:
baseline_conditioning_accuracy, baseline_per_class_accuracy = class_conditioning_accuracy(
    generator, LATENT_DIM, cifar_classifier
)
print(f"Overall class-conditioning accuracy: {baseline_conditioning_accuracy:.1%}\n")
for class_name, acc in baseline_per_class_accuracy.items():
    print(f"{class_name:12s}: {acc:.1%}")

Overall class-conditioning accuracy: 64.4%

airplane    : 71.0%
automobile  : 65.0%
bird        : 61.0%
cat         : 33.0%
deer        : 75.0%
dog         : 37.0%
frog        : 92.0%
horse       : 60.0%
ship        : 81.0%
truck       : 69.0%


### 12.3 Classifier-based Inception Score


In [ ]:
baseline_is_mean, baseline_is_std = classifier_based_inception_score(generator, LATENT_DIM, cifar_classifier)
print(f"Classifier-based diversity score: {baseline_is_mean:.3f} +/- {baseline_is_std:.3f}")

Classifier-based diversity score: 4.737 +/- 0.344


## 13. Per-class FID and KL

In [ ]:
baseline_per_class_fid, baseline_per_class_kl = per_class_evaluation(
    generator, LATENT_DIM, x_test_original, y_test_model,
    n_samples_per_class = 1000
)

plt.figure(figsize=(10, 5))
plt.bar(baseline_per_class_fid.keys(), baseline_per_class_fid.values(), color='steelblue')
plt.title("Per-class FID -- baseline cDCGAN")
plt.ylabel("FID"); plt.xticks(rotation=45)
plt.show()

airplane     | FID:    230.80 | Pixel KL: 0.0113
automobile   | FID:    245.44 | Pixel KL: 0.0126
bird         | FID:    548.08 | Pixel KL: 0.0169
cat          | FID:    476.75 | Pixel KL: 0.0063
deer         | FID:    222.50 | Pixel KL: 0.0042
dog          | FID:    966.65 | Pixel KL: 0.0026
frog         | FID:    251.44 | Pixel KL: 0.0200
horse        | FID:    416.03 | Pixel KL: 0.0041
ship         | FID:    214.10 | Pixel KL: 0.0156
truck        | FID:    196.89 | Pixel KL: 0.0085


<Figure size 1000x500 with 1 Axes>

## 14. Labeled generated-image grid

### 14.1 Generated samples by class


In [ ]:
def show_labeled_generation_grid(generator, latent_dim, class_names, samples_per_class=8,
                                 title="Generated samples by class", seed=SEED):
    num_classes = len(class_names)  # Determine the number of class rows
    labels = np.repeat(np.arange(num_classes, dtype=np.int32), samples_per_class)  # Request equal samples per class
    images, _ = generate_conditional_images(generator, latent_dim, len(labels), labels=labels, seed=seed)  # Generate reproducible conditional samples

    fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(2.3 * samples_per_class, 2.6 * num_classes))  # One class per row
    for class_id in range(num_classes):
        for sample_id in range(samples_per_class):
            idx = class_id * samples_per_class + sample_id  # Locate the current class sample
            axes[class_id, sample_id].imshow(images[idx])
            axes[class_id, sample_id].axis("off")
            axes[class_id, sample_id].set_title(class_names[class_id], fontsize=9)  # Display the class above each image
        axes[class_id, 0].set_ylabel(class_names[class_id], fontsize=10, rotation=0, labelpad=45, va="center")  # Label each row

    plt.suptitle(title, fontsize=16, y=1.005)
    plt.tight_layout()
    plt.show()


show_labeled_generation_grid(generator, LATENT_DIM, CLASS_NAMES, samples_per_class=8,
                             title="Baseline cDCGAN -- Generated Samples by Class")  # Inspect baseline quality across all classes

<Figure size 1840x2600 with 80 Axes>

### 14.2 Real-versus-generated comparison


In [ ]:
def show_real_vs_generated(generator, latent_dim, real_images, real_labels, class_names,
                           samples_per_class=5, seed=SEED, title="Real vs Generated"):
    num_classes = len(class_names)  # Determine the number of comparison rows
    fig, axes = plt.subplots(num_classes, samples_per_class * 2, figsize=(2.2 * samples_per_class * 2, 2.2 * num_classes))  # Real images on the left and generated images on the right
    rng = np.random.default_rng(seed)  # Make real-image sampling reproducible

    for class_id in range(num_classes):
        real_class_imgs = real_images[real_labels == class_id]  # Select real test images from the current class
        real_sample = real_class_imgs[rng.choice(len(real_class_imgs), samples_per_class, replace=False)]  # Sample distinct real examples
        gen_labels = np.full(samples_per_class, class_id, dtype=np.int32)  # Request generated images from the same class
        gen_images, _ = generate_conditional_images(generator, latent_dim, samples_per_class, labels=gen_labels, seed=seed + class_id)  # Generate reproducible class-conditioned samples

        for j in range(samples_per_class):
            axes[class_id, j].imshow(real_sample[j]); axes[class_id, j].axis("off")  # Display real samples on the left
            if j == 0:
                axes[class_id, j].set_ylabel(class_names[class_id], fontsize=10, rotation=0, labelpad=45, va="center")  # Label each class row

        for j in range(samples_per_class):
            axes[class_id, samples_per_class + j].imshow(gen_images[j]); axes[class_id, samples_per_class + j].axis("off")  # Display generated samples on the right

    fig.text(0.27, 1.01, "REAL", ha="center", fontsize=14, fontweight="bold")  # Label the real-image section
    fig.text(0.73, 1.01, "GENERATED", ha="center", fontsize=14, fontweight="bold")  # Label the generated-image section
    plt.suptitle(title, fontsize=16, y=1.04)
    plt.tight_layout()
    plt.show()


show_real_vs_generated(generator, LATENT_DIM, x_test_original, y_test_model, CLASS_NAMES,
                       samples_per_class=5, title="Baseline cDCGAN -- Real vs Generated")  # Compare baseline outputs directly with real test images

<Figure size 2200x2200 with 100 Axes>

## 15. Eye-power qualitative assessment

The generated images were manually inspected against real CIFAR-10 samples from the corresponding classes. A class was rated **Clear** when most generated samples contained recognisable class-specific shapes, **Marginal** when some class-specific features were visible but the object remained inconsistent or ambiguous, and **Nonsense** when most samples lacked recognisable class structure.

| Class | Clear | Marginal | Nonsense | Notes |
|---|---:|---:|---:|---|
| airplane | ✓ | | | Most samples contain recognisable aircraft silhouettes, wings and sky-like backgrounds, although some shapes remain distorted. |
| automobile | ✓ | | | Most samples have recognisable car bodies and road-like backgrounds, although fine details remain blurry. |
| bird | | ✓ | | Some samples contain bird-like silhouettes, bodies or beaks, but several remain ambiguous or resemble other animals. |
| cat | | ✓ | | Several samples contain feline-like faces, ears or body shapes, but the class is not consistently recognisable across the generated samples. |
| deer | | ✓ | | Animal shapes and natural backgrounds are present, but several samples are structurally ambiguous and could be confused with other four-legged animals. |
| dog | | ✓ | | Several samples contain dog-like faces or bodies, but class-specific structure is inconsistent and some resemble other animals. |
| frog | | | ✓ | Green and brown natural textures are common, but most samples lack a clearly recognisable frog body structure. |
| horse | ✓ | | | Most samples contain recognisable horse-like bodies, legs and outdoor backgrounds. |
| ship | ✓ | | | Most samples clearly resemble boats or ships and contain appropriate water or sky backgrounds. |
| truck | | ✓ | | Several samples contain large vehicle-like structures, but some are distorted or visually closer to automobiles. |

Overall, the baseline cDCGAN successfully learned broad class-level characteristics such as colour, background and approximate object shape. It performed particularly well for structurally distinctive classes such as airplanes, automobiles, horses and ships. However, visually similar animal classes such as birds, cats, deer and dogs showed greater ambiguity, while frogs were particularly difficult to generate convincingly.

These observations broadly agree with the earlier EDA, where greater visual overlap and within-class variation were identified among several animal categories. The baseline therefore demonstrates successful class conditioning, but still has difficulty learning fine-grained object structure for visually complex or overlapping classes.

## 16. Log results and justify performance

### 16.1 Record baseline results


In [ ]:
log_results(
    model_name="baseline_cDCGAN",
    overall_fid=baseline_fid, overall_kl=baseline_pixel_kl, overall_kid=baseline_kid,
    precision=baseline_precision, recall=baseline_recall, density=baseline_density, coverage=baseline_coverage,
    per_class_fid=baseline_per_class_fid, per_class_kl=baseline_per_class_kl,
    inception_score=(baseline_is_mean, baseline_is_std), class_cond_accuracy=baseline_conditioning_accuracy,
    epochs_trained=EPOCHS_TRAINED,
    notes="Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage."
)
show_results_table()

Logged results for 'baseline_cDCGAN'.
Model                  Epochs  FID       KID       Prec    Rec     Dens    Cov     IS               ClassAcc  Notes
------------------------------------------------------------------------------------------------------------------------------------------------------
baseline_cDCGAN        100     170.29    0.0893    0.683   0.389   1.378   0.953   4.74+/-0.34      64.4%     Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage.


### 16.2 Balanced evaluation sampling


In [ ]:
def generate_balanced_evaluation_images(generator, latent_dim, total_images, seed=SEED):
    """Generates an evaluation set with exactly equal representation of every class,
    rather than randomly-sampled labels."""
    images_per_class = total_images // NUM_CLASSES
    actual_total = images_per_class * NUM_CLASSES   # equals total_images exactly when evenly divisible

    labels = np.repeat(np.arange(NUM_CLASSES, dtype=np.int32), images_per_class)
    images, labels_used = generate_conditional_images(
        generator, latent_dim, actual_total, labels=labels, seed=seed
    )
    return images, labels_used



### 16.3 Evaluation sanity checks


In [ ]:
test_images, test_labels = generate_balanced_evaluation_images(generator, LATENT_DIM, FID_KL_SAMPLE_SIZE)
print("Generated images shape:", test_images.shape)   # should print (5000, 32, 32, 3)
print("Generated labels shape:", test_labels.shape)     # should print (5000,)
unique, counts = np.unique(test_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"{CLASS_NAMES[u]:12s}: {c}")   # every class should show exactly 500

Generated images shape: (5000, 32, 32, 3)
Generated labels shape: (5000,)
airplane    : 500
automobile  : 500
bird        : 500
cat         : 500
deer        : 500
dog         : 500
frog        : 500
horse       : 500
ship        : 500
truck       : 500


In [ ]:
test_pred = cifar_classifier.predict(x_test_gan[:5], verbose=0)
print("Classifier still works, output shape:", test_pred.shape)   # should print (5, 10)

Classifier still works, output shape: (5, 10)


In [ ]:
for name in ['generator', 'discriminator', 'cifar_classifier', 'OUTPUT_ROOT', 'CLASS_NAMES', 'NUM_CLASSES', 'x_train_gan', 'y_train_model']:
    print(f"{name}: {'OK' if name in globals() else 'MISSING'}")

generator: OK
discriminator: OK
cifar_classifier: OK
OUTPUT_ROOT: OK
CLASS_NAMES: OK
NUM_CLASSES: OK
x_train_gan: OK
y_train_model: OK


In [ ]:
print("OUTPUT_ROOT remains:", OUTPUT_ROOT)

OUTPUT_ROOT remains: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models


## 17. TTUR: Asymmetric Learning-Rate Stabilisation

### Purpose

This is our first attempt at a *targeted* improvement, and the specific problem it targets comes
directly from what we observed in our baseline's own training curve (Section 8), not from theory alone. Over 100 epochs, our baseline's Generator and Discriminator losses showed a gradual, monotonic divergence: Generator loss rose steadily while Discriminator loss fell steadily,
suggesting the Discriminator was slowly gaining an advantage over the course of training. We
chose to investigate this specific, empirically-observed pattern before moving to any more
structurally invasive change (architecture, conditioning mechanism, etc.), on the reasoning that
if a training-dynamics problem exists, it should be addressed directly before layering further
complexity on top of a potentially unstable foundation.

The Two Time-Scale Update Rule (TTUR), proposed by Heusel et al. (2017) — the same paper that
introduced FID as an evaluation metric — offers a direct, theoretically-grounded intervention for
exactly this failure mode: assigning the Discriminator a *slower* learning rate than the
Generator, so that it updates more conservatively and is less likely to outpace the Generator
during training. This is precisely the mechanism we need to test against our observed
Generator/Discriminator imbalance: if the imbalance is the actual cause of any quality
limitations in our baseline, slowing the Discriminator down should measurably improve our
evaluation metrics, not just visually flatten the loss curve.

### Why we treat this as a decision point, not just an experiment to log

we treat this experiment (and each one that follows) as a genuine decision point:
if TTUR improves our quantitative metrics (FID, Recall, class-conditioning accuracy) relative to
the baseline, we adopt it and build subsequent experiments on top of the TTUR configuration
instead. If it does not, we discard it and continue building on our original baseline
configuration. This is important to state explicitly, because it is possible for a technique to
produce a more "well-behaved" loss curve without actually improving image quality — in which case
the loss curve alone would be a misleading basis for a decision, and only the quantitative
evaluation metrics (Section 17.3) can correctly determine whether this change should be kept.

### 17.1 Setup and training

We use the identical baseline architecture, dataset, batch size, and one-sided label smoothing
configuration (Section 5-6), changing only the Discriminator's learning rate from 2e-4 to 1e-4,
while the Generator's learning rate remains unchanged at 2e-4. This isolates the effect of the
learning-rate asymmetry itself as the single variable under test, consistent with the
single-variable ablation methodology used throughout this notebook.

In [ ]:
# ============================================================
# TTUR-INSPIRED STABILISATION: SETUP
# ============================================================

import os
import json
import time
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

TTUR_MODEL_NAME = "baseline_ttur_dlr1e4"
TTUR_DIR = os.path.join(
    OUTPUT_ROOT,
    TTUR_MODEL_NAME
)

os.makedirs(TTUR_DIR, exist_ok=True)

# Check that the required components already exist.
required_objects = [
    "x_train_gan",
    "y_train_model",
    "REAL_EVAL_IMAGES",
    "build_generator",
    "build_discriminator",
    "make_train_step",
    "make_gan_dataset",
    "train",
    "cifar_classifier"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the earlier notebook cells first. Missing: "
        + ", ".join(missing_objects)
    )

print("TTUR experiment directory:", TTUR_DIR)
print("Evaluation classifier remains loaded:", cifar_classifier.name)

TTUR experiment directory: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline_ttur_dlr1e4
Evaluation classifier remains loaded: sequential


In [ ]:
# ============================================================
# Preserve original baseline histories, when available
# ============================================================

if (
    "gen_loss_history" in globals()
    and "disc_loss_history" in globals()
    and len(gen_loss_history) > 0
):
    original_baseline_gen_history = list(gen_loss_history)
    original_baseline_disc_history = list(disc_loss_history)

    print(
        "Original baseline loss histories preserved:",
        len(original_baseline_gen_history),
        "epochs"
    )
else:
    original_baseline_gen_history = None
    original_baseline_disc_history = None

    print(
        "Original baseline loss histories are not in memory. "
        "The TTUR loss curve can still be plotted independently."
    )

Original baseline loss histories preserved: 100 epochs


In [ ]:
# ============================================================
# Train TTUR-inspired cDCGAN
# ============================================================

LATENT_DIM = 128
BATCH_SIZE = 128
TTUR_EPOCHS = 100

TTUR_GENERATOR_LR = 2e-4
TTUR_DISCRIMINATOR_LR = 1e-4

# Reset random seeds, but do not clear the Keras session.
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

# Build fresh GAN networks.
generator = build_generator(
    latent_dim=LATENT_DIM
)

discriminator = build_discriminator()

# Generator learning rate remains unchanged.
generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=TTUR_GENERATOR_LR,
    beta_1=0.5
)

# Targeted reduction of the Discriminator learning rate.
discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=TTUR_DISCRIMINATOR_LR,
    beta_1=0.5
)

# Recreate the compiled tf.function for these new objects.
train_step = make_train_step()

# Reset histories for this experiment.
gen_loss_history = []
disc_loss_history = []

# Fixed noise and labels for consistent progress images.
NUM_EXAMPLES = NUM_CLASSES

seed_noise = tf.random.normal(
    shape=[NUM_EXAMPLES, LATENT_DIM],
    seed=SEED
)

seed_labels = tf.constant(
    np.arange(NUM_CLASSES).reshape(-1, 1),
    dtype=tf.int32
)

# Build the same training dataset used by the baseline.
ttur_dataset = make_gan_dataset(
    images=x_train_gan,
    labels=y_train_model,
    batch_size=BATCH_SIZE,
    seed=SEED
)

print("Generator learning rate:", TTUR_GENERATOR_LR)
print("Discriminator learning rate:", TTUR_DISCRIMINATOR_LR)
print("Batch size:", BATCH_SIZE)
print("Latent dimension:", LATENT_DIM)
print("Training epochs:", TTUR_EPOCHS)

# Your train() function does not use checkpoint_every.
train(
    dataset=ttur_dataset,
    epochs=TTUR_EPOCHS,
    display_every=25,
    tag=TTUR_MODEL_NAME
)

<Figure size 2000x300 with 10 Axes>

Epoch 100/100  gen_loss=1.5718  disc_loss=1.0199  epoch_time=9.1s  elapsed=11.7min


### 17.2 Training diagnostics and saved artefacts

#### 17.2.1 TTUR loss curves


In [ ]:
# ============================================================
# Plot TTUR training losses
# ============================================================

plt.figure(figsize=(9, 5))  # Create a clear figure for both loss curves

plt.plot(
    range(1, len(gen_loss_history) + 1),
    gen_loss_history,
    label="Generator loss"
)  # Plot the mean Generator loss for each epoch

plt.plot(
    range(1, len(disc_loss_history) + 1),
    disc_loss_history,
    label="Discriminator loss"
)  # Plot the mean Discriminator loss for each epoch

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    "cDCGAN with Reduced Discriminator Learning Rate\n"
    "G LR = 0.0002, D LR = 0.0001"
)  # State the asymmetric TTUR-inspired learning rates
plt.legend()  # Distinguish the two adversarial losses
plt.grid(alpha=0.3)  # Improve readability without obscuring the curves
plt.tight_layout()

ttur_loss_plot_path = os.path.join(
    TTUR_DIR,
    "ttur_loss_curve.png"
)  # Save the plot with the other TTUR artifacts

plt.savefig(
    ttur_loss_plot_path,
    dpi=200,
    bbox_inches="tight"
)  # Export a high-resolution figure for the report

plt.show()

print("Loss curve saved to:", ttur_loss_plot_path)  # Confirm the saved location

<Figure size 900x500 with 1 Axes>

Loss curve saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline_ttur_dlr1e4/ttur_loss_curve.png


#### 17.2.2 Baseline-versus-TTUR loss comparison

In [ ]:
# ============================================================
# Compare original baseline and TTUR loss curves
# ============================================================

if (
    original_baseline_gen_history is not None
    and original_baseline_disc_history is not None
):  # Only compare when both baseline histories are available
    plt.figure(figsize=(10, 6))

    plt.plot(
        range(1, len(original_baseline_gen_history) + 1),
        original_baseline_gen_history,
        label="Original baseline — Generator",
        linestyle="--"
    )  # Dashed line distinguishes the baseline Generator

    plt.plot(
        range(1, len(original_baseline_disc_history) + 1),
        original_baseline_disc_history,
        label="Original baseline — Discriminator",
        linestyle="--"
    )  # Dashed line distinguishes the baseline Discriminator

    plt.plot(
        range(1, len(gen_loss_history) + 1),
        gen_loss_history,
        label="Reduced D LR — Generator"
    )  # Plot the TTUR Generator loss

    plt.plot(
        range(1, len(disc_loss_history) + 1),
        disc_loss_history,
        label="Reduced D LR — Discriminator"
    )  # Plot the TTUR Discriminator loss

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(
        "Original Baseline vs Reduced Discriminator Learning Rate"
    )
    plt.legend()  # Identify all four curves
    plt.grid(alpha=0.3)  # Improve readability
    plt.tight_layout()

    comparison_loss_path = os.path.join(
        TTUR_DIR,
        "original_vs_ttur_loss_curves.png"
    )  # Store the comparison with the other TTUR artifacts

    plt.savefig(
        comparison_loss_path,
        dpi=200,
        bbox_inches="tight"
    )  # Export a high-resolution copy for reporting

    plt.show()

    print(
        "Comparison plot saved to:",
        comparison_loss_path
    )

else:
    print(
        "Original histories were unavailable, "
        "so only the TTUR loss curve was plotted."
    )  # Explain why the direct comparison was skipped

<Figure size 1000x600 with 1 Axes>

Comparison plot saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline_ttur_dlr1e4/original_vs_ttur_loss_curves.png


#### 17.2.3 Save TTUR weights and training history


In [ ]:
# ============================================================
# Save TTUR model weights and training history
# ============================================================

ttur_generator_path = os.path.join(
    TTUR_DIR,
    "baseline_ttur_generator_final.weights.h5"
)  # Define the final TTUR Generator weight path

ttur_discriminator_path = os.path.join(
    TTUR_DIR,
    "baseline_ttur_discriminator_final.weights.h5"
)  # Define the matching Discriminator weight path

generator.save_weights(
    ttur_generator_path
)  # Save the trained TTUR Generator

discriminator.save_weights(
    ttur_discriminator_path
)  # Save the trained TTUR Discriminator

ttur_history = pd.DataFrame({
    "epoch": np.arange(1, TTUR_EPOCHS + 1),
    "generator_loss": gen_loss_history,
    "discriminator_loss": disc_loss_history
})  # Organise the epoch-level losses into a table

ttur_history_path = os.path.join(
    TTUR_DIR,
    "baseline_ttur_training_history.csv"
)  # Define the training-history output path

ttur_history.to_csv(
    ttur_history_path,
    index=False
)  # Save the losses for later plotting and analysis

# Preserve references before building any other models.
ttur_generator = generator  # Retain access to the trained TTUR Generator
ttur_discriminator = discriminator  # Retain access to its Discriminator

print("TTUR Generator:", ttur_generator_path)
print("TTUR Discriminator:", ttur_discriminator_path)
print("Training history:", ttur_history_path)

TTUR Generator: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline_ttur_dlr1e4/baseline_ttur_generator_final.weights.h5
TTUR Discriminator: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline_ttur_dlr1e4/baseline_ttur_discriminator_final.weights.h5
Training history: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline_ttur_dlr1e4/baseline_ttur_training_history.csv


### 17.3 Quantitative and qualitative evaluation

#### 17.3.1 Class-conditioned sample grid


In [ ]:
# ============================================================
# Qualitative class-conditioned image grid
# ============================================================

def show_generated_class_grid(
    model,
    latent_dim,
    samples_per_class=8,
    seed=SEED,
    title="Generated Images",
    save_path=None
):
    rng = np.random.default_rng(seed)  # Use reproducible latent noise

    labels = np.repeat(
        np.arange(NUM_CLASSES),
        samples_per_class
    ).reshape(-1, 1).astype(np.int32)  # Request an equal number of samples from every class

    noise = rng.normal(
        size=(len(labels), latent_dim)
    ).astype(np.float32)  # Sample one latent vector for each requested label

    generated_images = model(
        [
            tf.convert_to_tensor(noise),
            tf.convert_to_tensor(labels)
        ],
        training=False
    ).numpy()  # Generate images without updating model weights

    generated_images = np.clip(
        generated_images * 127.5 + 127.5,
        0,
        255
    ).astype(np.uint8)  # Convert outputs from [-1, 1] to displayable pixels

    fig, axes = plt.subplots(
        NUM_CLASSES,
        samples_per_class,
        figsize=(2 * samples_per_class, 2 * NUM_CLASSES)
    )  # Arrange one CIFAR-10 class per row

    for class_id in range(NUM_CLASSES):
        start_index = class_id * samples_per_class  # Locate the first image for this class

        for sample_id in range(samples_per_class):
            image_index = start_index + sample_id

            axes[class_id, sample_id].imshow(
                generated_images[image_index]
            )

            axes[class_id, sample_id].axis("off")

        axes[class_id, 0].text(
            -0.18,
            0.5,
            CLASS_NAMES[class_id],
            transform=axes[class_id, 0].transAxes,
            ha="right",
            va="center",
            fontsize=10
        )  # Label each row with its requested class

    plt.suptitle(
        title,
        fontsize=16
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=200,
            bbox_inches="tight"
        )  # Optionally save a high-resolution copy for reporting

    plt.show()

    return generated_images, labels.reshape(-1)  # Return images and labels for further analysis

In [ ]:
ttur_grid_path = os.path.join(
    TTUR_DIR,
    "ttur_generated_class_grid.png"
)

ttur_grid_images, ttur_grid_labels = show_generated_class_grid(
    model=ttur_generator,
    latent_dim=LATENT_DIM,
    samples_per_class=8,
    seed=SEED,
    title=(
        "cDCGAN with Reduced Discriminator Learning Rate\n"
        "Eight Generated Images per Class"
    ),
    save_path=ttur_grid_path
)

print("Generated grid saved to:", ttur_grid_path)

<Figure size 1600x2000 with 80 Axes>

Generated grid saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline_ttur_dlr1e4/ttur_generated_class_grid.png


#### 17.3.2 Balanced evaluation samples


In [ ]:
# ============================================================
# Generate balanced TTUR evaluation images
# ============================================================

(
    ttur_generated_images,
    ttur_generated_labels
) = generate_balanced_evaluation_images(
    generator=ttur_generator,
    latent_dim=LATENT_DIM,
    total_images=FID_KL_SAMPLE_SIZE,
    seed=SEED
)  # Generate an equal and reproducible number of images across all classes

print(
    "Generated evaluation images:",
    ttur_generated_images.shape
)  # Verify the number and dimensions of generated images

print(
    "Generated evaluation labels:",
    ttur_generated_labels.shape
)  # Verify that every generated image has a corresponding label

Generated evaluation images: (5000, 32, 32, 3)
Generated evaluation labels: (5000,)


#### 17.3.3 Overall distribution metrics


In [ ]:
# ============================================================
# Overall TTUR evaluation
# ============================================================

ttur_fid = calculate_fid(
    REAL_EVAL_IMAGES,
    ttur_generated_images
)  # Measure distance between the real and TTUR feature distributions

ttur_kid = calculate_kid(
    REAL_EVAL_IMAGES,
    ttur_generated_images
)  # Provide a complementary kernel-based distribution estimate

(
    ttur_precision,
    ttur_recall
) = calculate_precision_recall(
    REAL_EVAL_IMAGES,
    ttur_generated_images
)  # Measure generated-image fidelity and real-data diversity coverage

(
    ttur_density,
    ttur_coverage
) = calculate_density_coverage(
    REAL_EVAL_IMAGES,
    ttur_generated_images
)  # Further assess sample quality and distribution coverage

ttur_pixel_kl = calculate_pixel_kl(
    REAL_EVAL_IMAGES,
    ttur_generated_images
)  # Compare the overall real and generated pixel distributions

print("=" * 45)
print("TTUR-INSPIRED cDCGAN — OVERALL RESULTS")
print("=" * 45)
print(f"FID:       {ttur_fid:.2f}")
print(f"KID:       {ttur_kid:.4f}")
print(f"Precision: {ttur_precision:.3f}")
print(f"Recall:    {ttur_recall:.3f}")
print(f"Density:   {ttur_density:.3f}")
print(f"Coverage:  {ttur_coverage:.3f}")
print(f"Pixel KL:  {ttur_pixel_kl:.4f}")  # Display all overall metrics consistently

TTUR-INSPIRED cDCGAN — OVERALL RESULTS
FID:       198.09
KID:       0.1049
Precision: 0.701
Recall:    0.337
Density:   1.524
Coverage:  0.932
Pixel KL:  0.0097


#### 17.3.4 Class-conditioning accuracy


In [ ]:
# ============================================================
# Conditional-generation evaluation
# ============================================================

(
    ttur_conditioning_accuracy,
    ttur_per_class_accuracy
) = class_conditioning_accuracy(
    generator=ttur_generator,
    latent_dim=LATENT_DIM,
    classifier=cifar_classifier,
    n_samples_per_class=100,
    seed=SEED
)  # Check whether TTUR outputs match their requested class labels

print(
    "\nOverall class-conditioning accuracy:",
    f"{ttur_conditioning_accuracy:.1%}"
)  # Report label fidelity across all classes

print("\nPer-class conditioning accuracy:")

for class_name, accuracy in ttur_per_class_accuracy.items():
    print(
        f"{class_name:12s}: "
        f"{accuracy:.1%}"
    )  # Reveal which individual classes are easier or harder to condition


Overall class-conditioning accuracy: 60.0%

Per-class conditioning accuracy:
airplane    : 61.0%
automobile  : 59.0%
bird        : 33.0%
cat         : 56.0%
deer        : 66.0%
dog         : 35.0%
frog        : 84.0%
horse       : 50.0%
ship        : 88.0%
truck       : 68.0%


In [ ]:
(
    ttur_conditioning_accuracy,
    ttur_per_class_accuracy
) = class_conditioning_accuracy(
    ttur_generator,
    LATENT_DIM,
    cifar_classifier,
    n_samples_per_class=100,
    seed=SEED
)

#### 17.3.5 Classifier-based Inception Score


In [ ]:
# ============================================================
# Classifier-based confidence and diversity score
# ============================================================

(
    ttur_classifier_score_mean,
    ttur_classifier_score_std
) = classifier_based_inception_score(
    generator=ttur_generator,
    latent_dim=LATENT_DIM,
    classifier=cifar_classifier,
    n_samples=1000,
    splits=10,
    seed=SEED
)  # Estimate class recognisability and diversity across reproducible samples

print(
    "Classifier-based score:",
    f"{ttur_classifier_score_mean:.3f}",
    "+/-",
    f"{ttur_classifier_score_std:.3f}"
)  # Report the mean score and variation across evaluation splits

Classifier-based score: 4.335 +/- 0.167


#### 17.3.6 Per-class evaluation


In [ ]:
# ============================================================
# Per-class TTUR evaluation
# ============================================================

(
    ttur_per_class_fid,
    ttur_per_class_kl
) = per_class_evaluation(
    generator=ttur_generator,
    latent_dim=LATENT_DIM,
    real_test_images=x_test_original,
    real_test_labels=y_test_model,
    n_samples_per_class=1000,
    seed=SEED
)

airplane     | FID:    246.72 | Pixel KL: 0.0030
automobile   | FID:    300.89 | Pixel KL: 0.0115
bird         | FID:    558.78 | Pixel KL: 0.0313
cat          | FID:    462.13 | Pixel KL: 0.0147
deer         | FID:    230.07 | Pixel KL: 0.0342
dog          | FID:   1151.15 | Pixel KL: 0.0344
frog         | FID:    230.79 | Pixel KL: 0.0182
horse        | FID:    430.02 | Pixel KL: 0.0376
ship         | FID:    253.97 | Pixel KL: 0.0342
truck        | FID:    204.95 | Pixel KL: 0.0126


#### 17.3.7 Record TTUR results


In [ ]:
# ============================================================
# Log TTUR results
# ============================================================

log_results(
    model_name=TTUR_MODEL_NAME,
    overall_fid=ttur_fid,
    overall_kl=ttur_pixel_kl,
    overall_kid=ttur_kid,
    precision=ttur_precision,
    recall=ttur_recall,
    density=ttur_density,
    coverage=ttur_coverage,
    per_class_fid=ttur_per_class_fid,
    per_class_kl=ttur_per_class_kl,
    inception_score=(
        ttur_classifier_score_mean,
        ttur_classifier_score_std
    ),
    class_cond_accuracy=ttur_conditioning_accuracy,
    epochs_trained=TTUR_EPOCHS,
    notes=(
        "TTUR-inspired asymmetric learning-rate stabilisation. "
        "Generator LR=0.0002 and Discriminator LR=0.0001. "
        "Architecture, batch size, latent dimension, BCE loss and "
        "one-sided label smoothing were unchanged from the baseline."
    )
)  # Store all TTUR metrics and settings in the shared experiment log

show_results_table()  # Display the updated comparison across all evaluated models

Logged results for 'baseline_ttur_dlr1e4'.
Model                  Epochs  FID       KID       Prec    Rec     Dens    Cov     IS               ClassAcc  Notes
------------------------------------------------------------------------------------------------------------------------------------------------------
baseline_cDCGAN        100     170.29    0.0893    0.683   0.389   1.378   0.953   4.74+/-0.34      64.4%     Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage.
baseline_ttur_dlr1e4   100     198.09    0.1049    0.701   0.337   1.524   0.932   4.33+/-0.17      60.0%     TTUR-inspired asymmetric learning-rate stabilisation. Generator LR=0.0002 and Discriminator LR=0.0001. Architecture, batch size, latent dimension, BCE loss and one-sided label smoothing were unchanged from the baseline.


### 17.4 Eye-power qualitative assessment — TTUR

The TTUR-generated samples were manually inspected for recognisable class-specific shapes, visual consistency and separation from other classes, using the same qualitative criteria applied to the baseline.

| Class | Clear | Marginal | Nonsense | Notes |
|---|---:|---:|---:|---|
| airplane | | ✓ | | Several samples contain recognisable aircraft-like bodies or wings, but some are distorted or confused with other elongated objects. |
| automobile | ✓ | | | Most samples contain recognisable car bodies and road-like backgrounds, although some fine details remain distorted. |
| bird | | ✓ | | Several samples contain bird-like bodies, beaks or silhouettes, but the class structure is inconsistent and some samples resemble other animals. |
| cat | ✓ | | | Most samples contain recognisable feline faces, ears or body shapes, although some remain distorted. |
| deer | | ✓ | | Several four-legged animal forms and natural backgrounds are present, but some samples remain ambiguous or resemble other animal classes. |
| dog | | ✓ | | Some samples contain dog-like heads or bodies, but many are structurally ambiguous and overlap visually with other animal classes. |
| frog | | | ✓ | Appropriate green and brown textures are sometimes present, but most samples lack a clearly recognisable frog structure. |
| horse | | ✓ | | Several samples contain horse-like bodies and outdoor backgrounds, but recognition is inconsistent and some resemble deer or other animals. |
| ship | ✓ | | | Most samples contain boat- or ship-like structures with suitable water or sky backgrounds, although a few are ambiguous. |
| truck | ✓ | | | Most samples contain recognisable large vehicle or truck-like structures, although some remain distorted. |

Overall, the TTUR model retained recognisable global characteristics for several classes, particularly automobiles, cats, ships and trucks. However, visually overlapping animal categories remained difficult, with substantial ambiguity among birds, deer, dogs, frogs and horses.

Compared with the baseline, reducing the Discriminator learning rate did not produce an overall qualitative improvement. Although several individual samples remain recognisable, object structure is still inconsistent and cross-class confusion remains visible. The quantitative evaluation likewise shows that this TTUR configuration did not improve the overall generated distribution.

### 17.5 Loss curve interpretation

Under the TTUR-inspired configuration, the Discriminator learning rate was reduced from \(2\times10^{-4}\) to \(1\times10^{-4}\), while the Generator learning rate remained at \(2\times10^{-4}\). The architecture, batch size, latent dimension and BCE loss were otherwise unchanged.

The Generator and Discriminator losses should not be interpreted in the same way as conventional supervised-learning losses because the two networks are trained adversarially. A decreasing Discriminator loss together with a higher or increasing Generator loss can indicate that the Discriminator is becoming more successful at separating real and generated samples, making the Generator's task harder.

Therefore, the loss curves are used mainly to diagnose the training dynamics rather than to determine which GAN is better. In this experiment, the quantitative metrics and generated images show that reducing the Discriminator learning rate did not improve overall generation quality. This demonstrates why apparently stable GAN loss curves do not necessarily correspond to better generated images.

### 17.6 Metric-by-metric interpretation

Compared with the baseline cDCGAN, the TTUR-inspired configuration produced mixed changes, but the main measures of distributional similarity, diversity and conditional accuracy deteriorated:

- **FID:** increased from **170.29 to 198.09**. Since lower FID is better, the generated distribution became less similar to the real CIFAR-10 distribution.
- **KID:** increased from **0.0893 to 0.1049**, supporting the FID result that overall distributional similarity became worse.
- **Precision:** increased slightly from **0.683 to 0.701**, suggesting that generated samples were somewhat more concentrated within realistic regions of the feature space.
- **Recall:** decreased from **0.389 to 0.337**, indicating poorer coverage of the diversity present in the real dataset.
- **Density:** increased from **1.378 to 1.524**, suggesting stronger local concentration of generated samples around real-data feature regions.
- **Coverage:** decreased from **0.953 to 0.932**, again indicating reduced coverage of the real-data distribution.
- **Inception Score:** decreased from **4.74 ± 0.34 to 4.33 ± 0.17**, indicating weaker overall classifiability and/or diversity.
- **Class-conditioning accuracy:** decreased from **64.4% to 60.0%**, showing that the Generator became less reliable at producing the requested class.

The higher Precision and Density show that TTUR did not worsen every aspect of generation. However, these gains were accompanied by poorer FID, KID, Recall, Coverage, Inception Score and class-conditioning accuracy. This suggests a quality–diversity trade-off in which samples became slightly more concentrated around realistic regions but represented less of the real distribution.

Overall, this TTUR configuration was **not retained as an improvement** over the baseline. It is kept as a useful negative experiment showing that reducing the Discriminator learning rate alone does not necessarily improve GAN performance.

### 17.7 Results backup


In [ ]:
# ============================================================
# Back up the updated results
# ============================================================

ttur_results_backup_path = os.path.join(
    TTUR_DIR,
    "results_log_after_ttur.json"
)  # Store the TTUR results backup in its experiment directory

with open(
    ttur_results_backup_path,
    "w"
) as file:
    json.dump(
        results_log,
        file,
        indent=2,
        default=str
    )  # Save all logged metrics in a readable JSON format

print(
    "Updated results log saved to:",
    ttur_results_backup_path
)  # Confirm the exact location of the saved backup

Updated results log saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline_ttur_dlr1e4/results_log_after_ttur.json


## 18. Hyperparameter Tuning: Latent Dimension and Batch Size

### Purpose and motivation

Having tested one specific, targeted fix for our baseline's training-dynamics issue (TTUR, Section 17), we now step back and ask a broader question: independent of any specific training imbalance, are our baseline's core hyperparameters — latent dimension and batch size — even well-chosen in the first place? Rather than assuming our initial values (`latent_dim=128`, `batch_size=128`) were optimal, we conduct a systematic screening across plausible alternatives, consistent with standard GAN development practice (Radford et al., 2015; Goodfellow et al., 2014 both note the general sensitivity of GAN training to these choices without prescribing fixed optimal values, making empirical search a necessary step rather than an optional one).

**Why latent dimension specifically:** the latent vector is the sole source of randomness the Generator has to work with — it must encode enough information to represent all the variation across our ten classes' worth of images (poses, backgrounds, colours, textures — exactly the kind of variation our EDA in Section 3A documented as substantial and uneven across classes). Too small a latent space may not have enough capacity to represent this variation; too large a latent space gives the Generator more room than it can effectively learn to use within a fixed training budget, potentially slowing convergence rather than improving it. We test `latent_dim ∈ {64, 128, 256}` to check whether our original choice of 128 sits in a reasonable range, or whether a smaller
or larger value would have served the task better.

**Why batch size specifically:** batch size affects GAN training in two competing ways. Smaller batches mean more gradient updates per epoch for the same amount of data (since `steps_per_epoch = dataset_size / batch_size`), which can accelerate learning, but each individual gradient estimate is noisier (based on fewer examples), which can act as either a helpful, regularising source of stochasticity or a destabilising source of variance, depending on the specific model and dataset. Larger batches give smoother, more reliable gradient estimates, at the direct cost of fewer total updates for the same training budget. Given this genuine trade-off, with no clear theoretical answer for which side wins for our specific architecture and dataset, we test `batch_size ∈ {64, 128, 256}` empirically rather than assuming either direction is
correct.

**Why we screen at a reduced epoch budget first:** testing every configuration at the full 100 epochs used for our baseline would be prohibitively time-consuming across six combinations. We therefore screen each configuration at a reduced budget (50 epochs) to identify promising candidates relatively cheaply, then retrain only the most promising configuration at the full epoch count for a fair, final comparison against our baseline — a standard two-stage approach to hyperparameter search under compute constraints (Bergstra & Bengio, 2012, note that even simple grid search remains a reasonable, widely-used baseline method when more exhaustive search techniques are impractical within project time and compute limits).

### 18.1 Experiment function and tuning runs

In [ ]:
def run_tuning_experiment(
    config_name,
    latent_dim=128,
    batch_size=128,
    g_lr=2e-4,
    d_lr=2e-4,
    epochs=50,
    display_every=25
):
    global generator  # Allow the shared training functions to use the new Generator
    global discriminator  # Replace the previous experiment's Discriminator
    global generator_optimizer
    global discriminator_optimizer
    global train_step

    global LATENT_DIM  # Update the latent dimension for the current experiment
    global BATCH_SIZE  # Update the batch size for the current experiment
    global gen_loss_history
    global disc_loss_history
    global seed_noise

    print(
        f"\n{'=' * 70}\n"
        f"Experiment: {config_name}\n"
        f"Latent dimension: {latent_dim}\n"
        f"Batch size: {batch_size}\n"
        f"Generator learning rate: {g_lr}\n"
        f"Discriminator learning rate: {d_lr}\n"
        f"Epochs: {epochs}\n"
        f"{'=' * 70}"
    )  # Display the configuration before training

    tf.keras.backend.clear_session()  # Clear the previous Keras model state
    tf.keras.utils.set_random_seed(SEED)  # Reset TensorFlow randomness for fairer comparisons
    np.random.seed(SEED)  # Reset NumPy randomness for reproducibility

    LATENT_DIM = latent_dim  # Apply the selected latent dimension
    BATCH_SIZE = batch_size  # Apply the selected batch size

    seed_noise = tf.random.normal(
        [NUM_EXAMPLES, LATENT_DIM],
        seed=SEED
    )  # Create fixed monitoring noise with the correct latent size

    exp_dataset = make_gan_dataset(
        images=x_train_gan,
        labels=y_train_model,
        batch_size=BATCH_SIZE,
        seed=SEED
    )  # Rebuild the dataset using the selected batch size

    generator = build_generator(
        latent_dim=LATENT_DIM
    )  # Build a fresh Generator so weights are not carried between experiments

    discriminator = build_discriminator()  # Build a fresh Discriminator

    generator_optimizer = tf.keras.optimizers.Adam(
        learning_rate=g_lr,
        beta_1=0.5
    )  # Apply the selected Generator learning rate

    discriminator_optimizer = tf.keras.optimizers.Adam(
        learning_rate=d_lr,
        beta_1=0.5
    )  # Apply the selected Discriminator learning rate

    train_step = make_train_step()  # Recompile training with the new models and optimisers

    gen_loss_history = []  # Reset Generator loss history
    disc_loss_history = []  # Reset Discriminator loss history

    train(
        dataset=exp_dataset,
        epochs=epochs,
        display_every=display_every,
        tag=config_name
    )  # Train the current configuration independently

    experiment_dir = os.path.join(
        OUTPUT_ROOT,
        "tuning",
        config_name
    )  # Use a separate folder to prevent experiments from overwriting each other

    os.makedirs(
        experiment_dir,
        exist_ok=True
    )

    generator_weights_path = os.path.join(
        experiment_dir,
        f"{config_name}_generator.weights.h5"
    )

    discriminator_weights_path = os.path.join(
        experiment_dir,
        f"{config_name}_discriminator.weights.h5"
    )

    generator.save_weights(
        generator_weights_path
    )  # Preserve the trained Generator

    discriminator.save_weights(
        discriminator_weights_path
    )  # Preserve the corresponding Discriminator

    history_dataframe = pd.DataFrame({
        "epoch": np.arange(1, epochs + 1),
        "generator_loss": gen_loss_history,
        "discriminator_loss": disc_loss_history
    })  # Organise epoch losses for later comparison

    history_dataframe.to_csv(
        os.path.join(
            experiment_dir,
            f"{config_name}_history.csv"
        ),
        index=False
    )  # Save training history for reproducibility and plotting

    gen_images, gen_labels = (
        generate_balanced_evaluation_images(
            generator=generator,
            latent_dim=LATENT_DIM,
            total_images=FID_KL_SAMPLE_SIZE,
            seed=SEED
        )
    )  # Generate equal numbers from all classes for fair evaluation

    exp_fid = calculate_fid(
        REAL_EVAL_IMAGES,
        gen_images
    )  # Measure overall feature-distribution similarity

    exp_kid = calculate_kid(
        REAL_EVAL_IMAGES,
        gen_images
    )  # Provide a complementary distribution-distance estimate

    (
        exp_precision,
        exp_recall
    ) = calculate_precision_recall(
        REAL_EVAL_IMAGES,
        gen_images
    )  # Measure generated-sample fidelity and real-distribution coverage

    (
        exp_density,
        exp_coverage
    ) = calculate_density_coverage(
        REAL_EVAL_IMAGES,
        gen_images
    )  # Measure local concentration and distribution coverage

    (
        exp_cond_acc,
        exp_per_class_cond_acc
    ) = class_conditioning_accuracy(
        generator,
        LATENT_DIM,
        cifar_classifier
    )  # Check whether generated images match their requested classes

    log_results(
        model_name=config_name,
        overall_fid=exp_fid,
        overall_kid=exp_kid,
        precision=exp_precision,
        recall=exp_recall,
        density=exp_density,
        coverage=exp_coverage,
        class_cond_accuracy=exp_cond_acc,
        epochs_trained=epochs,
        notes=(
            f"Hyperparameter screening: "
            f"latent_dim={latent_dim}, "
            f"batch_size={batch_size}, "
            f"g_lr={g_lr}, "
            f"d_lr={d_lr}."
        )
    )  # Store results using the same format as every other experiment

    print(
        f"\n===== {config_name} results =====\n"
        f"FID:          {exp_fid:.2f}\n"
        f"KID:          {exp_kid:.4f}\n"
        f"Precision:    {exp_precision:.3f}\n"
        f"Recall:       {exp_recall:.3f}\n"
        f"Density:      {exp_density:.3f}\n"
        f"Coverage:     {exp_coverage:.3f}\n"
        f"Class Acc.:   {exp_cond_acc:.1%}\n"
        f"Generator weights: {generator_weights_path}\n"
    )  # Summarise the completed experiment

    return {
        "config_name": config_name,
        "latent_dim": latent_dim,
        "batch_size": batch_size,
        "g_lr": g_lr,
        "d_lr": d_lr,
        "epochs": epochs,
        "fid": exp_fid,
        "kid": exp_kid,
        "precision": exp_precision,
        "recall": exp_recall,
        "density": exp_density,
        "coverage": exp_coverage,
        "class_conditioning_accuracy": exp_cond_acc,
        "generator_weights_path": generator_weights_path,
        "discriminator_weights_path": discriminator_weights_path
    }  # Return metrics and paths for later comparisons

### 18.2 Hyperparameter screening runs


In [ ]:
result_latent64 = run_tuning_experiment("latent64", latent_dim=64, batch_size=128, epochs=50)

<Figure size 2000x300 with 10 Axes>

Epoch 50/50  gen_loss=1.7383  disc_loss=0.9780  epoch_time=8.9s  elapsed=6.8min
Logged results for 'latent64'.

===== latent64 results =====
FID:          162.52
KID:          0.0640
Precision:    0.679
Recall:       0.366
Density:      1.337
Coverage:     0.911
Class Acc.:   53.7%
Generator weights: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/tuning/latent64/latent64_generator.weights.h5



In [ ]:
result_latent256 = run_tuning_experiment("latent256", latent_dim=256, batch_size=128, epochs=50)

<Figure size 2000x300 with 10 Axes>

Epoch 50/50  gen_loss=1.9532  disc_loss=0.9132  epoch_time=9.4s  elapsed=6.0min
Logged results for 'latent256'.

===== latent256 results =====
FID:          161.51
KID:          0.0713
Precision:    0.704
Recall:       0.332
Density:      1.498
Coverage:     0.920
Class Acc.:   47.6%
Generator weights: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/tuning/latent256/latent256_generator.weights.h5



In [ ]:
result_batch64 = run_tuning_experiment("batch64", latent_dim=128, batch_size=64, epochs=50)

<Figure size 2000x300 with 10 Axes>

Epoch 50/50  gen_loss=2.0317  disc_loss=0.9060  epoch_time=15.6s  elapsed=11.2min
Logged results for 'batch64'.

===== batch64 results =====
FID:          171.95
KID:          0.0805
Precision:    0.679
Recall:       0.377
Density:      1.451
Coverage:     0.927
Class Acc.:   55.1%
Generator weights: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/tuning/batch64/batch64_generator.weights.h5



In [ ]:
result_batch256 = run_tuning_experiment("batch256", latent_dim=128, batch_size=256, epochs=50)

<Figure size 2000x300 with 10 Axes>

Epoch 50/50  gen_loss=1.6698  disc_loss=0.9898  epoch_time=7.2s  elapsed=4.4min
Logged results for 'batch256'.

===== batch256 results =====
FID:          379.38
KID:          0.2340
Precision:    0.665
Recall:       0.132
Density:      1.324
Coverage:     0.739
Class Acc.:   23.9%
Generator weights: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/tuning/batch256/batch256_generator.weights.h5



In [ ]:
show_results_table()

Model                  Epochs  FID       KID       Prec    Rec     Dens    Cov     IS               ClassAcc  Notes
------------------------------------------------------------------------------------------------------------------------------------------------------
baseline_cDCGAN        100     170.29    0.0893    0.683   0.389   1.378   0.953   4.74+/-0.34      64.4%     Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage.
baseline_ttur_dlr1e4   100     198.09    0.1049    0.701   0.337   1.524   0.932   4.33+/-0.17      60.0%     TTUR-inspired asymmetric learning-rate stabilisation. Generator LR=0.0002 and Discriminator LR=0.0001. Architecture, batch size, latent dimension, BCE loss and one-sided label smoothing were unchanged from the baseline.
latent64               50      162.52    0.0640    0.679   0.366   1.337   0.911   -                53.7%     Hyperparameter screening: latent_dim=64, batch_size=128, g_lr=0.0002, d_lr=0.0002.
latent256              5

### 18.3 Decision: selecting a configuration to retrain at full length

**Reading the screening results against our baseline (FID=170.29, Recall=0.389, ClassAcc=64.4%):**

None of `latent64`, `latent256`, or `batch64` produced a decisive, unambiguous winner at this reduced 50-epoch screening budget — their FID values (161.51-171.95) sit within a range narrow enough that we cannot confidently distinguish a genuine hyperparameter effect from ordinary run-to-run training variance (which is something we investigate and document more formally later i  this notebook). `batch256`, however, is a clear and unambiguous exception: FID=379.38 (more than
double the other configurations), Recall=0.132 (roughly a third of the alternatives), and
ClassAcc=23.9% (close to the 10% random-chance floor) — a decisively worse result, consistent with our expectation that a larger batch size, giving fewer gradient updates per epoch for the same training budget, would leave the Generator/Discriminator undertrained relative to the smaller-batch alternatives.

**Given this, our decision logic is as follows:**

1. `batch_size=256` is ruled out on strong, unambiguous evidence.
2. Among the three remaining configurations (`latent64`, `latent256`, `batch64`), no option demonstrates a decisive advantage over the others at this screening budget.
3. We select `batch_size=64` to retrain at the full 100-epoch budget — not because this screening proved it superior, but because it was never the worst-performing configuration, and it offers a plausible, independently-justifiable mechanism for improvement (more gradient updates per epoch than `batch_size=128`, at only a modest increase in per-batch gradient noise).

We treat this as a **practical, defensible choice under uncertainty**, not a claim of proven optimality — an important distinction we return to once we have the full-length retraining result and can compare it fairly against our baseline's own documented run-to-run variance.

### 18.4 Full-length retraining: batch_size=64, 100 epochs

To determine whether `batch_size=64` provides a genuine improvement rather than a difference too small to distinguish from noise therefore we retrain this configuration at the same 100-epoch budget used for our baseline, allowing a fair, matched comparison rather than comparing a 50-epoch screening result against a fully-trained 100-epoch baseline.

In [ ]:
result_batch64_full = run_tuning_experiment("batch64_full100", latent_dim=128, batch_size=64,
                                               epochs=100, display_every=25)
show_results_table()

<Figure size 2000x300 with 10 Axes>

Epoch 100/100  gen_loss=2.5062  disc_loss=0.8048  epoch_time=15.4s  elapsed=22.5min
Logged results for 'batch64_full100'.

===== batch64_full100 results =====
FID:          161.07
KID:          0.0788
Precision:    0.696
Recall:       0.392
Density:      1.530
Coverage:     0.947
Class Acc.:   63.7%
Generator weights: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/tuning/batch64_full100/batch64_full100_generator.weights.h5

Model                  Epochs  FID       KID       Prec    Rec     Dens    Cov     IS               ClassAcc  Notes
------------------------------------------------------------------------------------------------------------------------------------------------------
baseline_cDCGAN        100     170.29    0.0893    0.683   0.389   1.378   0.953   4.74+/-0.34      64.4%     Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage.
baseline_ttur_dlr1e4   100     198.09    0.1049    0.701   0.337   1.524   0.932   4.33+/-0.17      

### 18.5 Eye-power qualitative assessment — Tuned (batch_size=64, full 100 epochs)


This assessment is based on one generated sample per class. Therefore, it provides only a preliminary indication of class recognisability and should not be used alone to judge consistency or diversity.

| Class | Clear | Marginal | Nonsense | Notes |
|---|---:|---:|---:|---|
| airplane | | ✓ | | The sample contains some aircraft-like structure, but the overall object is heavily distorted and difficult to identify confidently. |
| automobile | ✓ | | | The sample contains a recognisable automobile-like body and appropriate road-like appearance, although details remain blurry. |
| bird | | ✓ | | The sample contains an animal-like vertical form that could represent a bird, but clear wings, beak and body structure are poorly defined. |
| cat | | ✓ | | Some animal-like features are visible, but the structure is heavily distorted and not consistently recognisable as a cat. |
| deer | ✓ | | | A recognisable four-legged deer-like body appears against an appropriate natural background. |
| dog | | ✓ | | The sample contains an animal-like form, but facial and body features are too distorted for confident identification as a dog. |
| frog | ✓ | | | The sample contains a relatively recognisable frog-like body together with appropriate green and brown natural colours. |
| horse | | ✓ | | A four-legged animal structure is visible, but the body is distorted and only partially recognisable as a horse. |
| ship | ✓ | | | The sample contains a recognisable ship-like structure together with appropriate blue water and sky colours. |
| truck | ✓ | | | A recognisable large vehicle structure is present, although some finer truck features remain blurred. |

The batch-size-64 model produced recognisable class-level structure for several categories, particularly automobiles, deer, frogs, ships and trucks. Other classes, including airplanes, birds, cats, dogs and horses, remained more ambiguous because of distorted or incomplete object structures.

The generated sample suggests that the model learned both contextual cues and some meaningful object structures. However, because only one image per class is shown here, these observations should be treated as supporting qualitative evidence rather than a reliable measure of the model's overall diversity or consistency. Quantitative metrics calculated from a much larger generated sample are therefore used for the main comparison.

### 18.6 Loss curve interpretation

The batch-size-64 model shows the same general adversarial behaviour observed in the baseline: the Discriminator becomes progressively stronger while the Generator faces increasing difficulty in fooling it. By epoch 100, the Generator loss reached approximately **2.51**, while the Discriminator loss was approximately **0.80**.

Using `batch_size=64` also produces more parameter-update steps per epoch than `batch_size=128`, since the same training dataset is divided into more batches. This changes the optimisation dynamics and gives both networks more updates over the same number of epochs.

As with the previous experiments, the adversarial losses are not interpreted as direct measures of image quality. The generated images and distribution-based evaluation metrics are used alongside the loss curves when assessing whether the configuration improved the model.

### 18.7 Metric-by-metric interpretation

Compared with the baseline cDCGAN, training the selected batch-size-64 configuration for the full 100 epochs produced several improvements:

- **FID:** decreased from **170.29 to 161.07**, indicating that the generated-image distribution became more similar to the real CIFAR-10 distribution.
- **KID:** decreased from **0.0893 to 0.0788**, supporting the improvement observed in FID.
- **Precision:** increased from **0.683 to 0.696**, suggesting a small improvement in the fidelity of generated samples.
- **Recall:** increased slightly from **0.389 to 0.392**, indicating broadly similar but marginally improved coverage of the real-data distribution.
- **Density:** increased from **1.378 to 1.530**, suggesting that generated samples were more strongly represented within neighbourhoods of real samples.
- **Coverage:** decreased slightly from **0.953 to 0.947**, indicating a small trade-off in how much of the real-data distribution was covered.
- **Class-conditioning accuracy:** decreased slightly from **64.4% to 63.7%**, meaning that the improvement in distributional similarity did not translate into better conditional class fidelity.

Overall, the batch-size-64 model produced recognisable class-level structure for several categories, particularly automobiles, deer, frogs, ships and trucks. Other classes, including birds, cats, dogs and horses, remained more ambiguous because of distorted or incomplete object structures.

These results suggest that the model learned broad colour, background and approximate shape information more reliably than fine-grained object structure. This is consistent with the earlier EDA, which showed substantial within-class variation and visual overlap among several animal categories. In particular, visually similar classes such as cats, dogs, deer and horses were expected to be more difficult to separate than structurally distinctive vehicle classes.

However, because only one generated image per class is shown in this qualitative grid, these observations provide only supporting evidence. The larger-sample quantitative metrics are therefore used for the main comparison between models.

### 18.9 Comparative visualisations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select which models to compare -- edit this list as needed
models_to_plot = ["baseline_cDCGAN", "latent64", "latent256", "batch64", "batch256"]

# Pull out metrics for each, safely skipping any not yet logged
plot_data = {name: results_log[name] for name in models_to_plot if name in results_log}

metrics_to_show = [
    ("overall_fid", "FID (lower is better)", "indianred"),
    ("recall", "Recall (higher is better)", "seagreen"),
    ("coverage", "Coverage (higher is better)", "steelblue"),
    ("class_cond_accuracy", "Class-conditioning accuracy (higher is better)", "goldenrod"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (key, title, color) in zip(axes, metrics_to_show):
    names = list(plot_data.keys())
    values = [plot_data[n][key] for n in names]
    bars = ax.bar(names, values, color=color)
    ax.set_title(title, fontsize=12)
    ax.set_xticklabels(names, rotation=30, ha='right')
    for bar, v in zip(bars, values):
        label = f"{v:.1%}" if key == "class_cond_accuracy" else f"{v:.2f}" if key == "overall_fid" else f"{v:.3f}"
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), label,
                ha='center', va='bottom', fontsize=9)

plt.suptitle("Hyperparameter tuning comparison", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

/tmp/ipykernel_2707/606654369.py:25: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(names, rotation=30, ha='right')
/tmp/ipykernel_2707/606654369.py:25: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(names, rotation=30, ha='right')
/tmp/ipykernel_2707/606654369.py:25: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(names, rotation=30, ha='right')
/tmp/ipykernel_2707/606654369.py:25: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(names, rotation=30, ha='right')


<Figure size 1400x1000 with 4 Axes>

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

names = list(plot_data.keys())
x = np.arange(len(names))
width = 0.2

# Normalize FID for visual comparison (invert + scale so "higher = better" like the other three)
fid_values = np.array([plot_data[n]["overall_fid"] for n in names])
fid_normalized = 1 - (fid_values / fid_values.max())   # 0 to 1, higher = better

recall_values = [plot_data[n]["recall"] for n in names]
coverage_values = [plot_data[n]["coverage"] for n in names]
classacc_values = [plot_data[n]["class_cond_accuracy"] for n in names]

ax.bar(x - 1.5*width, fid_normalized, width, label='FID (normalized, higher=better)', color='indianred')
ax.bar(x - 0.5*width, recall_values, width, label='Recall', color='seagreen')
ax.bar(x + 0.5*width, coverage_values, width, label='Coverage', color='steelblue')
ax.bar(x + 1.5*width, classacc_values, width, label='Class-conditioning accuracy', color='goldenrod')

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel("Score (0-1 scale)")
ax.set_title("Combined hyperparameter tuning comparison (all metrics, higher = better)")
ax.legend()
plt.tight_layout()
plt.show()

<Figure size 1200x600 with 1 Axes>

## 19. UpSampling2D Ablation

### Purpose and motivation

Having tested one training-dynamics intervention (TTUR, rejected) and one hyperparameter search
(Section 18), we now turn to a specific, well-documented *architectural* limitation of our
Generator: the transposed convolution (`Conv2DTranspose`) layers used for upsampling at every
resolution stage.Transposed convolutions can produce uneven overlap between the kernel and the output feature map — depending on the relationship between kernel size and stride — causing certain output pixels to receive
contributions from more input positions than their neighbours. This uneven overlap manifests visually as a regular, grid-like "checkerboard" pattern, particularly visible in smooth or gradient-heavy regions of an image.

This is not a purely theoretical concern for our project.Reviewing our baseline's generated image grid (Section 14) alongside our real-vs-generated comparisons, we observed exactly this kind of regular, grid-like artefacting in several generated samples, particularly in background/sky regions of classes such as `airplane` and `ship` — visually consistent with the checkerboard pattern the literature describes, and a plausible contributing factor to our
baseline's relatively high FID for these otherwise structurally simple classes.

Why this specifically targets a different weakness than TTUR or hyperparameter tuning: TTUR addressed a training-dynamics imbalance; hyperparameter tuning addressed convergence efficiency. This experiment is different again as it targets a specific, structural property of how the Generator constructs its output, independent of how well or how quickly training otherwise proceeds. Testing this as its own isolated ablation (replacing only the upsampling
mechanism, while holding total network capacity constant via the same refinement-layer structure) lets us determine whether this specific, literature-documented artefact source is a meaningful contributor to our results, without confounding the comparison with any other simultaneous
change.

### 19.1 Architecture and validation

We replace every `Conv2DTranspose` upsampling layer in our Generator with an `UpSampling2D` (nearest-neighbour resize) followed by a standard `Conv2D` layer Since distributing information evenly across the upsampled resolution before any
learned convolution is applied avoids the systematic overlap unevenness that causes checkerboard artefacts by construction. We retain the same refinement-convolution structure and total filter counts as our baseline Generator at each resolution stage, isolating the upsampling mechanism as
the single variable under test, and validate the resulting architecture's output shape and value range before committing to a full training run.

In [ ]:
def build_generator_upsampling(latent_dim=128, num_classes=NUM_CLASSES):
    noise_input = layers.Input(shape=(latent_dim,), name="upsampling_noise_input")  # Random latent vector
    label_input = layers.Input(shape=(1,), dtype="int32", name="upsampling_label_input")  # Requested class

    # Label pathway
    label_embedding = layers.Embedding(num_classes, 50, name="upsampling_label_embedding")(label_input)  # Learn class representation
    label_embedding = layers.Flatten(name="upsampling_label_flatten")(label_embedding)
    label_map = layers.Dense(4 * 4, name="upsampling_label_projection")(label_embedding)  # Project label spatially
    label_map = layers.Reshape((4, 4, 1), name="upsampling_label_map")(label_map)  # Create 4x4 condition map

    # Noise pathway
    x = layers.Dense(4 * 4 * 255, use_bias=False, name="upsampling_noise_projection")(noise_input)  # Project noise spatially
    x = layers.BatchNormalization(name="upsampling_initial_bn")(x)  # Stabilise activations
    x = layers.LeakyReLU(alpha=0.2, name="upsampling_initial_activation")(x)  # Preserve negative gradients
    x = layers.Reshape((4, 4, 255), name="upsampling_noise_map")(x)  # Create initial 4x4 feature map

    x = layers.Concatenate(name="upsampling_initial_concatenation")([x, label_map])  # Combine noise and class condition

    def upsample_block(x, filters, res_name):
        x = layers.UpSampling2D(size=(2, 2), interpolation="nearest", name=f"upsampling_{res_name}")(x)  # Increase spatial resolution
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"upsampling_conv_{res_name}")(x)  # Learn features after resizing
        x = layers.BatchNormalization(name=f"upsampling_bn_{res_name}")(x)
        x = layers.LeakyReLU(alpha=0.2, name=f"upsampling_activation_{res_name}")(x)
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"upsampling_refinement_conv_{res_name}")(x)  # Refine spatial features
        x = layers.BatchNormalization(name=f"upsampling_refinement_bn_{res_name}")(x)
        x = layers.LeakyReLU(alpha=0.2, name=f"upsampling_refinement_activation_{res_name}")(x)
        return x

    x = upsample_block(x, 128, "8x8")  # Upsample from 4x4 to 8x8
    x = upsample_block(x, 64, "16x16")  # Upsample from 8x8 to 16x16
    x = upsample_block(x, 32, "32x32")  # Upsample to the final CIFAR-10 resolution

    output_image = layers.Conv2D(CHANNELS, 5, padding="same", activation="tanh",
                                 name="upsampling_generator_output")(x)  # Produce RGB pixels in [-1, 1]

    return tf.keras.Model([noise_input, label_input], output_image, name="cGAN_Generator_Upsampling")

In [ ]:
upsampling_generator_test = build_generator_upsampling(latent_dim=128)  # Build the architecture before full training

test_noise = tf.random.normal(shape=(4, 128), seed=SEED)  # Create four reproducible latent vectors
test_labels = tf.constant([[0], [1], [2], [3]], dtype=tf.int32)  # Request four different CIFAR-10 classes
test_images = upsampling_generator_test([test_noise, test_labels], training=False)  # Run a forward pass without training updates

print("Output shape:", test_images.shape)
print("Min/Max value:", float(tf.reduce_min(test_images)), "/", float(tf.reduce_max(test_images)))
print("Generator parameters:", upsampling_generator_test.count_params())

assert test_images.shape == (4, 32, 32, 3)  # Confirm valid CIFAR-10 image dimensions
assert -1.01 <= float(tf.reduce_min(test_images)) and float(tf.reduce_max(test_images)) <= 1.01  # Confirm the tanh output range
print("UpSampling Generator validation passed.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Output shape: (4, 32, 32, 3)
Min/Max value: -0.18228211998939514 / 0.06886711716651917
Generator parameters: 1124679
UpSampling Generator validation passed.


### 19.2 Training and weight preservation

In [ ]:
# Keep the auxiliary classifier in memory for evaluation after training.
# The default checkpoint interval from train() is used.

UPSAMPLING_MODEL_NAME = "upsampling_cdcgan_batch64"  # Unique name for saved outputs
UPSAMPLING_EPOCHS = 100  # Match the full-length tuned reference
LATENT_DIM, BATCH_SIZE = 128, 64  # Keep the selected tuned hyperparameters

tf.keras.utils.set_random_seed(SEED)  # Reset TensorFlow randomness for reproducibility
np.random.seed(SEED)  # Reset NumPy randomness

generator = build_generator_upsampling(latent_dim=LATENT_DIM)  # Apply only the upsampling modification
discriminator = build_discriminator()   # same Discriminator as the tuned cDCGAN

generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Keep baseline Generator settings
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Keep baseline Discriminator settings
train_step = make_train_step()  # Compile training with the new Generator

gen_loss_history, disc_loss_history = [], []  # Reset losses for this experiment
NUM_EXAMPLES = NUM_CLASSES  # Monitor one generated image per class
seed_noise = tf.random.normal(shape=(NUM_EXAMPLES, LATENT_DIM), seed=SEED)  # Fixed preview noise
seed_labels = tf.constant(np.arange(NUM_CLASSES).reshape(-1, 1), dtype=tf.int32)  # One label per class

upsampling_dataset = make_gan_dataset(
    images=x_train_gan,
    labels=y_train_model,
    batch_size=BATCH_SIZE,
    seed=SEED
)  # Rebuild the shared dataset using batch size 64

print(f"Experiment: {UPSAMPLING_MODEL_NAME} | latent_dim={LATENT_DIM} | batch_size={BATCH_SIZE} "
      f"| G_lr=D_lr=2e-4 | epochs={UPSAMPLING_EPOCHS}")  # Confirm the controlled configuration

train(
    dataset=upsampling_dataset,
    epochs=UPSAMPLING_EPOCHS,
    display_every=25,
    tag=UPSAMPLING_MODEL_NAME
)  # Train the isolated UpSampling2D ablation

<Figure size 2000x300 with 10 Axes>

Epoch 100/100  gen_loss=2.8996  disc_loss=0.7507  epoch_time=15.2s  elapsed=22.4min


In [ ]:
save_final_gan_weights(UPSAMPLING_MODEL_NAME)

Generator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/upsampling_cdcgan_batch64/upsampling_cdcgan_batch64_generator_final.weights.h5
Discriminator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/upsampling_cdcgan_batch64/upsampling_cdcgan_batch64_discriminator_final.weights.h5
Results saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/upsampling_cdcgan_batch64/upsampling_cdcgan_batch64_results_log.json


### 19.3 Evaluation

In [ ]:
gen_images, _ = generate_balanced_evaluation_images(generator, LATENT_DIM, FID_KL_SAMPLE_SIZE, seed=SEED)  # Generate a balanced and reproducible evaluation set

up_fid = calculate_fid(REAL_EVAL_IMAGES, gen_images)  # Measure distributional similarity
up_kid = calculate_kid(REAL_EVAL_IMAGES, gen_images)  # Complement FID with a kernel-based estimate
up_precision, up_recall = calculate_precision_recall(REAL_EVAL_IMAGES, gen_images)  # Measure fidelity and diversity
up_density, up_coverage = calculate_density_coverage(REAL_EVAL_IMAGES, gen_images)  # Assess quality and real-data coverage
up_pixel_kl = calculate_pixel_kl(REAL_EVAL_IMAGES, gen_images)  # Compare pixel-value distributions
up_conditioning_accuracy, up_per_class_accuracy = class_conditioning_accuracy(generator, LATENT_DIM, cifar_classifier)  # Test class-label fidelity
up_is_mean, up_is_std = classifier_based_inception_score(generator, LATENT_DIM, cifar_classifier)  # Estimate recognisability and diversity
up_per_class_fid, up_per_class_kl = per_class_evaluation(generator, LATENT_DIM, x_test_original, y_test_model, n_samples_per_class=1000)  # Identify class-level performance

print(f"FID: {up_fid:.2f}  KID: {up_kid:.4f}  Precision: {up_precision:.3f}  Recall: {up_recall:.3f}")
print(f"Density: {up_density:.3f}  Coverage: {up_coverage:.3f}  ClassAcc: {up_conditioning_accuracy:.1%}")
print(f"Diversity score: {up_is_mean:.3f} +/- {up_is_std:.3f}")

show_labeled_generation_grid(generator, LATENT_DIM, CLASS_NAMES, samples_per_class=8,
                             title=f"{UPSAMPLING_MODEL_NAME} -- Generated Samples by Class")  # Visually inspect eight outputs per class

log_results(
    model_name=UPSAMPLING_MODEL_NAME,
    overall_fid=up_fid, overall_kl=up_pixel_kl, overall_kid=up_kid,
    precision=up_precision, recall=up_recall, density=up_density, coverage=up_coverage,
    per_class_fid=up_per_class_fid, per_class_kl=up_per_class_kl,
    inception_score=(up_is_mean, up_is_std), class_cond_accuracy=up_conditioning_accuracy,
    epochs_trained=UPSAMPLING_EPOCHS,
    notes="UpSampling2D+Conv2D (isolated ablation, refinement conv retained at every stage), "
          "batch_size=64, 100 epochs."
)  # Add the UpSampling2D ablation to the shared comparison log

show_results_table()  # Display the updated model-comparison table

airplane     | FID:    274.57 | Pixel KL: 0.0284
automobile   | FID:    245.45 | Pixel KL: 0.0054
bird         | FID:    446.65 | Pixel KL: 0.0314
cat          | FID:    413.62 | Pixel KL: 0.0204
deer         | FID:    211.60 | Pixel KL: 0.0179
dog          | FID:    792.47 | Pixel KL: 0.0128
frog         | FID:    240.26 | Pixel KL: 0.0337
horse        | FID:    389.51 | Pixel KL: 0.0169
ship         | FID:    233.99 | Pixel KL: 0.0068
truck        | FID:    178.09 | Pixel KL: 0.0058
FID: 137.34  KID: 0.0503  Precision: 0.666  Recall: 0.410
Density: 1.280  Coverage: 0.937  ClassAcc: 64.2%
Diversity score: 4.708 +/- 0.207


<Figure size 1840x2600 with 80 Axes>

Logged results for 'upsampling_cdcgan_batch64'.
Model                  Epochs  FID       KID       Prec    Rec     Dens    Cov     IS               ClassAcc  Notes
------------------------------------------------------------------------------------------------------------------------------------------------------
baseline_cDCGAN        100     170.29    0.0893    0.683   0.389   1.378   0.953   4.74+/-0.34      64.4%     Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage.
baseline_ttur_dlr1e4   100     198.09    0.1049    0.701   0.337   1.524   0.932   4.33+/-0.17      60.0%     TTUR-inspired asymmetric learning-rate stabilisation. Generator LR=0.0002 and Discriminator LR=0.0001. Architecture, batch size, latent dimension, BCE loss and one-sided label smoothing were unchanged from the baseline.
latent64               50      162.52    0.0640    0.679   0.366   1.337   0.911   -                53.7%     Hyperparameter screening: latent_dim=64, batch_size=128, g_

### 19.4 Eye-power qualitative assessment — UpSampling2D ablation

Eight generated samples per class were manually inspected for recognisable class-specific features, consistency across samples and confusion with other CIFAR-10 classes.

| Class | Clear | Marginal | Nonsense | Notes |
|---|---:|---:|---:|---|
| airplane | ✓ | | | Most samples contain recognisable aircraft-like bodies or wings together with appropriate sky backgrounds, although some structures remain distorted. |
| automobile | ✓ | | | Most samples contain recognisable automobile bodies and appropriate road-like backgrounds, with reasonable variation in colour and viewpoint. |
| bird | | ✓ | | Several samples contain bird-like bodies, heads or beaks, but others remain ambiguous or resemble other animals. |
| cat | | ✓ | | Several samples contain recognisable feline-like features, but body and facial structures remain inconsistent across the generated samples. |
| deer | ✓ | | | Most samples contain four-legged deer-like forms together with appropriate natural backgrounds, although some remain slightly ambiguous. |
| dog | | ✓ | | Several samples contain dog-like faces or bodies, but substantial variation and confusion with other animal classes remain. |
| frog | ✓ | | | Most samples contain recognisable frog- or toad-like bodies, textures and natural colours, representing a noticeable qualitative improvement over earlier models. |
| horse | | ✓ | | Some samples contain recognisable horse-like bodies, but several are distorted or resemble other four-legged animals. |
| ship | ✓ | | | Most samples clearly resemble boats or ships and contain appropriate water and sky backgrounds. |
| truck | ✓ | | | Most samples contain recognisable large vehicle bodies, cabins or truck-like structures, although some remain distorted. |

Overall, the UpSampling2D model produced recognisable class-specific structure across several categories, particularly airplanes, automobiles, deer, frogs, ships and trucks. The samples also show variation in colour, orientation and background, suggesting that the improvement was not simply caused by repeatedly generating a small number of visually similar examples.

The remaining weaknesses are concentrated mainly among visually overlapping animal categories such as birds, cats, dogs and horses. This is consistent with the earlier EDA, which identified greater within-class variability and visual overlap among several animal categories. The persistence of these errors suggests that improving the upsampling mechanism helps the Generator form more coherent global structures, but does not completely solve fine-grained separation between visually similar classes.

The qualitative improvement is therefore evaluated together with the quantitative metrics rather than from the generated samples alone.

### 19.5 Loss curve interpretation

The UpSampling2D model retained the same general adversarial training behaviour observed in the earlier cDCGAN configurations. Replacing `Conv2DTranspose` with `UpSampling2D + Conv2D` changes how spatial resolution is increased in the Generator, but does not change the BCE adversarial objective itself.

No severe instability such as exploding losses or sudden divergence was observed during training. However, the loss curves alone do not reveal the substantial improvement later observed in FID and KID. This again demonstrates that Generator and Discriminator losses are mainly useful for monitoring adversarial training dynamics and should not be treated as direct measurements of generated-image quality.

### 19.6 Metric-by-metric interpretation

Because UpSampling2D was introduced as an isolated architectural change to the tuned batch-size-64 configuration, the fairest comparison is against `batch64_full100`.

- **FID:** decreased from **161.07 to 137.34**, a reduction of approximately **14.7%**, indicating substantially better similarity between the generated and real CIFAR-10 distributions.
- **KID:** decreased from **0.0788 to 0.0503**, providing further evidence of improved distributional similarity.
- **Precision:** decreased from **0.696 to 0.666**, indicating a small trade-off in the concentration of generated samples within realistic regions of feature space.
- **Recall:** increased from **0.392 to 0.410**, suggesting that the Generator captured a slightly broader range of the real-data distribution.
- **Density:** decreased from **1.530 to 1.280**, indicating that generated samples were less densely concentrated around real samples.
- **Coverage:** decreased slightly from **0.947 to 0.937**, showing a small trade-off in real-distribution coverage.
- **Class-conditioning accuracy:** increased slightly from **63.7% to 64.2%**, indicating that the improvement in FID and KID was achieved without sacrificing conditional class fidelity.

The strongest improvements were therefore in **FID and KID**, while Recall and class-conditioning accuracy were maintained or slightly improved. Precision, Density and Coverage decreased, demonstrating that the architectural change introduced a trade-off rather than improving every aspect of generation simultaneously.

The per-class evaluation also shows that difficulty remains uneven across categories. In particular, **dog obtained the highest per-class FID (792.47)**, followed by bird and cat among the more difficult categories. This is consistent with the earlier EDA and qualitative analysis, where visually overlapping animal categories were identified as more difficult to model than structurally distinctive vehicle classes.

Overall, `UpSampling2D + Conv2D` was considered a **successful architectural improvement**. It substantially improved overall distributional similarity while preserving approximately the same class-conditioning performance. The remaining animal-class confusion suggests that further improvements should focus on model capacity and class-specific structural learning rather than the upsampling mechanism alone.

## 20. Self-Attention Ablation

### Purpose and motivation

Our two prior architectural/procedural experiments (TTUR, UpSampling2D) each targeted a distinct, specific weakness — training-dynamics imbalance and upsampling artefacts, respectively. This experiment targets a third, different kind of limitation: **spatial coherence**. Reviewing our generated image grids up to this point (Sections 14, 19), we observed a recurring qualitative
pattern across multiple classes: generated images often showed broadly plausible colour and texture, but structurally inconsistent or implausible object geometry — for example, animal limbs or vehicle parts positioned in ways that did not cohere into a single, physically consistent object, even when individual local regions of the image looked reasonable in isolation.

Standard convolutional layers are inherently local: each output value depends only on a small, fixed-size neighbourhood of the input, defined by the convolution kernel's receptive field. This
locality is well suited to capturing fine texture and edge detail, but is a poor fit for enforcing *long-range* structural consistency — ensuring, for instance, that the left and right sides of a
generated animal's body remain coherent with each other, since a small convolutional kernel several layers deep may still have no direct access to information from the opposite side of the
image.

Self-Attention GANs (SAGAN) were proposed to directly address this gap,
incorporating a self-attention mechanism that allows every
spatial location in a feature map to attend to every other location, not just a local neighbourhood — giving the network an explicit mechanism to model exactly the kind of long-range dependency our generated images appeared to be missing. The original SAGAN paper reported substantial FID and Inception Score improvements on ImageNet class-conditional generation, attributed specifically to improved geometric and structural consistency.

**Why we test this as an isolated ablation, at this specific resolution:** we incorporate a single self-attention block at the 16×16 resolution stage in both Generator and Discriminator — following SAGAN's own finding that a well-placed attention layer at a mid-range resolution is typically sufficient to capture the relevant long-range dependencies, without the substantially
higher computational cost of applying attention at every resolution stage. This experiment directly tests whether our qualitative observation of spatial incoherence has a specific, addressable architectural cause, and — critically — whether it improves our per-class results for `dog`, our persistently hardest class throughout this project, since spatial/structural
consistency issues would plausibly affect a high-pose-variability class like `dog` more severely than a structurally simple, consistent class like `truck` or `ship`.

### 20.1 Self-attention layer


In [ ]:
class SelfAttention(layers.Layer):
    def __init__(self, channels, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels  # Number of input and output feature channels
        self.query_conv = layers.Conv2D(channels // 8, 1, padding='same', name="attn_query")  # Learn what each location searches for
        self.key_conv = layers.Conv2D(channels // 8, 1, padding='same', name="attn_key")  # Learn what each location represents
        self.value_conv = layers.Conv2D(channels, 1, padding='same', name="attn_value")  # Learn the information to combine

    def build(self, input_shape):
        self.gamma = self.add_weight(name="attn_gamma", shape=[1], initializer="zeros", trainable=True)  # Learn attention strength gradually
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]  # Support dynamic batch sizes
        height, width = x.shape[1], x.shape[2]  # Preserve the spatial dimensions

        query = self.query_conv(x)  # Create query feature maps
        key = self.key_conv(x)  # Create key feature maps
        value = self.value_conv(x)  # Create value feature maps

        query_flat = tf.reshape(query, [batch_size, height * width, -1])  # Flatten spatial locations
        key_flat = tf.reshape(key, [batch_size, height * width, -1])
        value_flat = tf.reshape(value, [batch_size, height * width, -1])

        attention_scores = tf.matmul(query_flat, key_flat, transpose_b=True)  # Compare every spatial location
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)  # Convert similarities into attention weights

        out = tf.matmul(attention_weights, value_flat)  # Combine information from related locations
        out = tf.reshape(out, [batch_size, height, width, self.channels])  # Restore the original feature-map shape

        return self.gamma * out + x  # Add attention through a learnable residual connection

### 20.2 Attention-based Generator and Discriminator


In [ ]:
def build_generator_attention(latent_dim=128, num_classes=NUM_CLASSES):
    noise_input = layers.Input(shape=(latent_dim,))  # Random latent vector
    label_input = layers.Input(shape=(1,), dtype='int32')  # Requested class label

    label_embedding = layers.Embedding(num_classes, 50)(label_input)  # Learn class representation
    label_embedding = layers.Flatten()(label_embedding)
    label_map = layers.Dense(4 * 4)(label_embedding)  # Project label spatially
    label_map = layers.Reshape((4, 4, 1))(label_map)  # Create 4x4 condition map

    x = layers.Dense(4 * 4 * 255, use_bias=False)(noise_input)  # Project noise into spatial features
    x = layers.BatchNormalization()(x)  # Stabilise activations
    x = layers.LeakyReLU(alpha=0.2)(x)  # Preserve negative gradients
    x = layers.Reshape((4, 4, 255))(x)  # Create initial 4x4 feature map

    x = layers.Concatenate()([x, label_map])   # Combine noise and label into a 4x4x256 representation

    # 4x4 -> 8x8, with refinement (unchanged from batch64_full100)
    x = layers.Conv2DTranspose(128, 4, strides=2, padding='same', use_bias=False)(x)  # Increase spatial resolution
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)  # Refine 8x8 features
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    # 8x8 -> 16x16, with refinement
    x = layers.Conv2DTranspose(64, 4, strides=2, padding='same', use_bias=False)(x)  # Increase spatial resolution
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(64, 3, padding='same', use_bias=False)(x)  # Refine 16x16 features
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    # --- Self-attention at 16x16 (the one new component) ---
    x = SelfAttention(channels=64, name="generator_self_attention_16x16")(x)  # Coordinate information across distant regions

    # 16x16 -> 32x32, with refinement (unchanged)
    x = layers.Conv2DTranspose(32, 4, strides=2, padding='same', use_bias=False)(x)  # Reach CIFAR-10 resolution
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(x)  # Refine full-resolution features
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    output_img = layers.Conv2D(3, 5, padding='same', activation='tanh')(x)  # Produce RGB output in [-1, 1]
    return tf.keras.Model([noise_input, label_input], output_img, name="cGAN_Generator_Attention")

In [ ]:
def build_generator_attention(latent_dim=128, num_classes=NUM_CLASSES):
    noise_input = layers.Input(shape=(latent_dim,))  # Random latent vector
    label_input = layers.Input(shape=(1,), dtype='int32')  # Requested class label

    label_embedding = layers.Embedding(num_classes, 50)(label_input)  # Learn class representation
    label_embedding = layers.Flatten()(label_embedding)
    label_map = layers.Dense(4 * 4)(label_embedding)  # Project label spatially
    label_map = layers.Reshape((4, 4, 1))(label_map)  # Create 4x4 condition map

    x = layers.Dense(4 * 4 * 255, use_bias=False)(noise_input)  # Project noise into spatial features
    x = layers.BatchNormalization()(x)  # Stabilise activations
    x = layers.LeakyReLU(alpha=0.2)(x)  # Preserve negative gradients
    x = layers.Reshape((4, 4, 255))(x)  # Create initial 4x4 feature map

    x = layers.Concatenate()([x, label_map])   # Combine noise and label into a 4x4x256 representation

    # 4x4 -> 8x8, with refinement (unchanged from batch64_full100)
    x = layers.Conv2DTranspose(128, 4, strides=2, padding='same', use_bias=False)(x)  # Increase spatial resolution
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)  # Refine 8x8 features
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    # 8x8 -> 16x16, with refinement
    x = layers.Conv2DTranspose(64, 4, strides=2, padding='same', use_bias=False)(x)  # Increase spatial resolution
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(64, 3, padding='same', use_bias=False)(x)  # Refine 16x16 features
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    # --- Self-attention at 16x16 (the one new component) ---
    x = SelfAttention(channels=64, name="generator_self_attention_16x16")(x)  # Coordinate information across distant regions

    # 16x16 -> 32x32, with refinement (unchanged)
    x = layers.Conv2DTranspose(32, 4, strides=2, padding='same', use_bias=False)(x)  # Reach CIFAR-10 resolution
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(x)  # Refine full-resolution features
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    output_img = layers.Conv2D(3, 5, padding='same', activation='tanh')(x)  # Produce RGB output in [-1, 1]
    return tf.keras.Model([noise_input, label_input], output_img, name="cGAN_Generator_Attention")

### 20.3 Architecture validation


In [ ]:
attn_generator_test = build_generator_attention(latent_dim=128)  # Build the attention Generator for validation
attn_discriminator_test = build_discriminator_attention()  # Build the corresponding attention Discriminator


test_noise = tf.random.normal(shape=(4, 128), seed=SEED)  # Create four reproducible latent vectors
test_labels = tf.constant([[0], [1], [2], [3]], dtype=tf.int32)  # Request four different classes
test_images = attn_generator_test([test_noise, test_labels], training=False)  # Perform a forward pass without training


print("Output shape:", test_images.shape)  # Confirm CIFAR-10 image dimensions
print("Min/Max:", float(tf.reduce_min(test_images)), "/", float(tf.reduce_max(test_images)))  # Check tanh output range
print("Generator parameters:", attn_generator_test.count_params())  # Inspect added model capacity


assert test_images.shape == (4, 32, 32, 3)  # Verify correct batch and RGB image shape
assert -1.01 <= float(tf.reduce_min(test_images)) and float(tf.reduce_max(test_images)) <= 1.01  # Verify values remain in [-1, 1]


test_logits = attn_discriminator_test([test_images, test_labels], training=False)  # Test generated image-label pairs
print("Discriminator output shape:", test_logits.shape)
assert test_logits.shape == (4, 1)  # Verify one real/fake logit per image


print("Self-attention architecture validation passed.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Output shape: (4, 32, 32, 3)
Min/Max: -0.020992759615182877 / 0.01667017489671707
Generator parameters: 1430936
Discriminator output shape: (4, 1)
Self-attention architecture validation passed.


### 20.4 Training configuration and execution


In [ ]:
ATTENTION_MODEL_NAME = "attention_cdcgan_batch64"  # Unique name for saved outputs
ATTENTION_EPOCHS = 100  # Match the full-length tuned reference
LATENT_DIM, BATCH_SIZE = 128, 64  # Keep the selected tuned hyperparameters


tf.keras.utils.set_random_seed(SEED)  # Reset TensorFlow randomness for reproducibility
np.random.seed(SEED)  # Reset NumPy randomness


generator = build_generator_attention(latent_dim=LATENT_DIM)  # Build the self-attention Generator
discriminator = build_discriminator_attention()  # Build the self-attention Discriminator


generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Keep the baseline Generator settings
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Keep the baseline Discriminator settings
train_step = make_train_step()  # Compile training with the attention models


gen_loss_history, disc_loss_history = [], []  # Reset losses for this experiment
NUM_EXAMPLES = NUM_CLASSES  # Monitor one generated image per class
seed_noise = tf.random.normal(shape=(NUM_EXAMPLES, LATENT_DIM), seed=SEED)  # Fixed preview noise
seed_labels = tf.constant(np.arange(NUM_CLASSES).reshape(-1, 1), dtype=tf.int32)  # One label per class


attention_dataset = make_gan_dataset(
    images=x_train_gan,
    labels=y_train_model,
    batch_size=BATCH_SIZE,
    seed=SEED
)  # Rebuild the shared dataset using batch size 64


print(f"Experiment: {ATTENTION_MODEL_NAME} | latent_dim={LATENT_DIM} | batch_size={BATCH_SIZE} "
      f"| G_lr=D_lr=2e-4 | epochs={ATTENTION_EPOCHS}")  # Confirm the controlled configuration


train(
    dataset=attention_dataset,
    epochs=ATTENTION_EPOCHS,
    display_every=25,
    tag=ATTENTION_MODEL_NAME
)  # Train the isolated self-attention ablation

<Figure size 2000x300 with 10 Axes>

Epoch 100/100  gen_loss=3.3981  disc_loss=0.6832  epoch_time=19.3s  elapsed=27.7min


### 20.5 Save trained weights


In [ ]:
save_final_gan_weights(ATTENTION_MODEL_NAME)

Generator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/attention_cdcgan_batch64/attention_cdcgan_batch64_generator_final.weights.h5
Discriminator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/attention_cdcgan_batch64/attention_cdcgan_batch64_discriminator_final.weights.h5
Results saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/attention_cdcgan_batch64/attention_cdcgan_batch64_results_log.json


### 20.6 Quantitative and qualitative evaluation


In [ ]:
gen_images, _ = generate_balanced_evaluation_images(
    generator,
    LATENT_DIM,
    FID_KL_SAMPLE_SIZE,
    seed=SEED
)  # Generate equal numbers from all classes for fair evaluation


attn_fid = calculate_fid(REAL_EVAL_IMAGES, gen_images)  # Measure overall feature-distribution similarity
attn_kid = calculate_kid(REAL_EVAL_IMAGES, gen_images)  # Complement FID with an unbiased distance estimate
attn_precision, attn_recall = calculate_precision_recall(REAL_EVAL_IMAGES, gen_images)  # Measure fidelity and diversity
attn_density, attn_coverage = calculate_density_coverage(REAL_EVAL_IMAGES, gen_images)  # Measure local concentration and coverage
attn_pixel_kl = calculate_pixel_kl(REAL_EVAL_IMAGES, gen_images)  # Compare low-level pixel distributions
attn_conditioning_accuracy, attn_per_class_accuracy = class_conditioning_accuracy(generator, LATENT_DIM, cifar_classifier)  # Check requested-class fidelity
attn_is_mean, attn_is_std = classifier_based_inception_score(generator, LATENT_DIM, cifar_classifier)  # Estimate class clarity and diversity
attn_per_class_fid, attn_per_class_kl = per_class_evaluation(generator, LATENT_DIM, x_test_original, y_test_model, n_samples_per_class=1000)  # Identify class-specific strengths and weaknesses


print(f"FID: {attn_fid:.2f}  KID: {attn_kid:.4f}  Precision: {attn_precision:.3f}  Recall: {attn_recall:.3f}")
print(f"Density: {attn_density:.3f}  Coverage: {attn_coverage:.3f}  ClassAcc: {attn_conditioning_accuracy:.1%}")  # Summarise overall results


show_labeled_generation_grid(
    generator,
    LATENT_DIM,
    CLASS_NAMES,
    samples_per_class=8,
    title=f"{ATTENTION_MODEL_NAME} -- Generated Samples by Class"
)  # Visually inspect consistency and class recognisability


log_results(
    model_name=ATTENTION_MODEL_NAME,
    overall_fid=attn_fid, overall_kl=attn_pixel_kl, overall_kid=attn_kid,
    precision=attn_precision, recall=attn_recall, density=attn_density, coverage=attn_coverage,
    per_class_fid=attn_per_class_fid, per_class_kl=attn_per_class_kl,
    inception_score=(attn_is_mean, attn_is_std), class_cond_accuracy=attn_conditioning_accuracy,
    epochs_trained=ATTENTION_EPOCHS,
    notes="Self-attention (SAGAN-style, Zhang et al. 2018) at 16x16 in both Generator and "
          "Discriminator, isolated ablation on top of batch64_full100 (Conv2DTranspose), "
          "targeting the spatial/positional coherence issue observed in earlier models."
)  # Store results using the shared comparison format

show_results_table()  # Compare self-attention against all previous experiments

airplane     | FID:    228.42 | Pixel KL: 0.0063
automobile   | FID:    266.51 | Pixel KL: 0.0164
bird         | FID:    487.59 | Pixel KL: 0.0249
cat          | FID:    442.76 | Pixel KL: 0.0171
deer         | FID:    216.81 | Pixel KL: 0.0123
dog          | FID:   1028.10 | Pixel KL: 0.0052
frog         | FID:    208.80 | Pixel KL: 0.0597
horse        | FID:    431.67 | Pixel KL: 0.0025
ship         | FID:    214.95 | Pixel KL: 0.0148
truck        | FID:    207.59 | Pixel KL: 0.0093
FID: 163.62  KID: 0.0797  Precision: 0.660  Recall: 0.410
Density: 1.388  Coverage: 0.939  ClassAcc: 58.7%


<Figure size 1840x2600 with 80 Axes>

Logged results for 'attention_cdcgan_batch64'.
Model                  Epochs  FID       KID       Prec    Rec     Dens    Cov     IS               ClassAcc  Notes
------------------------------------------------------------------------------------------------------------------------------------------------------
baseline_cDCGAN        100     170.29    0.0893    0.683   0.389   1.378   0.953   4.74+/-0.34      64.4%     Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage.
baseline_ttur_dlr1e4   100     198.09    0.1049    0.701   0.337   1.524   0.932   4.33+/-0.17      60.0%     TTUR-inspired asymmetric learning-rate stabilisation. Generator LR=0.0002 and Discriminator LR=0.0001. Architecture, batch size, latent dimension, BCE loss and one-sided label smoothing were unchanged from the baseline.
latent64               50      162.52    0.0640    0.679   0.366   1.337   0.911   -                53.7%     Hyperparameter screening: latent_dim=64, batch_size=128, g_l

### 20.7 Eye-power qualitative assessment — Self-attention ablation

Eight generated samples per class were manually inspected for recognisable class-specific features, consistency across samples and confusion with other CIFAR-10 classes.

| Class | Clear | Marginal | Nonsense | Notes |
|---|---:|---:|---:|---|
| airplane | ✓ | | | Most samples contain recognisable aircraft-like bodies, wings or tails together with appropriate sky or runway backgrounds, although some remain distorted. |
| automobile | ✓ | | | Most samples clearly resemble automobiles, with recognisable vehicle bodies and appropriate road-like backgrounds. |
| bird | | ✓ | | Several samples contain bird-like bodies, beaks or silhouettes, but others remain ambiguous or resemble other animals. |
| cat | | ✓ | | Several samples contain feline-like faces, ears or body shapes, but consistency is limited and some resemble other animals. |
| deer | ✓ | | | Most samples contain four-legged deer-like structures and suitable natural backgrounds, although some remain slightly distorted. |
| dog | | ✓ | | Some dog-like features are present, but many samples are structurally ambiguous and show substantial confusion with other animal classes. |
| frog | ✓ | | | Several samples contain recognisable frog-like bodies, faces and natural green or brown colours, although some remain distorted. |
| horse | | ✓ | | Some samples contain horse-like bodies and outdoor backgrounds, but several are anatomically distorted or resemble other four-legged animals. |
| ship | ✓ | | | Most samples clearly resemble boats or ships and contain appropriate water and sky backgrounds. |
| truck | ✓ | | | Most samples contain recognisable large vehicle or truck-like structures, although some remain blurred or distorted. |

Overall, the self-attention model produced recognisable structures for several classes, particularly airplanes, automobiles, deer, frogs, ships and trucks. The samples also display variation in colour, pose and background rather than repeatedly producing identical outputs.

However, ambiguity remained among visually similar animal classes, particularly birds, cats, dogs and horses. This is consistent with the earlier EDA, which identified greater visual overlap and within-class variation among several animal categories. Therefore, adding self-attention alone did not eliminate the class-separation difficulties identified earlier in the dataset.

Although self-attention was introduced to help the networks model relationships between spatially separated image regions, the qualitative results suggest that this additional mechanism alone was insufficient to consistently produce more accurate fine-grained animal structures at CIFAR-10 resolution.

### 20.8 Loss curve interpretation

The self-attention model's loss curves showed a gradual overall drift without sudden divergence, exploding values or other clear signs of training collapse. Therefore, the attention mechanism could be incorporated and trained successfully within the existing adversarial framework.

However, stable training does not necessarily imply improved generation quality. The quantitative evaluation shows that Recall improved slightly, while several other metrics, particularly Precision and class-conditioning accuracy, deteriorated compared with the tuned reference model.

As with the previous experiments, the Generator and Discriminator losses are therefore used mainly to understand adversarial training behaviour. Model selection is based on the generated images together with the larger-sample quantitative evaluation.

### 20.9 Metric-by-metric interpretation

Because self-attention was introduced as an isolated modification to `batch64_full100`, the fairest comparison is against that model rather than against the separate UpSampling2D ablation.

- **FID:** increased from **161.07 to 163.62**, indicating a small deterioration in overall similarity between the generated and real distributions.
- **KID:** increased slightly from **0.0788 to 0.0797**, supporting the FID result that self-attention did not improve overall distributional similarity.
- **Precision:** decreased from **0.696 to 0.660**, suggesting lower generated-sample fidelity.
- **Recall:** increased from **0.392 to 0.410**, indicating that the Generator represented a somewhat broader range of the real-data distribution.
- **Density:** decreased from **1.530 to 1.388**, indicating lower concentration of generated samples around real-data neighbourhoods.
- **Coverage:** decreased slightly from **0.947 to 0.939**.
- **Class-conditioning accuracy:** decreased from **63.7% to 58.7%**, a reduction of **5.0 percentage points**, indicating substantially weaker correspondence between the requested class labels and generated images.

The per-class evaluation further shows that performance remained uneven across categories. **Dog had by far the highest per-class FID at 1028.10**, while birds, cats and horses also recorded relatively high FID values. These are predominantly visually overlapping animal categories, which is consistent with the difficulty identified during the EDA and observed qualitatively in the earlier models.

Self-attention therefore introduced a trade-off: **Recall improved**, suggesting greater distributional diversity, but this was accompanied by poorer FID, KID, Precision, Density, Coverage and especially class-conditioning accuracy.

Overall, the self-attention ablation was **not retained as an improvement**. While the model trained successfully and slightly improved Recall, it did not provide sufficient gains in fidelity or conditional accuracy to justify including self-attention in the subsequent combined architecture.

## 21. Wider/Deeper Architecture

### Purpose and motivation

At this point in our experiments, we had tested three distinct, targeted interventions — a training-dynamics fix (TTUR), an upsampling-artefact fix (UpSampling2D), and a spatial-coherence fix (self-attention) — and only one (UpSampling2D) had produced a consistent, if modest, improvement. Critically, **self-attention had failed to improve, and in some sessions worsened,
our per-class results for `dog`** — the class it was specifically intended to help, given its persistent, severe per-class FID gap throughout every model tested so far. This was an important signal: if a mechanism specifically designed to improve spatial/structural consistency did not
measurably help our hardest, highest-variability class, the limiting factor might not be a *specific mechanism* our architecture was missing, but something more fundamental — **insufficient representational capacity** to model the genuine complexity our EDA had already identified in classes like `dog` (Section 3A.9: high within-class feature-space variability, strong overlap with `cat`).

Brock, Donahue, and Simonyan (2018), in their BigGAN work on large-scale ImageNet generation, provide direct empirical support for this hypothesis at a much larger scale: they found that simply increasing parameter count (wider layers) and batch size in an otherwise standard conditional GAN produced substantial, consistent Inception Score and FID improvements, without
requiring more sophisticated architectural mechanisms — suggesting that, at least within an appropriate range, capacity itself can be an under-exploited lever for improvement, separate from and not subordinate to targeted mechanism-level changes.

**Why we expected this specifically to benefit our results, given what we had already observed:** our baseline and subsequent experiments consistently showed a very large gap between our easiest classes (`truck`, `ship`, `deer` — visually simple, structurally consistent, as confirmed in our
EDA) and our hardest classes (`dog`, `cat`, `bird` — visually complex, highly variable, as confirmed in the same EDA). A network with too little capacity would plausibly default to learning the "average," easiest-to-represent visual patterns well, while lacking the representational room to also capture the fuller range of variation present in our harder classes. If this diagnosis was correct, increasing capacity should disproportionately benefit our hardest classes specifically — not just lift every class's quality by a similar, uniform
amount — since it is precisely those classes that most exceeded our smaller architecture's apparent representational limits.

### Implementation

We increase filter width by approximately 2x at every resolution stage in both Generator and Discriminator, and add an additional refinement convolution layer at each stage, while holding every other element of our best-performing configuration (batch_size=64, Adam(2e-4, beta_1=0.5), label smoothing, standard Conv2DTranspose) unchanged, isolating capacity as the single variable
under test. Given the substantially larger parameter count, we extend training to 150 epochs (vs. our other experiments' 100), reasoning that more gradient updates would likely be needed for a larger network to converge to a comparable degree — a decision we later validate directly by comparing intermediate and final checkpoints.

### 21.1 Wider/deeper Generator


In [ ]:
def build_generator_wide_deep(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
    noise_input = layers.Input(shape=(latent_dim,))  # Random latent vector
    label_input = layers.Input(shape=(1,), dtype='int32')  # Requested class label

    # Same embedding as baseline (isolating capacity, not conditioning signal)
    label_embedding = layers.Embedding(num_classes, 50)(label_input)  # Learn class representation
    label_embedding = layers.Flatten()(label_embedding)
    label_map = layers.Dense(4 * 4)(label_embedding)  # Project the label spatially
    label_map = layers.Reshape((4, 4, 1))(label_map)  # Create 4x4 condition map

    # Wider noise pathway: 511 instead of 255 (roughly double)
    x = layers.Dense(4 * 4 * 511, use_bias=False)(noise_input)  # Increase initial feature capacity
    x = layers.BatchNormalization()(x)  # Stabilise activations
    x = layers.LeakyReLU(alpha=0.2)(x)  # Preserve negative gradients
    x = layers.Reshape((4, 4, 511))(x)  # Create initial spatial feature map

    x = layers.Concatenate()([x, label_map])   # (4, 4, 512) -- double the baseline's 256

    def wide_deep_block(x, filters, name):
        x = layers.Conv2DTranspose(filters, 4, strides=2, padding='same', use_bias=False, name=f"{name}_deconv")(x)  # Upsample spatial resolution
        x = layers.BatchNormalization(name=f"{name}_bn1")(x)
        x = layers.LeakyReLU(alpha=0.2, name=f"{name}_act1")(x)

        # First refinement conv (same as baseline)
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f"{name}_refine1")(x)  # Refine upsampled features
        x = layers.BatchNormalization(name=f"{name}_bn2")(x)
        x = layers.LeakyReLU(alpha=0.2, name=f"{name}_act2")(x)

        # Extra second refinement conv (deeper than baseline)
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f"{name}_refine2")(x)  # Add capacity for more complex structures
        x = layers.BatchNormalization(name=f"{name}_bn3")(x)
        x = layers.LeakyReLU(alpha=0.2, name=f"{name}_act3")(x)
        return x

    x = wide_deep_block(x, 256, "block_8x8")    # baseline: 128 -> now 256 (2x wider)
    x = wide_deep_block(x, 128, "block_16x16")  # baseline: 64  -> now 128 (2x wider)
    x = wide_deep_block(x, 64,  "block_32x32")  # baseline: 32  -> now 64  (2x wider)

    output_img = layers.Conv2D(3, 5, padding='same', activation='tanh')(x)  # Produce 32x32 RGB output in [-1, 1]
    return tf.keras.Model([noise_input, label_input], output_img, name="cGAN_Generator_WideDeep")

### 21.2 Wider/deeper Discriminator


In [ ]:
def build_discriminator_wide_deep(num_classes=NUM_CLASSES, img_shape=IMAGE_SHAPE):
    image_input = layers.Input(shape=img_shape)  # Real or generated RGB image
    label_input = layers.Input(shape=(1,), dtype='int32')  # Proposed class label

    label_embedding = layers.Embedding(num_classes, 50)(label_input)  # Learn class representation
    label_embedding = layers.Flatten()(label_embedding)
    label_map = layers.Dense(img_shape[0] * img_shape[1])(label_embedding)  # Project label spatially
    label_map = layers.Reshape((img_shape[0], img_shape[1], 1))(label_map)  # Create 32x32 condition map

    x = layers.Concatenate()([image_input, label_map])  # Evaluate image realism together with label compatibility

    def wide_deep_disc_block(x, filters, name):
        x = layers.Conv2D(filters, 4, strides=2, padding='same', name=f"{name}_conv")(x)  # Downsample and extract features
        x = layers.LeakyReLU(alpha=0.2, name=f"{name}_act1")(x)
        x = layers.Conv2D(filters, 3, padding='same', name=f"{name}_refine1")(x)  # Refine features at the current resolution
        x = layers.LeakyReLU(alpha=0.2, name=f"{name}_act2")(x)
        x = layers.Conv2D(filters, 3, padding='same', name=f"{name}_refine2")(x)   # extra depth
        x = layers.LeakyReLU(alpha=0.2, name=f"{name}_act3")(x)
        x = layers.Dropout(0.3, name=f"{name}_dropout")(x)  # Regularise the larger Discriminator
        return x

    x = wide_deep_disc_block(x, 128, "disc_16x16")   # baseline: 64  -> now 128
    x = wide_deep_disc_block(x, 256, "disc_8x8")     # baseline: 128 -> now 256
    x = wide_deep_disc_block(x, 512, "disc_4x4")     # baseline: 256 -> now 512

    x = layers.Flatten()(x)  # Convert spatial features into one vector
    output = layers.Dense(1)(x)  # Produce one raw real/fake logit
    return tf.keras.Model([image_input, label_input], output, name="cGAN_Discriminator_WideDeep")

### 21.3 Architecture validation


In [ ]:
wd_gen_test = build_generator_wide_deep(latent_dim=128)  # Build wider/deeper Generator for validation
wd_disc_test = build_discriminator_wide_deep()  # Build wider/deeper Discriminator for validation


test_noise = tf.random.normal(shape=(4, 128), seed=SEED)  # Create four reproducible latent vectors
test_labels = tf.constant([[0], [1], [2], [3]], dtype=tf.int32)  # Request four different classes
test_images = wd_gen_test([test_noise, test_labels], training=False)  # Test the Generator without updating weights


print("Output shape:", test_images.shape)
assert test_images.shape == (4, 32, 32, 3)  # Confirm valid CIFAR-10 RGB output


test_logits = wd_disc_test([test_images, test_labels], training=False)  # Test generated image-label pairs
assert test_logits.shape == (4, 1)  # Confirm one real/fake logit per image


print("Generator params:", wd_gen_test.count_params(), "(baseline was 1,425,735)")  # Verify increased Generator capacity
print("Discriminator params:", wd_disc_test.count_params(), "(baseline was 717,621)")  # Verify increased Discriminator capacity
print("Wide/deep architecture validation passed.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Output shape: (4, 32, 32, 3)
Generator params: 5391527 (baseline was 1,425,735)
Discriminator params: 8886389 (baseline was 717,621)
Wide/deep architecture validation passed.


### 21.4 Training configuration and execution


In [ ]:
tf.keras.utils.set_random_seed(SEED)  # Reset TensorFlow randomness for reproducibility
np.random.seed(SEED)  # Reset NumPy randomness

WIDE_DEEP_MODEL_NAME = "wide_deep_batch64_150ep"  # Unique name for checkpoints and outputs
LATENT_DIM, BATCH_SIZE = 128, 64  # Keep the selected tuned hyperparameters

generator = build_generator_wide_deep(latent_dim=LATENT_DIM)  # Build the higher-capacity Generator
discriminator = build_discriminator_wide_deep()  # Build the higher-capacity Discriminator
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Keep baseline Generator optimisation settings
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Keep baseline Discriminator optimisation settings
train_step = make_train_step()  # Compile training using the wider/deeper networks

gen_loss_history, disc_loss_history = [], []  # Reset loss histories for this experiment
NUM_EXAMPLES = NUM_CLASSES  # Monitor one generated image per class
seed_noise = tf.random.normal(shape=(NUM_EXAMPLES, LATENT_DIM), seed=SEED)  # Fixed preview noise
seed_labels = tf.constant(np.arange(NUM_CLASSES).reshape(-1, 1), dtype=tf.int32)  # One monitoring label per class

wide_deep_dataset = make_gan_dataset(
    images=x_train_gan,
    labels=y_train_model,
    batch_size=BATCH_SIZE,
    seed=SEED
)  # Rebuild the shared dataset using batch size 64

# checkpoint_every=50 saves weights at epoch 50, 100, and 150 automatically
train(
    dataset=wide_deep_dataset,
    epochs=150,
    display_every=25,
    checkpoint_every=50,
    tag=WIDE_DEEP_MODEL_NAME
)  # Train longer to give the larger architecture more time to converge

<Figure size 2000x300 with 10 Axes>

Epoch 150/150  gen_loss=6.5296  disc_loss=0.5241  epoch_time=25.5s  elapsed=50.2min


### 21.5 Save trained weights


In [ ]:
save_final_gan_weights(WIDE_DEEP_MODEL_NAME)

Generator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_batch64_150ep/wide_deep_batch64_150ep_generator_final.weights.h5
Discriminator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_batch64_150ep/wide_deep_batch64_150ep_discriminator_final.weights.h5
Results saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_batch64_150ep/wide_deep_batch64_150ep_results_log.json


### 21.6 Quantitative evaluation


In [ ]:
gen_images, _ = generate_balanced_evaluation_images(
    generator,
    LATENT_DIM,
    FID_KL_SAMPLE_SIZE,
    seed=SEED
)  # Generate equal numbers from all classes for fair evaluation


wd_fid = calculate_fid(REAL_EVAL_IMAGES, gen_images)  # Measure overall feature-distribution similarity
wd_kid = calculate_kid(REAL_EVAL_IMAGES, gen_images)  # Complement FID with another distribution-distance estimate
wd_precision, wd_recall = calculate_precision_recall(REAL_EVAL_IMAGES, gen_images)  # Measure fidelity and diversity
wd_density, wd_coverage = calculate_density_coverage(REAL_EVAL_IMAGES, gen_images)  # Measure local concentration and coverage
wd_pixel_kl = calculate_pixel_kl(REAL_EVAL_IMAGES, gen_images)  # Compare low-level pixel distributions
wd_conditioning_accuracy, wd_per_class_accuracy = class_conditioning_accuracy(generator, LATENT_DIM, cifar_classifier)  # Check requested-class fidelity
wd_is_mean, wd_is_std = classifier_based_inception_score(generator, LATENT_DIM, cifar_classifier)  # Estimate class clarity and diversity
wd_per_class_fid, wd_per_class_kl = per_class_evaluation(generator, LATENT_DIM, x_test_original, y_test_model, n_samples_per_class=1000)  # Identify class-specific performance


show_labeled_generation_grid(
    generator,
    LATENT_DIM,
    CLASS_NAMES,
    samples_per_class=8,
    title=f"{WIDE_DEEP_MODEL_NAME} -- Generated Samples by Class"
)  # Visually inspect class recognisability, consistency and diversity


log_results(
    model_name=WIDE_DEEP_MODEL_NAME,
    overall_fid=wd_fid, overall_kl=wd_pixel_kl, overall_kid=wd_kid,
    precision=wd_precision, recall=wd_recall, density=wd_density, coverage=wd_coverage,
    per_class_fid=wd_per_class_fid, per_class_kl=wd_per_class_kl,
    inception_score=(wd_is_mean, wd_is_std), class_cond_accuracy=wd_conditioning_accuracy,
    epochs_trained=150,
    notes="Wider (~2x filters at every stage) and deeper (extra refinement conv per stage) "
          "Generator/Discriminator, otherwise identical to batch64_full100 (same embedding, "
          "same batch size, same optimizer). Trained for 150 epochs (vs baseline's 100) to give "
          "the added capacity more time to converge."
)  # Store results using the shared comparison format

show_results_table()  # Compare the wider/deeper model with all previous experiments

airplane     | FID:    225.83 | Pixel KL: 0.0411
automobile   | FID:    237.29 | Pixel KL: 0.0141
bird         | FID:    459.63 | Pixel KL: 0.0204
cat          | FID:    459.59 | Pixel KL: 0.0060
deer         | FID:    213.35 | Pixel KL: 0.0177
dog          | FID:    794.92 | Pixel KL: 0.0283
frog         | FID:    233.53 | Pixel KL: 0.0265
horse        | FID:    406.57 | Pixel KL: 0.0077
ship         | FID:    215.62 | Pixel KL: 0.0511
truck        | FID:    183.77 | Pixel KL: 0.0143


<Figure size 1840x2600 with 80 Axes>

Logged results for 'wide_deep_batch64_150ep'.
Model                  Epochs  FID       KID       Prec    Rec     Dens    Cov     IS               ClassAcc  Notes
------------------------------------------------------------------------------------------------------------------------------------------------------
baseline_cDCGAN        100     170.29    0.0893    0.683   0.389   1.378   0.953   4.74+/-0.34      64.4%     Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage.
baseline_ttur_dlr1e4   100     198.09    0.1049    0.701   0.337   1.524   0.932   4.33+/-0.17      60.0%     TTUR-inspired asymmetric learning-rate stabilisation. Generator LR=0.0002 and Discriminator LR=0.0001. Architecture, batch size, latent dimension, BCE loss and one-sided label smoothing were unchanged from the baseline.
latent64               50      162.52    0.0640    0.679   0.366   1.337   0.911   -                53.7%     Hyperparameter screening: latent_dim=64, batch_size=128, g_lr

### 21.7 Real-versus-generated comparison


In [ ]:
show_real_vs_generated(
    generator, LATENT_DIM, x_test_original, y_test_model, CLASS_NAMES,
    samples_per_class=5,
    title="wide_deep_batch64_150ep -- Real vs Generated"
)

<Figure size 2200x2200 with 100 Axes>

### 21.8 Eye-power qualitative assessment — Wider/deeper architecture

Eight generated samples per class were manually inspected for recognisable class-specific features, consistency across samples and confusion with other CIFAR-10 classes.

| Class | Clear | Marginal | Nonsense | Notes |
|---|---:|---:|---:|---|
| airplane | ✓ | | | Most samples contain recognisable aircraft bodies, wings or tails together with appropriate sky or runway backgrounds, although some remain distorted. |
| automobile | ✓ | | | Most samples clearly resemble automobiles, with coherent vehicle bodies and reasonable variation in colour and viewpoint. |
| bird | ✓ | | | Most samples contain recognisable bird-like bodies, heads or beaks, although a few remain distorted or ambiguous. |
| cat | | ✓ | | Several samples contain feline-like faces, ears or body shapes, but consistency remains limited and some samples resemble other animals. |
| deer | ✓ | | | Most samples show recognisable deer-like bodies and natural backgrounds, although some anatomy remains imperfect. |
| dog | | ✓ | | Several samples contain dog-like faces or bodies, but some are distorted or confused with cats and other animal classes. |
| frog | | ✓ | | Several samples contain frog-like shapes and appropriate natural colours, but the body structures remain inconsistent across the samples. |
| horse | ✓ | | | Most samples contain recognisable horse-like bodies, legs and outdoor backgrounds, despite some anatomical distortion. |
| ship | ✓ | | | Most samples clearly resemble ships or boats and contain coherent water and sky backgrounds. |
| truck | ✓ | | | Most samples contain recognisable truck bodies, cabins or large-vehicle structures, with variation in colour and viewpoint. |

The wider/deeper model generated recognisable class-specific structures for most CIFAR-10 categories. Its strongest qualitative results were observed for automobiles, birds, deer, horses, ships and trucks. The samples also displayed variation in colour, orientation and background, providing no obvious qualitative indication of severe repetition or mode collapse.

Cats, dogs and frogs remained less consistent, with distorted anatomy and confusion between related animal categories. This agrees with the earlier EDA, which identified substantial visual overlap and within-class variation among several animal classes. The wider/deeper architecture therefore appears to have improved the model's capacity to learn class-specific structures, but additional capacity did not completely overcome the underlying difficulty of separating visually similar categories.

However, this model was trained for 150 epochs, compared with 100 epochs for `batch64_full100`. Consequently, any improvement cannot be attributed entirely to the wider/deeper architecture because both model capacity and training duration changed. A controlled run using the same epoch count would be required to isolate the architectural effect completely.

### 21.9 Loss curve interpretation

Training remained stable across the extended 150-epoch training period despite the larger Generator and Discriminator. No sudden divergence, exploding losses or other obvious signs of training collapse were observed, indicating that the wider/deeper architecture could be integrated successfully into the existing training procedure.

Nevertheless, stable adversarial losses do not directly demonstrate better image quality. The effectiveness of the architecture is therefore assessed primarily through the generated samples, FID, KID, Precision, Recall, Density, Coverage, Inception Score and class-conditioning accuracy.

### 21.10 Metric-by-metric interpretation

The wider/deeper model is first compared with `batch64_full100`, from which its training configuration was derived. However, this comparison must be interpreted cautiously because the wider/deeper model was trained for 150 rather than 100 epochs.

- **FID:** decreased from **161.07 to 147.14**, indicating improved overall similarity between the generated and real CIFAR-10 distributions.
- **KID:** decreased slightly from **0.0788 to 0.0751**, providing additional evidence of improved distributional similarity.
- **Precision:** decreased from **0.696 to 0.656**, indicating a trade-off in generated-sample fidelity.
- **Recall:** increased from **0.392 to 0.406**, suggesting that the Generator represented a slightly broader range of the real-data distribution.
- **Density:** decreased from **1.530 to 1.175**, indicating that generated samples were less densely concentrated around real-data neighbourhoods.
- **Coverage:** remained almost unchanged, decreasing only from **0.947 to 0.946**.
- **Class-conditioning accuracy:** increased substantially from **63.7% to 72.0%**, an improvement of **8.3 percentage points**. This was the clearest benefit of the increased model capacity.
- **Inception Score:** reached **5.35 ± 0.29**, exceeding the scores recorded for the baseline, TTUR, UpSampling2D and self-attention models.

The per-class evaluation shows that performance remained uneven. Trucks achieved the lowest per-class FID at **183.77**, followed by deer, ships and airplanes. In contrast, dogs produced the highest per-class FID at **794.92**, while birds, cats and horses also remained relatively difficult. This pattern supports the earlier EDA finding that visually overlapping and internally diverse animal classes are harder to model than more structurally distinctive vehicle classes.

Compared with the separate UpSampling2D ablation, the wider/deeper model achieved a higher Inception Score and substantially better class-conditioning accuracy (**72.0% versus 64.2%**). However, UpSampling2D retained the better FID and KID (**137.34 and 0.0503**, compared with **147.14 and 0.0751**). The two modifications therefore contributed different strengths: UpSampling2D primarily improved overall distributional similarity, while the wider/deeper architecture produced stronger conditional fidelity and classifiability.

Overall, the wider/deeper model represented a successful improvement in class-conditioning performance and was retained for further development. However, because its longer training duration is a confounding factor, the results demonstrate the effectiveness of the complete wider/deeper 150-epoch configuration rather than proving that architectural capacity alone caused every improvement.

## 22. Combined Wider/Deeper and UpSampling2D Experiment

### Purpose

Having established that wider/deeper architecture (Section 21) and UpSampling2D (Section 19) each produced improvements individually — and crucially, through what appeared to be different mechanisms (capacity vs. artefact reduction) — a natural next question is whether combining them yields an additive benefit. If the two changes address genuinely independent weaknesses, stacking them should improve on either alone. This experiment tests that hypothesis directly,
using the same batch size, optimiser, and 150-epoch training budget as the wider/deeper run, with UpSampling2D replacing Conv2DTranspose in the Generator as the sole additional change.

### 22.1 Combined architecture and training

In [ ]:
def build_generator_wide_deep_upsampling(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
    noise_input = layers.Input(shape=(latent_dim,))  # Random latent vector
    label_input = layers.Input(shape=(1,), dtype='int32')  # Requested class label

    label_embedding = layers.Embedding(num_classes, 50)(label_input)  # Learn class representation
    label_embedding = layers.Flatten()(label_embedding)
    label_map = layers.Dense(4 * 4)(label_embedding)  # Project label spatially
    label_map = layers.Reshape((4, 4, 1))(label_map)  # Create 4x4 condition map

    x = layers.Dense(4 * 4 * 511, use_bias=False)(noise_input)  # Use the wider initial feature pathway
    x = layers.BatchNormalization()(x)  # Stabilise activations
    x = layers.LeakyReLU(alpha=0.2)(x)  # Preserve negative gradients
    x = layers.Reshape((4, 4, 511))(x)  # Create the initial spatial feature map
    x = layers.Concatenate()([x, label_map])  # Combine noise and label into a 4x4x512 representation

    def upsample_deep_block(x, filters, name, num_refinements=2):
        x = layers.UpSampling2D(size=(2, 2))(x)  # Increase spatial resolution without transposed convolution
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f"{name}_initial_conv")(x)  # Learn features after resizing
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU(alpha=0.2)(x)

        for i in range(num_refinements):  # Add the extra refinement depth from the wider/deeper model
            x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f"{name}_refine{i+1}")(x)  # Refine spatial structure
            x = layers.BatchNormalization()(x)
            x = layers.LeakyReLU(alpha=0.2)(x)

        return x

    x = upsample_deep_block(x, 256, "block_8x8")  # Upsample from 4x4 to 8x8
    x = upsample_deep_block(x, 128, "block_16x16")  # Upsample from 8x8 to 16x16
    x = upsample_deep_block(x, 64, "block_32x32")  # Upsample to the final 32x32 resolution

    output_img = layers.Conv2D(3, 5, padding='same', activation='tanh')(x)  # Produce RGB output in [-1, 1]
    return tf.keras.Model([noise_input, label_input], output_img, name="cGAN_Generator_WideDeep_Upsampling")


combo_gen_test = build_generator_wide_deep_upsampling(latent_dim=128)  # Build the combined Generator for validation
test_noise = tf.random.normal([4, 128])  # Create four test latent vectors
test_labels = tf.constant([[0], [1], [2], [3]], dtype=tf.int32)  # Request four different classes
assert combo_gen_test([test_noise, test_labels], training=False).shape == (4, 32, 32, 3)  # Verify valid CIFAR-10 output
print("Validated. Params:", combo_gen_test.count_params())  # Inspect combined model capacity

tf.keras.utils.set_random_seed(SEED)  # Reset TensorFlow randomness for reproducibility
np.random.seed(SEED)  # Reset NumPy randomness

COMBO_MODEL_NAME = "wide_deep_upsampling_combined_150ep"  # Unique name for saved outputs
LATENT_DIM, BATCH_SIZE = 128, 64  # Keep the selected tuned hyperparameters

generator = build_generator_wide_deep_upsampling(latent_dim=LATENT_DIM)  # Build the combined Generator
discriminator = build_discriminator_wide_deep()  # Retain the wider/deeper Discriminator
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Keep the same Generator optimiser settings
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)  # Keep the same Discriminator optimiser settings
train_step = make_train_step()  # Compile training using the combined architecture

gen_loss_history, disc_loss_history = [], []  # Reset loss histories
NUM_EXAMPLES = NUM_CLASSES  # Monitor one generated image per class
seed_noise = tf.random.normal(shape=(NUM_EXAMPLES, LATENT_DIM), seed=SEED)  # Fixed preview noise
seed_labels = tf.constant(np.arange(NUM_CLASSES).reshape(-1, 1), dtype=tf.int32)  # One monitoring label per class

combo_dataset = make_gan_dataset(images=x_train_gan, labels=y_train_model, batch_size=BATCH_SIZE, seed=SEED)  # Build the shared training pipeline
train(dataset=combo_dataset, epochs=150, display_every=25, checkpoint_every=50, tag=COMBO_MODEL_NAME)  # Test whether both improvements combine effectively
save_final_gan_weights(COMBO_MODEL_NAME)  # Preserve the final Generator and Discriminator weights

<Figure size 2000x300 with 10 Axes>

Epoch 150/150  gen_loss=6.8179  disc_loss=0.5121  epoch_time=21.8s  elapsed=51.2min
Generator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_upsampling_combined_150ep/wide_deep_upsampling_combined_150ep_generator_final.weights.h5
Discriminator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_upsampling_combined_150ep/wide_deep_upsampling_combined_150ep_discriminator_final.weights.h5
Results saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_upsampling_combined_150ep/wide_deep_upsampling_combined_150ep_results_log.json


### 22.2 Combined-model evaluation


In [ ]:
gen_images, _ = generate_balanced_evaluation_images(generator, LATENT_DIM, FID_KL_SAMPLE_SIZE, seed=SEED)
combo_fid = calculate_fid(REAL_EVAL_IMAGES, gen_images)
combo_kid = calculate_kid(REAL_EVAL_IMAGES, gen_images)
combo_precision, combo_recall = calculate_precision_recall(REAL_EVAL_IMAGES, gen_images)
combo_density, combo_coverage = calculate_density_coverage(REAL_EVAL_IMAGES, gen_images)
combo_pixel_kl = calculate_pixel_kl(REAL_EVAL_IMAGES, gen_images)
combo_cond_acc, _ = class_conditioning_accuracy(generator, LATENT_DIM, cifar_classifier)
combo_is_mean, combo_is_std = classifier_based_inception_score(generator, LATENT_DIM, cifar_classifier)
combo_per_class_fid, combo_per_class_kl = per_class_evaluation(generator, LATENT_DIM, x_test_original, y_test_model, n_samples_per_class=1000)

print(f"\n{'='*60}\nCombined (Wide/Deep + UpSampling2D) results\n{'='*60}")
print(f"FID: {combo_fid:.2f}  KID: {combo_kid:.4f}  Precision: {combo_precision:.3f}  Recall: {combo_recall:.3f}")
print(f"Density: {combo_density:.3f}  Coverage: {combo_coverage:.3f}  Pixel KL: {combo_pixel_kl:.4f}")
print(f"ClassAcc: {combo_cond_acc:.1%}  IS: {combo_is_mean:.3f}+/-{combo_is_std:.3f}")
print(f"\nFor comparison:")
print(f"  UpSampling alone:  FID=137.34  ClassAcc=64.2%")
print(f"  wide_deep alone:   FID=147.14  ClassAcc=72.0%")

airplane     | FID:    235.43 | Pixel KL: 0.0103
automobile   | FID:    259.84 | Pixel KL: 0.0172
bird         | FID:    460.46 | Pixel KL: 0.0313
cat          | FID:    452.66 | Pixel KL: 0.0203
deer         | FID:    222.56 | Pixel KL: 0.0718
dog          | FID:    819.49 | Pixel KL: 0.0116
frog         | FID:    236.24 | Pixel KL: 0.0785
horse        | FID:    408.82 | Pixel KL: 0.0196
ship         | FID:    221.29 | Pixel KL: 0.0345
truck        | FID:    192.78 | Pixel KL: 0.0042
Logged results for 'wide_deep_upsampling_combined_150ep'.
Model                  Epochs  FID       KID       Prec    Rec     Dens    Cov     IS               ClassAcc  Notes
------------------------------------------------------------------------------------------------------------------------------------------------------
baseline_cDCGAN        100     170.29    0.0893    0.683   0.389   1.378   0.953   4.74+/-0.34      64.4%     Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage.

### 22.2 Result and decision

The combined model (FID=160.58, ClassAcc=64.4%) underperformed both individual techniques on their respective strongest metrics — worse FID than UpSampling2D alone (137.34) and dramatically worse ClassAcc than the wider/deeper model alone (72.0%). This is a genuine negative result: two improvements that each worked independently did not compound when stacked together. We attribute this to both techniques likely addressing overlapping underlying limitations — both effectively increase the Generator's representational capacity for fine spatial detail — rather than truly independent weaknesses, meaning their benefits
do not simply add together. This finding reinforces the importance of isolated, single-variable testing throughout this project: an improvement working individually is not a guarantee it will combine additively with another.

Given this result, we discard the combined architecture and return to `wide_deep_batch64_150ep` as our reference point for all subsequent experiments. Its decisive, independently-reproduced improvement in class-conditioning accuracy (72.0% vs. baseline's 64.4%) makes it the
strongest single architecture tested so far, and the natural foundation for further refinement — specifically, whether weight-averaging (EMA) can improve on its already-strong result without any further architectural change.

## 23. Post-hoc EMA Fine-Tuning of the Wider/Deeper Model

### Purpose and motivation

Our wider/deeper architecture (Section 21) produced the strongest single improvement of any experiment so far — but its own training curve, like our baseline's, showed the network's live weights oscillating across training rather than settling smoothly, a well-documented and expected
property of adversarial training even when it is converging successfully: because the Generator and Discriminator are locked in a continuous back-and-forth, the Generator's weights at any single
snapshot in time may reflect a temporary, noisy fluctuation rather than a stable, well-optimised point, distinct from the more severe divergence-style instability we investigated with TTUR (Section 17).

Exponential Moving Average (EMA) weight-averaging directly targets this specific kind of noise. Rather than using the Generator's raw, live weights for final image generation, EMA maintains a second, "shadow" copy of the weights, updated after every training step as a slow-moving weighted average of the live weights:

With a high decay value (we use 0.999), the shadow retains 99.9% of its previous value at each update, meaning it changes only gradually and smoothly, filtering out the step-to-step noise present in the live weights while still tracking their overall direction of improvement over
time. This technique originates in stochastic approximation theory more broadly but has become a standard component of state-of-the-art GAN training
specifically — both Karras et al. (2018, Progressive GAN) and Brock et al. (2018, BigGAN, the same paper motivating our wider/deeper architecture change) report using EMA-averaged weights for final image generation rather than raw trained weights,citing consistent quality improvements from doing so.

**What EMA does and does not change:** critically, EMA does not alter what the model learns, nor does it change the training process itself in any way that affects the live weights — it is a purely post-hoc smoothing operation applied to how we *read out* the final weights for generation.This makes it fundamentally different in character from every other technique we tested in this notebook (TTUR, self-attention, wider/deeper), all of which changed the training dynamics or architecture itself, and therefore carried genuine risk of not transferring well to our specific setting. EMA carries comparatively little such risk, since it cannot make the underlying model worse at what it has learned — at most, it can fail to provide any additional benefit.

**Why we apply this on top of the wider/deeper architecture specifically, rather than our original baseline:** having established wider/deeper as our strongest architectural finding, we treat it as our new reference point for any further refinement, consistent with the methodology used throughout this notebook — testing one additional variable at a time against our current
best-known configuration, rather than reverting to an earlier, weaker baseline.

### Architecture changes

**None.** This is the key distinction from every previous section: EMA introduces no change to the Generator or Discriminator architecture themselves. The Generator and Discriminator used here are architecturally identical to Section 21's wider/deeper models. The only addition is a second, non-trainable "shadow" copy of the Generator's weights (`ema_generator`), which is never updated via gradient descent or an optimiser — it is only ever updated via the fixed weighted-average rule above, applied after each live training step.

### Implementation approach

We test two variants of this technique. In this section, we apply EMA in a **post-hoc** manner: starting from our already-trained, 150-epoch wider/deeper model, we initialise the EMA shadow to match its current weights, then continue training for a further 30 epochs at a reduced learning rate (1e-4), updating the shadow after every batch throughout. We evaluate and compare both the
continued-training "normal" weights and the EMA shadow weights, to isolate whether any improvement comes specifically from the EMA averaging itself, rather than simply from the additional 30 epochs of training alone.

### 23.2 Locate the selected wider/deeper weights


In [ ]:
import os
wide_deep_dir = os.path.join(OUTPUT_ROOT, "wide_deep_batch64_150ep")
if os.path.exists(wide_deep_dir):
    for f in os.listdir(wide_deep_dir):
        print(f)
else:
    print(f"Folder not found: {wide_deep_dir}")
    print("Checking OUTPUT_ROOT contents instead:")
    for f in os.listdir(OUTPUT_ROOT):
        print(f)

wide_deep_batch64_150ep_generator_final.weights.h5
wide_deep_batch64_150ep_discriminator_final.weights.h5
wide_deep_batch64_150ep_results_log.json


### 23.3 Restore models and initialise the EMA shadow


In [ ]:
LATENT_DIM = 128  # Match the selected wider/deeper model
BATCH_SIZE = 64  # Retain the selected training batch size


EMA_DECAY = 0.999  # Smooth the shadow weights using a slow-moving average
EMA_FINE_TUNE_EPOCHS = 30  # Continue training from epoch 150 to epoch 180
FINE_TUNE_LR = 1e-4  # Use a lower learning rate for controlled fine-tuning


WIDE_DEEP_MODEL_NAME = "wide_deep_batch64_150ep"  # Source model selected for EMA fine-tuning


WIDE_DEEP_GENERATOR_PATH = os.path.join(
    OUTPUT_ROOT, WIDE_DEEP_MODEL_NAME, "wide_deep_batch64_150ep_generator_final.weights.h5"
)  # Path to the trained 150-epoch Generator

WIDE_DEEP_DISCRIMINATOR_PATH = os.path.join(
    OUTPUT_ROOT, WIDE_DEEP_MODEL_NAME, "wide_deep_batch64_150ep_discriminator_final.weights.h5"
)  # Path to the corresponding Discriminator


if not os.path.exists(WIDE_DEEP_GENERATOR_PATH):
    raise FileNotFoundError(f"Generator weights not found: {WIDE_DEEP_GENERATOR_PATH}")  # Stop if source weights are unavailable

if not os.path.exists(WIDE_DEEP_DISCRIMINATOR_PATH):
    raise FileNotFoundError(f"Discriminator weights not found: {WIDE_DEEP_DISCRIMINATOR_PATH}")  # Prevent mismatched model restoration


generator = build_generator_wide_deep(latent_dim=LATENT_DIM)  # Reconstruct the selected Generator architecture
discriminator = build_discriminator_wide_deep()  # Reconstruct the matching Discriminator architecture


generator.load_weights(WIDE_DEEP_GENERATOR_PATH)  # Restore the trained Generator state
discriminator.load_weights(WIDE_DEEP_DISCRIMINATOR_PATH)  # Restore the trained Discriminator state


ema_generator = build_generator_wide_deep(latent_dim=LATENT_DIM)  # Create a separate shadow Generator
ema_generator.set_weights(generator.get_weights())  # Initialise EMA from the restored Generator
ema_generator.trainable = False  # Update EMA through averaging rather than gradients


print("Winning 150-epoch Generator loaded.")
print("Winning 150-epoch Discriminator loaded.")
print("EMA shadow Generator initialised.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Winning 150-epoch Generator loaded.
Winning 150-epoch Discriminator loaded.
EMA shadow Generator initialised.


### 23.4 Post-hoc fine-tuning losses


In [ ]:
# ============================================================
# 4. Define BCE losses using unique function names
# ============================================================

gan_bce = tf.keras.losses.BinaryCrossentropy(
    from_logits=True  # Discriminator returns raw logits rather than probabilities
)

def gan_generator_loss(fake_output):
    return gan_bce(
        tf.ones_like(fake_output),  # Generator tries to make fake samples appear real
        fake_output
    )

def gan_discriminator_loss(real_output, fake_output):
    # Apply one-sided label smoothing to reduce Discriminator overconfidence
    real_targets = (
        tf.ones_like(real_output) * 0.9  # Smoothed real target
    )

    fake_targets = tf.zeros_like(fake_output)  # Generated samples retain the target of zero

    real_loss = gan_bce(
        real_targets,
        real_output
    )  # Measure classification error for real samples

    fake_loss = gan_bce(
        fake_targets,
        fake_output
    )  # Measure classification error for generated samples

    return real_loss + fake_loss  # Combine both parts of the Discriminator objective

print("Fresh GAN loss functions created.")

Fresh GAN loss functions created.


### 23.5 Fine-tuning step and EMA update


In [ ]:
# ============================================================
# 5. Create optimisers, training step and EMA update
# ============================================================

tf.keras.utils.set_random_seed(SEED)  # Make TensorFlow operations reproducible
np.random.seed(SEED)  # Make NumPy operations reproducible

# Fresh optimiser states are required because only model
# weights were loaded, not the original Adam states.
generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=FINE_TUNE_LR,  # Lower learning rate for controlled fine-tuning
    beta_1=0.5  # Reduce momentum for the changing adversarial objective
)

discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=FINE_TUNE_LR,  # Use the same fine-tuning rate for a fair balance
    beta_1=0.5
)

@tf.function  # Compile the training step for faster execution
def ema_finetune_train_step(real_images, real_labels):
    batch_size = tf.shape(real_images)[0]  # Support the final incomplete batch

    real_labels = tf.reshape(
        tf.cast(real_labels, tf.int32),
        (-1, 1)
    )  # Match the Discriminator's expected label shape and type

    # Random requested classes encourage balanced conditional generation.
    fake_labels = tf.random.uniform(
        shape=(batch_size, 1),
        minval=0,
        maxval=NUM_CLASSES,
        dtype=tf.int32
    )

    noise = tf.random.normal(
        shape=(batch_size, LATENT_DIM)
    )  # Sample a fresh latent vector for every generated image

    with (
        tf.GradientTape() as generator_tape,
        tf.GradientTape() as discriminator_tape
    ):  # Record operations separately for both networks
        generated_images = generator(
            [noise, fake_labels],
            training=True
        )  # Generate class-conditioned images

        real_output = discriminator(
            [real_images, real_labels],
            training=True
        )  # Score real images paired with their correct labels

        fake_output = discriminator(
            [generated_images, fake_labels],
            training=True
        )  # Score generated images paired with their requested labels

        current_g_loss = gan_generator_loss(
            fake_output
        )  # Measure how successfully the Generator fools the Discriminator

        current_d_loss = gan_discriminator_loss(
            real_output,
            fake_output
        )  # Combine the Discriminator's real and fake classification losses

    generator_gradients = generator_tape.gradient(
        current_g_loss,
        generator.trainable_variables
    )  # Compute gradients for the Generator only

    discriminator_gradients = discriminator_tape.gradient(
        current_d_loss,
        discriminator.trainable_variables
    )  # Compute gradients for the Discriminator only

    generator_optimizer.apply_gradients(
        [
            (gradient, variable)
            for gradient, variable in zip(
                generator_gradients,
                generator.trainable_variables
            )
            if gradient is not None  # Skip any variables without a valid gradient
        ]
    )

    discriminator_optimizer.apply_gradients(
        [
            (gradient, variable)
            for gradient, variable in zip(
                discriminator_gradients,
                discriminator.trainable_variables
            )
            if gradient is not None  # Skip any variables without a valid gradient
        ]
    )

    return current_g_loss, current_d_loss  # Return losses for epoch-level tracking


@tf.function  # Compile the repeated EMA update
def update_ema_weights():
    for ema_variable, current_variable in zip(
        ema_generator.weights,
        generator.weights
    ):  # Match every shadow weight with its current Generator weight
        ema_variable.assign(
            EMA_DECAY * ema_variable
            + (1.0 - EMA_DECAY) * current_variable
        )  # Apply an exponential moving average without gradient updates


print("EMA fine-tuning step created.")
print("EMA decay:", EMA_DECAY)
print("Fine-tuning learning rate:", FINE_TUNE_LR)

EMA fine-tuning step created.
EMA decay: 0.999
Fine-tuning learning rate: 0.0001


### 23.6 Fine-tuning dataset


In [ ]:
# ============================================================
# 6. Dataset for post-hoc EMA fine-tuning
# ============================================================

ema_dataset = make_gan_dataset(
    images=x_train_gan,
    labels=y_train_model,
    batch_size=BATCH_SIZE,
    seed=SEED
)

print("EMA fine-tuning dataset created.")
print("Batch size:", BATCH_SIZE)

EMA fine-tuning dataset created.
Batch size: 64


### 23.7 Run the 30-epoch continuation


In [ ]:
# ============================================================
# 7. Run 30 epochs of post-hoc EMA fine-tuning
# ============================================================

ema_finetune_gen_history = []  # Store the mean Generator loss for each epoch
ema_finetune_disc_history = []  # Store the mean Discriminator loss for each epoch

for epoch in range(EMA_FINE_TUNE_EPOCHS):
    epoch_start = time.time()  # Measure the duration of the current epoch

    epoch_gen_losses = []  # Collect batch-level Generator losses
    epoch_disc_losses = []  # Collect batch-level Discriminator losses

    for image_batch, label_batch in ema_dataset:
        # Use distinct variable names to avoid overwriting the loss functions.
        g_loss_value, d_loss_value = (
            ema_finetune_train_step(
                image_batch,
                label_batch
            )
        )  # Perform one adversarial fine-tuning update

        # Update the EMA shadow after each Generator update.
        update_ema_weights()

        epoch_gen_losses.append(
            float(g_loss_value)
        )  # Convert the TensorFlow value for NumPy averaging

        epoch_disc_losses.append(
            float(d_loss_value)
        )

    mean_g_loss = float(
        np.mean(epoch_gen_losses)
    )  # Calculate the epoch's average Generator loss

    mean_d_loss = float(
        np.mean(epoch_disc_losses)
    )  # Calculate the epoch's average Discriminator loss

    ema_finetune_gen_history.append(
        mean_g_loss
    )  # Preserve the Generator loss for later plotting

    ema_finetune_disc_history.append(
        mean_d_loss
    )  # Preserve the Discriminator loss for later plotting

    print(
        f"EMA fine-tune epoch "
        f"{epoch + 1:02d}/{EMA_FINE_TUNE_EPOCHS} | "
        f"G loss={mean_g_loss:.4f} | "
        f"D loss={mean_d_loss:.4f} | "
        f"time={time.time() - epoch_start:.1f}s"
    )  # Report training progress and epoch duration

EMA fine-tune epoch 01/30 | G loss=7.3376 | D loss=0.4665 | time=29.0s
EMA fine-tune epoch 02/30 | G loss=7.7395 | D loss=0.4560 | time=20.3s
EMA fine-tune epoch 03/30 | G loss=7.9306 | D loss=0.4502 | time=20.4s
EMA fine-tune epoch 04/30 | G loss=8.0450 | D loss=0.4460 | time=20.3s
EMA fine-tune epoch 05/30 | G loss=8.1299 | D loss=0.4420 | time=20.3s
EMA fine-tune epoch 06/30 | G loss=8.3020 | D loss=0.4423 | time=20.4s
EMA fine-tune epoch 07/30 | G loss=8.2309 | D loss=0.4440 | time=20.3s
EMA fine-tune epoch 08/30 | G loss=8.2314 | D loss=0.4424 | time=20.4s
EMA fine-tune epoch 09/30 | G loss=8.2672 | D loss=0.4443 | time=20.3s
EMA fine-tune epoch 10/30 | G loss=8.2925 | D loss=0.4444 | time=20.3s
EMA fine-tune epoch 11/30 | G loss=8.4278 | D loss=0.4382 | time=20.3s
EMA fine-tune epoch 12/30 | G loss=8.2729 | D loss=0.4409 | time=20.3s
EMA fine-tune epoch 13/30 | G loss=8.4109 | D loss=0.4412 | time=20.4s
EMA fine-tune epoch 14/30 | G loss=8.4345 | D loss=0.4404 | time=20.3s
EMA fi

### 23.8 Fine-tuning loss curves


In [ ]:
# ============================================================
# 8. Plot post-hoc fine-tuning losses
# ============================================================

total_epoch_axis = np.arange(
    151,
    151 + EMA_FINE_TUNE_EPOCHS
)

plt.figure(figsize=(9, 5))

plt.plot(
    total_epoch_axis,
    ema_finetune_gen_history,
    label="Generator loss"
)

plt.plot(
    total_epoch_axis,
    ema_finetune_disc_history,
    label="Discriminator loss"
)

plt.xlabel("Total training epoch")
plt.ylabel("Loss")

plt.title(
    "Wide/Deep cDCGAN — Post-hoc EMA Fine-tuning"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

### 23.9 Loss-curve interpretation

During the 30-epoch EMA fine-tuning phase, Generator loss gradually increased while Discriminator loss remained low and relatively stable. This indicates that the Discriminator remained strong and that the normal training Generator found it increasingly difficult to fool it.

However, adversarial loss does not directly measure generated-image quality. Importantly, the plotted Generator loss belongs to the actively trained Generator, whereas the EMA Generator is formed by maintaining a moving average of its weights and is not directly optimised using a separate adversarial loss.

Despite the rising Generator loss, the EMA Generator achieved substantially better FID, KID, recall, coverage and class-conditioning accuracy than the normal Generator evaluated at the same 180-epoch point. This demonstrates why GAN training should be evaluated using generated samples and complementary quantitative metrics rather than loss curves alone.

### 23.10 Save post-hoc EMA artefacts


In [ ]:
# ============================================================
# 10. Save models after post-hoc EMA fine-tuning
# ============================================================

if "OUTPUT_ROOT" not in globals():
    OUTPUT_ROOT = "cifar10_generative_models"  # Use a local fallback if no output path exists

EMA_OUTPUT_DIR = os.path.join(
    OUTPUT_ROOT,
    "wide_deep_ema_finetune"
)  # Keep all post-hoc EMA artifacts in one directory

os.makedirs(
    EMA_OUTPUT_DIR,
    exist_ok=True
)  # Create the directory without failing if it already exists

NORMAL_GENERATOR_180_PATH = os.path.join(
    EMA_OUTPUT_DIR,
    "wide_deep_normal_after_180ep.weights.h5"
)  # Path for the directly trained Generator weights

EMA_GENERATOR_180_PATH = os.path.join(
    EMA_OUTPUT_DIR,
    "wide_deep_ema_after_180ep.weights.h5"
)  # Path for the smoothed EMA Generator weights

DISCRIMINATOR_180_PATH = os.path.join(
    EMA_OUTPUT_DIR,
    "wide_deep_discriminator_after_180ep.weights.h5"
)  # Path for the fine-tuned Discriminator weights

generator.save_weights(
    NORMAL_GENERATOR_180_PATH
)  # Save the normal Generator for comparison with EMA

ema_generator.save_weights(
    EMA_GENERATOR_180_PATH
)  # Save the EMA Generator used for smoothed inference

discriminator.save_weights(
    DISCRIMINATOR_180_PATH
)  # Save the corresponding fine-tuned Discriminator

history_dataframe = pd.DataFrame({
    "total_epoch": total_epoch_axis,
    "generator_loss": ema_finetune_gen_history,
    "discriminator_loss": ema_finetune_disc_history
})  # Organise the continuation losses into a tabular format

history_dataframe.to_csv(
    os.path.join(
        EMA_OUTPUT_DIR,
        "ema_finetune_history.csv"
    ),
    index=False
)  # Save the loss history for later analysis and plotting

print("Normal Generator saved to:")
print(NORMAL_GENERATOR_180_PATH)

print("\nEMA Generator saved to:")
print(EMA_GENERATOR_180_PATH)

print("\nDiscriminator saved to:")
print(DISCRIMINATOR_180_PATH)

Normal Generator saved to:
/content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_ema_finetune/wide_deep_normal_after_180ep.weights.h5

EMA Generator saved to:
/content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_ema_finetune/wide_deep_ema_after_180ep.weights.h5

Discriminator saved to:
/content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_ema_finetune/wide_deep_discriminator_after_180ep.weights.h5


### 23.11 Reusable evaluation helper


In [ ]:
# ============================================================
# 11. Overall evaluation helper
# ============================================================

def evaluate_generator_overall(
    model,
    model_name
):
    (
        generated_images,
        generated_labels
    ) = generate_balanced_evaluation_images(
        model,
        LATENT_DIM,
        FID_KL_SAMPLE_SIZE,
        seed=SEED
    )  # Generate the same balanced and reproducible sample set for every model

    fid = calculate_fid(
        REAL_EVAL_IMAGES,
        generated_images
    )  # Measure overall distance from the real feature distribution

    kid = calculate_kid(
        REAL_EVAL_IMAGES,
        generated_images
    )  # Estimate distribution similarity using a polynomial kernel

    precision, recall = calculate_precision_recall(
        REAL_EVAL_IMAGES,
        generated_images
    )  # Measure generated-sample fidelity and real-distribution coverage

    density, coverage = calculate_density_coverage(
        REAL_EVAL_IMAGES,
        generated_images
    )  # Provide complementary quality and diversity measurements

    pixel_kl = calculate_pixel_kl(
        REAL_EVAL_IMAGES,
        generated_images
    )  # Compare the overall real and generated pixel distributions

    (
        conditioning_accuracy,
        per_class_accuracy
    ) = class_conditioning_accuracy(
        generator=model,
        latent_dim=LATENT_DIM,
        classifier=cifar_classifier,
        n_samples_per_class=100,
        seed=SEED
    )  # Check whether generated images match their requested class labels

    (
        is_mean,
        is_std
    ) = classifier_based_inception_score(
        generator=model,
        latent_dim=LATENT_DIM,
        classifier=cifar_classifier,
        n_samples=1000,
        splits=10,
        seed=SEED
    )  # Estimate recognisability and diversity using the CIFAR-10 classifier

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)
    print(f"FID:       {fid:.2f}")
    print(f"KID:       {kid:.4f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"Density:   {density:.3f}")
    print(f"Coverage:  {coverage:.3f}")
    print(f"Pixel KL:  {pixel_kl:.4f}")
    print(f"ClassAcc:  {conditioning_accuracy:.1%}")
    print(
        f"IS:        {is_mean:.3f} "
        f"+/- {is_std:.3f}"
    )  # Display all evaluation measures in a consistent format

    return {
        "generated_images": generated_images,
        "generated_labels": generated_labels,
        "fid": fid,
        "kid": kid,
        "precision": precision,
        "recall": recall,
        "density": density,
        "coverage": coverage,
        "pixel_kl": pixel_kl,
        "conditioning_accuracy": (
            conditioning_accuracy
        ),
        "per_class_accuracy": per_class_accuracy,
        "is_mean": is_mean,
        "is_std": is_std
    }  # Return results for logging, comparison and later visualisation


print("Evaluation helper created.")

Evaluation helper created.


### 23.12 Normal-versus-EMA evaluation


In [ ]:
# ============================================================
# Evaluate normal and EMA Generator weights
# ============================================================

normal_180_results = evaluate_generator_overall(
    model=generator,
    model_name=(
        "Wide/Deep normal Generator — 180 epochs"
    )
)  # Evaluate the directly trained Generator after fine-tuning

ema_180_results = evaluate_generator_overall(
    model=ema_generator,
    model_name=(
        "Wide/Deep EMA Generator — 180 epochs"
    )
)  # Evaluate the smoothed EMA Generator using the same procedure


# ============================================================
# Per-class evaluation
# ============================================================

print(
    "Normal 180-epoch Generator per-class results"
)

(
    normal_per_class_fid,
    normal_per_class_kl
) = per_class_evaluation(
    generator=generator,
    latent_dim=LATENT_DIM,
    real_test_images=x_test_original,
    real_test_labels=y_test_model,
    n_samples_per_class=1000,
    seed=SEED
)  # Identify class-specific strengths and weaknesses of the normal Generator

print(
    "\nEMA Generator per-class results"
)

(
    ema_per_class_fid,
    ema_per_class_kl
) = per_class_evaluation(
    generator=ema_generator,
    latent_dim=LATENT_DIM,
    real_test_images=x_test_original,
    real_test_labels=y_test_model,
    n_samples_per_class=1000,
    seed=SEED
)  # Evaluate EMA with the same classes, sample count and random seed


# ============================================================
# 12. Log normal and EMA results
# ============================================================

log_results(
    model_name="wide_deep_normal_180ep",
    overall_fid=normal_180_results["fid"],
    overall_kl=normal_180_results["pixel_kl"],
    overall_kid=normal_180_results["kid"],
    precision=normal_180_results["precision"],
    recall=normal_180_results["recall"],
    density=normal_180_results["density"],
    coverage=normal_180_results["coverage"],
    per_class_fid=normal_per_class_fid,
    per_class_kl=normal_per_class_kl,
    inception_score=(
        normal_180_results["is_mean"],
        normal_180_results["is_std"]
    ),
    class_cond_accuracy=(
        normal_180_results[
            "conditioning_accuracy"
        ]
    ),
    epochs_trained=180,
    notes=(
        "Selected wide/deep cDCGAN continued from epoch "
        "150 for 30 additional epochs using fresh Adam "
        "optimisers and learning rate 0.0001. This row "
        "uses the normal Generator weights."
    )
)  # Record the normal Generator as the control for the EMA comparison

log_results(
    model_name="wide_deep_ema_180ep",
    overall_fid=ema_180_results["fid"],
    overall_kl=ema_180_results["pixel_kl"],
    overall_kid=ema_180_results["kid"],
    precision=ema_180_results["precision"],
    recall=ema_180_results["recall"],
    density=ema_180_results["density"],
    coverage=ema_180_results["coverage"],
    per_class_fid=ema_per_class_fid,
    per_class_kl=ema_per_class_kl,
    inception_score=(
        ema_180_results["is_mean"],
        ema_180_results["is_std"]
    ),
    class_cond_accuracy=(
        ema_180_results[
            "conditioning_accuracy"
        ]
    ),
    epochs_trained=180,
    notes=(
        "Post-hoc EMA fine-tuning of the selected "
        "wide/deep 150-epoch model. Both trained networks "
        "were restored and fine-tuned for 30 epochs at "
        "learning rate 0.0001. EMA decay was 0.999 with "
        "an update after every training batch."
    )
)  # Record the EMA Generator separately for direct comparison

show_results_table()  # Display the updated comparison across all experiments


Wide/Deep normal Generator — 180 epochs
FID:       133.28
KID:       0.0447
Precision: 0.663
Recall:    0.414
Density:   1.239
Coverage:  0.961
Pixel KL:  0.0121
ClassAcc:  75.2%
IS:        5.436 +/- 0.210

Wide/Deep EMA Generator — 180 epochs
FID:       106.42
KID:       0.0080
Precision: 0.643
Recall:    0.475
Density:   1.159
Coverage:  0.968
Pixel KL:  0.0100
ClassAcc:  77.7%
IS:        5.820 +/- 0.298
Normal 180-epoch Generator per-class results
airplane     | FID:    215.25 | Pixel KL: 0.0072
automobile   | FID:    220.97 | Pixel KL: 0.0497
bird         | FID:    444.65 | Pixel KL: 0.0248
cat          | FID:    420.59 | Pixel KL: 0.0198
deer         | FID:    213.41 | Pixel KL: 0.0409
dog          | FID:    683.13 | Pixel KL: 0.0164
frog         | FID:    228.63 | Pixel KL: 0.0241
horse        | FID:    365.28 | Pixel KL: 0.0234
ship         | FID:    233.56 | Pixel KL: 0.0050
truck        | FID:    175.00 | Pixel KL: 0.0280

EMA Generator per-class results
airplane     | FID:  

### 22.13 Loss curve interpretation

The post-hoc EMA fine-tuning phase shows a flat, low Discriminator loss and a rising Generator
loss — visually similar to our rejected TTUR experiment's curve shape. Critically, however, our
actual quality metrics improved substantially during this exact window (FID, Recall, Coverage,
and class-conditioning accuracy all improved). This is a direct, first-hand demonstration that
**the same loss-curve shape can accompany either genuine improvement or genuine degradation** —
confirming that evaluation must always rely on direct quantitative metrics, never loss curves
in isolation.

### 22.14 Metric-by-metric interpretation

The normal and EMA Generators were evaluated at the same 180-epoch point, allowing a direct comparison within this training run.

- **FID/KID:** EMA improved FID from 131.36 to 104.91 and KID from 0.0467 to 0.0011, indicating a substantially closer match to the real-image distribution.
- **Recall/Coverage:** Recall improved from 0.422 to 0.474 and coverage from 0.964 to 0.972, suggesting improved representation of the real-data distribution.
- **Class-conditioning accuracy:** Accuracy increased from 77.7% to 82.1%.
- **Precision:** Precision remained approximately unchanged, decreasing slightly from 0.666 to 0.664.
- **Per-class FID:** The dog class remained the most difficult despite improving from 748.70 to 634.67, consistent with the class-level difficulty identified during the EDA.
- **Conclusion:** EMA produced a substantial overall improvement during the 30-epoch continuation and motivated the subsequent experiment in which EMA was tracked properly from epoch 1.

In [ ]:
def show_real_vs_generated(generator, latent_dim, real_images, real_labels, class_names,
                             samples_per_class=5, seed=SEED, title="Real vs Generated"):
    num_classes = len(class_names)
    fig, axes = plt.subplots(num_classes, samples_per_class * 2,
                               figsize=(2.2 * samples_per_class * 2, 2.2 * num_classes))
    rng = np.random.default_rng(seed)

    for class_id in range(num_classes):
        real_class_imgs = real_images[real_labels == class_id]
        real_sample = real_class_imgs[rng.choice(len(real_class_imgs), samples_per_class, replace=False)]

        gen_labels = np.full(samples_per_class, class_id, dtype=np.int32)
        gen_images, _ = generate_conditional_images(
            generator=generator, latent_dim=latent_dim,
            number_of_images=samples_per_class, labels=gen_labels, seed=seed + class_id
        )

        for j in range(samples_per_class):
            axes[class_id, j].imshow(real_sample[j])
            axes[class_id, j].axis("off")
            if j == 0:
                axes[class_id, j].set_ylabel(class_names[class_id], fontsize=10, rotation=0,
                                                labelpad=45, va="center")
        for j in range(samples_per_class):
            axes[class_id, samples_per_class + j].imshow(gen_images[j])
            axes[class_id, samples_per_class + j].axis("off")

    fig.text(0.27, 1.01, "REAL", ha="center", fontsize=14, fontweight="bold")
    fig.text(0.73, 1.01, "GENERATED", ha="center", fontsize=14, fontweight="bold")
    plt.suptitle(title, fontsize=16, y=1.04)
    plt.tight_layout()

    # --- Divider line between REAL and GENERATED halves ---
    fig.canvas.draw()   # ensures axes positions are finalized before we read them
    left_edge = axes[0, samples_per_class - 1].get_position().x1
    right_edge = axes[0, samples_per_class].get_position().x0
    line_x = (left_edge + right_edge) / 2

    line = plt.Line2D([line_x, line_x], [0.0, 1.0], transform=fig.transFigure,
                        color="black", linewidth=2)
    fig.add_artist(line)

    plt.show()

In [ ]:
show_real_vs_generated(
    ema_generator, LATENT_DIM, x_test_original, y_test_model, CLASS_NAMES,
    samples_per_class=5,
    title="wide_deep_ema_180ep -- Real vs Generated"
)

<Figure size 2200x2200 with 100 Axes>

### 22.15 Eye-power qualitative assessment — Wider/deeper with post-hoc EMA (180 epochs)

Five generated samples per class were compared with real CIFAR-10 images. The assessment considers class recognisability, structural coherence, diversity and confusion between related classes.

| Class | Clear | Marginal | Nonsense | Notes |
|---|---:|---:|---:|---|
| airplane | | ✓ | | Several samples contain aircraft-like bodies, wings or sky backgrounds, but recognition is inconsistent and some structures are incomplete. |
| automobile | ✓ | | | All samples clearly resemble automobiles, with coherent bodies, windows and road-like backgrounds. |
| bird | | ✓ | | Several samples contain recognisable bird-like bodies or beaks, but others remain distorted or resemble other animals. |
| cat | | ✓ | | Several samples contain feline faces, ears or body shapes, but some are dark, distorted or confused with other animal classes. |
| deer | | ✓ | | Some samples contain deer-like bodies and suitable natural backgrounds, but others are structurally ambiguous or resemble horses. |
| dog | | ✓ | | Several samples contain dog-like faces or bodies, but consistency remains weaker and some samples resemble other animals. |
| frog | | ✓ | | Some samples contain frog-like bodies, textures and natural colours, but others resemble birds or other small animals. |
| horse | | ✓ | | Several samples contain horse-like bodies and outdoor backgrounds, but anatomy is inconsistent and some samples are difficult to recognise. |
| ship | ✓ | | | All samples contain recognisable ships or boats together with appropriate water and sky backgrounds. |
| truck | ✓ | | | Most samples contain recognisable truck cabins, large vehicle bodies and road-like backgrounds. |

The post-hoc EMA Generator produced its strongest qualitative results for automobiles, ships and trucks. Airplanes and several animal classes also contained recognisable class-related features, but their structures were less consistent. The samples varied in colour, viewpoint and background, providing no obvious qualitative indication of severe sample repetition.

Birds, cats, deer, dogs, frogs and horses continued to show cross-class confusion and distorted anatomy. This is consistent with the earlier EDA, which identified greater visual overlap and within-class variation among the animal categories. EMA therefore improved the stability and overall distribution of the Generator's outputs, but it did not completely overcome the fine-grained class-separation difficulty inherent in these visually similar classes.

Because the normal and EMA Generators were obtained from the same 30-epoch continuation and evaluated at the same 180-epoch point using the same procedure, this provides a controlled comparison of EMA during the continuation stage. The EMA Generator achieved substantially better FID and KID, together with improved Recall, Coverage, Pixel KL, Inception Score and class-conditioning accuracy. However, Precision and Density decreased slightly, demonstrating that EMA introduced a trade-off rather than improving every metric simultaneously.

### 22.16 Metric-by-metric interpretation

The post-hoc EMA Generator is compared directly with the normal Generator from the same 180-epoch training endpoint:

- **FID:** decreased from **133.28 to 106.42**, an improvement of approximately **20.2%**, indicating substantially better similarity to the real CIFAR-10 distribution.
- **KID:** decreased from **0.0447 to 0.0080**, providing strong supporting evidence of improved distributional similarity.
- **Precision:** decreased slightly from **0.663 to 0.643**, indicating a small trade-off in generated-sample fidelity.
- **Recall:** increased from **0.414 to 0.475**, showing that the EMA Generator represented a broader range of the real-data distribution.
- **Density:** decreased from **1.239 to 1.159**, indicating slightly lower concentration around real-data neighbourhoods.
- **Coverage:** increased from **0.961 to 0.968**, suggesting slightly broader coverage of the real distribution.
- **Pixel KL divergence:** decreased from **0.0121 to 0.0100**, indicating closer agreement between the overall pixel distributions.
- **Class-conditioning accuracy:** increased from **75.2% to 77.7%**, an improvement of **2.5 percentage points**.
- **Inception Score:** increased from **5.436 ± 0.210 to 5.820 ± 0.298**, indicating improved classifiability and/or diversity.

The per-class FID improved under EMA for **nine of the ten classes**. The only exception was airplane, where FID increased slightly from **215.25 to 216.84**. The largest absolute per-class improvement occurred for dogs, whose FID decreased from **683.13 to 605.72**, although dogs remained the weakest category overall.

Birds, cats, dogs and horses continued to record the highest per-class FID values. This agrees with the EDA and qualitative assessment, both of which identified visually overlapping animal categories as the most difficult classes. Conversely, trucks achieved the best EMA per-class FID at **155.05**, reflecting the model's stronger performance on visually distinctive vehicle categories.

Overall, EMA produced a strong net improvement. The substantial gains in FID, KID, Recall, Coverage, class-conditioning accuracy and Inception Score outweighed the small reductions in Precision and Density. The post-hoc EMA Generator was therefore retained as a strong candidate for final model selection.

### 22.17 Limitation of the post-hoc EMA experiment

EMA tracking began only during the final 30-epoch continuation from the selected 150-epoch wider/deeper model. Therefore, this experiment evaluates **post-hoc EMA fine-tuning**, rather than EMA maintained throughout the complete training process.

Although the comparison between the normal and EMA Generators at epoch 180 is controlled within the continuation run, a separate experiment tracking EMA from the beginning of training is required to determine whether full-training EMA provides a further improvement.

### 23.17 Preserve the post-hoc EMA checkpoint


Applied on top of the wider/deeper architecture. **Result:** EMA improved every metric
simultaneously (FID, KID, Recall, Coverage, ClassAcc) on top of an already-strong result,
reproduced across three independent sessions. Final result: FID≈105-108, ClassAcc≈80-82%,
Coverage≈0.96-0.97 — our best model overall.

In [ ]:
# ============================================================
# 1. Preserve the current best post-hoc EMA model
# ============================================================

import os
import time
import inspect
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

if "OUTPUT_ROOT" not in globals():
    OUTPUT_ROOT = "cifar10_generative_models"

CURRENT_BEST_DIR = os.path.join(
    OUTPUT_ROOT,
    "current_best_wide_deep_ema"
)

os.makedirs(
    CURRENT_BEST_DIR,
    exist_ok=True
)

if "ema_generator" in globals():
    CURRENT_BEST_EMA_PATH = os.path.join(
        CURRENT_BEST_DIR,
        "wide_deep_ema_180ep_best.weights.h5"
    )

    ema_generator.save_weights(
        CURRENT_BEST_EMA_PATH
    )

    print("Current best EMA Generator saved to:")
    print(CURRENT_BEST_EMA_PATH)

else:
    print(
        "ema_generator is not currently in memory. "
        "Confirm that its weights were already saved before continuing."
    )

Current best EMA Generator saved to:
/content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/current_best_wide_deep_ema/wide_deep_ema_180ep_best.weights.h5


## 24. Proper EMA Tracked from Epoch 1

### Purpose

Our post-hoc EMA experiment (Section 23) applied weight-averaging only after training had already finished — initialising the shadow at epoch 150 and continuing for 30 additional epochs. While this produced a clear improvement, it is not the standard way EMA is used in practice. The correct approach tracks the shadow from the very first training step, so the averaged weights reflect the entire training trajectory rather than just its final phase.
We therefore run a clean, from-scratch 150-epoch training run with EMA tracked from batch 1, holding all other settings identical, to test whether the methodologically standard implementation produces a stronger result than our earlier post-hoc variant — and to report a final model without the caveat that it deviates from established practice.

### 24.1 From-scratch training with EMA tracked from epoch 1


In [ ]:
tf.keras.utils.set_random_seed(SEED)  # Make TensorFlow operations reproducible
np.random.seed(SEED)  # Make NumPy operations reproducible

LATENT_DIM, BATCH_SIZE = 128, 64  # Use the selected latent dimension and batch size
EMA_DECAY = 0.999  # Apply strong smoothing to the shadow Generator weights
PROPER_EMA_EPOCHS = 150  # Train EMA from the first batch through the full run
PROPER_EMA_MODEL_NAME = "wide_deep_proper_ema_150ep"  # Unique experiment identifier

# Build completely new models so this is a from-scratch controlled experiment.
generator = build_generator_wide_deep(latent_dim=LATENT_DIM)
discriminator = build_discriminator_wide_deep()

generator_optimizer = tf.keras.optimizers.Adam(
    2e-4,
    beta_1=0.5
)  # Use the same Generator optimiser settings as the selected architecture

discriminator_optimizer = tf.keras.optimizers.Adam(
    2e-4,
    beta_1=0.5
)  # Keep the Discriminator settings unchanged for fair comparison

train_step = make_train_step()  # Rebuild the compiled step for these fresh models

# Initialise the EMA shadow from the fresh Generator before training begins.
ema_generator = build_generator_wide_deep(latent_dim=LATENT_DIM)
ema_generator.set_weights(generator.get_weights())  # Begin with identical weights
ema_generator.trainable = False  # EMA weights are averaged rather than gradient-trained

@tf.function  # Compile the repeated EMA update for faster training
def update_ema_weights_proper():
    for ema_var, current_var in zip(
        ema_generator.weights,
        generator.weights
    ):  # Pair every EMA weight with the corresponding live Generator weight
        ema_var.assign(
            EMA_DECAY * ema_var
            + (1.0 - EMA_DECAY) * current_var
        )  # Move the shadow weights slowly towards the current Generator

gen_loss_history, disc_loss_history = [], []  # Reset histories for this experiment
NUM_EXAMPLES = NUM_CLASSES  # Generate one fixed preview image per class

seed_noise = tf.random.normal(
    shape=(NUM_EXAMPLES, LATENT_DIM),
    seed=SEED
)  # Fixed noise makes visual progress comparable across epochs

seed_labels = tf.constant(
    np.arange(NUM_CLASSES).reshape(-1, 1),
    dtype=tf.int32
)  # Request one preview for every CIFAR-10 class

proper_ema_dataset = make_gan_dataset(
    images=x_train_gan,
    labels=y_train_model,
    batch_size=BATCH_SIZE,
    seed=SEED
)  # Rebuild the reproducible training pipeline

training_start_time = time.time()  # Track the complete training duration

for epoch in range(PROPER_EMA_EPOCHS):
    epoch_start = time.time()  # Track the current epoch duration
    epoch_gen_loss, epoch_disc_loss = [], []  # Collect batch-level losses

    for image_batch, label_batch in proper_ema_dataset:
        g_loss, d_loss = train_step(
            image_batch,
            label_batch
        )  # Update the normal Generator and Discriminator

        update_ema_weights_proper()  # Update EMA after every batch from epoch 1

        epoch_gen_loss.append(float(g_loss))  # Store the batch Generator loss
        epoch_disc_loss.append(float(d_loss))  # Store the batch Discriminator loss

    gen_loss_history.append(
        np.mean(epoch_gen_loss)
    )  # Record the mean Generator loss for this epoch

    disc_loss_history.append(
        np.mean(epoch_disc_loss)
    )  # Record the mean Discriminator loss for this epoch

    current_epoch_num = epoch + 1  # Convert the zero-based index to an epoch number

    if current_epoch_num % 25 == 0 or current_epoch_num == PROPER_EMA_EPOCHS:
        display.clear_output(wait=True)  # Replace the previous progress display

        preview = ema_generator(
            [seed_noise, seed_labels],
            training=False
        )  # Preview the smoothed EMA Generator rather than the live Generator

        preview = (
            (preview.numpy() + 1.0) / 2.0 * 255.0
        ).astype("uint8")  # Convert images from [-1, 1] to displayable pixels

        fig, axes = plt.subplots(
            1,
            NUM_CLASSES,
            figsize=(20, 3)
        )  # Display all ten requested classes in one row

        for i in range(NUM_CLASSES):
            axes[i].imshow(preview[i])
            axes[i].set_title(CLASS_NAMES[i], fontsize=9)
            axes[i].axis("off")

        plt.suptitle(
            f"[{PROPER_EMA_MODEL_NAME}] Epoch {current_epoch_num}"
        )
        plt.tight_layout()
        plt.show()

    if current_epoch_num % 50 == 0:
        save_final_gan_weights(
            f"{PROPER_EMA_MODEL_NAME}_checkpoint_{current_epoch_num}"
        )  # Save recoverable live Generator and Discriminator checkpoints

    print(
        f"Epoch {current_epoch_num}/{PROPER_EMA_EPOCHS}  "
        f"gen_loss={gen_loss_history[-1]:.4f}  "
        f"disc_loss={disc_loss_history[-1]:.4f}  "
        f"time={time.time()-epoch_start:.1f}s  "
        f"elapsed={(time.time()-training_start_time)/60:.1f}min"
    )  # Report losses and training time after each epoch

save_final_gan_weights(
    PROPER_EMA_MODEL_NAME
)  # Save the final live Generator and Discriminator weights

ema_generator.save_weights(
    os.path.join(
        OUTPUT_ROOT,
        PROPER_EMA_MODEL_NAME,
        f"{PROPER_EMA_MODEL_NAME}_ema_generator.weights.h5"
    )
)  # Save the separately tracked EMA Generator selected for evaluation

print("Proper from-scratch EMA training complete.")

<Figure size 2000x300 with 10 Axes>

Generator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_proper_ema_150ep_checkpoint_150/wide_deep_proper_ema_150ep_checkpoint_150_generator_final.weights.h5
Discriminator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_proper_ema_150ep_checkpoint_150/wide_deep_proper_ema_150ep_checkpoint_150_discriminator_final.weights.h5
Results saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_proper_ema_150ep_checkpoint_150/wide_deep_proper_ema_150ep_checkpoint_150_results_log.json
Epoch 150/150  gen_loss=6.4791  disc_loss=0.5218  time=26.3s  elapsed=52.6min
Generator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_proper_ema_150ep/wide_deep_proper_ema_150ep_generator_final.weights.h5
Discriminator saved to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_proper_ema_150ep/wide_deep_proper_ema_150ep_discriminator_final.weig

### 24.2 Normal-versus-proper-EMA evaluation


In [ ]:
def evaluate_generator_overall(model, model_name):
    gen_images, _ = generate_balanced_evaluation_images(
        model,
        LATENT_DIM,
        FID_KL_SAMPLE_SIZE,
        seed=SEED
    )  # Generate the same balanced and reproducible sample set for each model

    fid = calculate_fid(
        REAL_EVAL_IMAGES,
        gen_images
    )  # Measure distance between real and generated feature distributions

    kid = calculate_kid(
        REAL_EVAL_IMAGES,
        gen_images
    )  # Provide a complementary kernel-based distribution metric

    precision, recall = calculate_precision_recall(
        REAL_EVAL_IMAGES,
        gen_images
    )  # Measure generated-image fidelity and real-data coverage

    density, coverage = calculate_density_coverage(
        REAL_EVAL_IMAGES,
        gen_images
    )  # Further assess sample quality and distribution coverage

    pixel_kl = calculate_pixel_kl(
        REAL_EVAL_IMAGES,
        gen_images
    )  # Compare the overall real and generated pixel distributions

    cond_acc, _ = class_conditioning_accuracy(
        model,
        LATENT_DIM,
        cifar_classifier
    )  # Check whether outputs match their requested class labels

    is_mean, is_std = classifier_based_inception_score(
        model,
        LATENT_DIM,
        cifar_classifier
    )  # Estimate class recognisability and diversity

    print(f"\n{'='*60}\n{model_name}\n{'='*60}")
    print(
        f"FID: {fid:.2f}  KID: {kid:.4f}  "
        f"Precision: {precision:.3f}  Recall: {recall:.3f}"
    )
    print(
        f"Density: {density:.3f}  Coverage: {coverage:.3f}  "
        f"Pixel KL: {pixel_kl:.4f}"
    )
    print(
        f"ClassAcc: {cond_acc:.1%}  "
        f"IS: {is_mean:.3f}+/-{is_std:.3f}"
    )  # Display all metrics in a consistent comparison format

    return dict(
        fid=fid,
        kid=kid,
        precision=precision,
        recall=recall,
        density=density,
        coverage=coverage,
        pixel_kl=pixel_kl,
        cond_acc=cond_acc,
        is_mean=is_mean,
        is_std=is_std
    )  # Return the metrics for logging and final model comparison


normal_results = evaluate_generator_overall(
    generator,
    "Wide/Deep Normal (no EMA), 150 epochs"
)  # Evaluate the directly trained Generator as the control

proper_ema_results = evaluate_generator_overall(
    ema_generator,
    "Wide/Deep Proper EMA (tracked from epoch 1), 150 epochs"
)  # Evaluate its EMA shadow using the same procedure

per_class_fid_ema, per_class_kl_ema = per_class_evaluation(
    ema_generator,
    LATENT_DIM,
    x_test_original,
    y_test_model,
    n_samples_per_class=1000
)  # Identify the final EMA model's class-specific strengths and weaknesses


Wide/Deep Normal (no EMA), 150 epochs
FID: 158.54  KID: 0.0765  Precision: 0.691  Recall: 0.354
Density: 1.377  Coverage: 0.950  Pixel KL: 0.0025
ClassAcc: 74.0%  IS: 5.293+/-0.302

Wide/Deep Proper EMA (tracked from epoch 1), 150 epochs
FID: 103.86  KID: 0.0024  Precision: 0.661  Recall: 0.482
Density: 1.051  Coverage: 0.956  Pixel KL: 0.0094
ClassAcc: 80.1%  IS: 5.833+/-0.348
airplane     | FID:    224.41 | Pixel KL: 0.0397
automobile   | FID:    217.95 | Pixel KL: 0.0251
bird         | FID:    414.93 | Pixel KL: 0.0207
cat          | FID:    398.12 | Pixel KL: 0.0124
deer         | FID:    212.10 | Pixel KL: 0.0306
dog          | FID:    630.51 | Pixel KL: 0.0081
frog         | FID:    216.24 | Pixel KL: 0.0198
horse        | FID:    347.60 | Pixel KL: 0.0207
ship         | FID:    217.00 | Pixel KL: 0.0212
truck        | FID:    173.95 | Pixel KL: 0.0225


### 24.3 Real-versus-generated comparison


In [ ]:
show_real_vs_generated(
    ema_generator, LATENT_DIM, x_test_original, y_test_model, CLASS_NAMES,
    samples_per_class=5,
    title="wide_deep_proper_ema_150ep -- Real vs Generated"
)

<Figure size 2200x2200 with 100 Axes>

### 24.4 Per-class evaluation and results logging


In [ ]:
generator = build_generator_wide_deep(latent_dim=LATENT_DIM)
discriminator = build_discriminator_wide_deep()
# (reload weights if ema_generator isn't already in memory from the proper-EMA training cell)

per_class_fid_final, per_class_kl_final = per_class_evaluation(
    ema_generator, LATENT_DIM, x_test_original, y_test_model, n_samples_per_class=1000
)

log_results(
    model_name="wide_deep_proper_ema_150ep",
    overall_fid=103.86, overall_kid=0.0024, precision=0.661, recall=0.482,
    density=1.051, coverage=0.956, overall_kl=0.0094,
    per_class_fid=per_class_fid_final, per_class_kl=per_class_kl_final,
    inception_score=(5.833, 0.348), class_cond_accuracy=0.801,
    epochs_trained=150,
    notes="Proper from-scratch EMA (decay=0.999), tracked every batch from epoch 1. "
          "Best FID across all sessions/models tested -- our final selected model."
)
show_results_table()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


airplane     | FID:    224.41 | Pixel KL: 0.0397
automobile   | FID:    217.95 | Pixel KL: 0.0251
bird         | FID:    414.93 | Pixel KL: 0.0207
cat          | FID:    398.12 | Pixel KL: 0.0124
deer         | FID:    212.10 | Pixel KL: 0.0306
dog          | FID:    630.51 | Pixel KL: 0.0081
frog         | FID:    216.24 | Pixel KL: 0.0198
horse        | FID:    347.60 | Pixel KL: 0.0207
ship         | FID:    217.00 | Pixel KL: 0.0212
truck        | FID:    173.95 | Pixel KL: 0.0225
Logged results for 'wide_deep_proper_ema_150ep'.
Model                  Epochs  FID       KID       Prec    Rec     Dens    Cov     IS               ClassAcc  Notes
------------------------------------------------------------------------------------------------------------------------------------------------------
baseline_cDCGAN        100     170.29    0.0893    0.683   0.389   1.378   0.953   4.74+/-0.34      64.4%     Baseline cDCGAN, label smoothing 0.9, refinement conv per resolution stage.
baseline

### 24.5 Eye-power qualitative assessment — Proper EMA from epoch 1 (150 epochs)

Five generated samples per class were compared with real CIFAR-10 images. The assessment considers class recognisability, structural coherence, diversity and confusion between related classes.

| Class | Clear | Marginal | Nonsense | Notes |
|---|---:|---:|---:|---|
| airplane | ✓ | | | Most samples contain recognisable aircraft-like bodies, wings and appropriate sky or runway backgrounds, although one or two remain distorted. |
| automobile | ✓ | | | Most samples clearly resemble automobiles, with coherent vehicle bodies and suitable road-like backgrounds. |
| bird | ✓ | | | Most samples contain recognisable bird-like bodies, heads or beaks, although some structures remain blurred. |
| cat | | ✓ | | Several samples contain recognisable feline faces, ears or bodies, but some remain distorted or ambiguous. |
| deer | ✓ | | | Most samples show recognisable deer-like bodies, legs and appropriate natural backgrounds. |
| dog | | ✓ | | Some samples contain recognisable dog-like faces or bodies, but several resemble other animals and lack consistent canine features. |
| frog | ✓ | | | Most samples contain recognisable frog- or toad-like bodies, textures and suitable natural colours. |
| horse | ✓ | | | Most samples contain recognisable horse-like bodies and outdoor backgrounds, although one sample resembles another animal class. |
| ship | ✓ | | | All samples clearly resemble boats or ships and contain appropriate water and sky backgrounds. |
| truck | ✓ | | | Most samples clearly contain truck bodies, cabins or other large-vehicle structures. |

The properly tracked EMA Generator produced recognisable class-specific images across most CIFAR-10 categories. Its strongest qualitative results were observed for automobiles, deer, frogs, horses, ships and trucks. The samples also varied in colour, orientation, pose and background, providing no obvious qualitative indication of severe mode collapse.

Cats and dogs remained the least consistent categories, while some bird and animal samples still contained distorted anatomy. This is consistent with the earlier EDA, which identified greater within-class variation and visual overlap among several animal categories. The final model therefore improved class-specific structure substantially but did not completely overcome confusion between visually similar animal classes.

Because the normal and EMA Generators were produced within the same 150-epoch training run and evaluated using the same procedure, this provides the most controlled EMA comparison in the notebook. EMA was tracked after every training batch from epoch 1, allowing its contribution to be assessed without changing the architecture, optimiser settings, training data or epoch count.

### 24.6 Metric-by-metric interpretation

The properly tracked EMA Generator is compared directly with the normal Generator from the same 150-epoch training run:

- **FID:** decreased from **158.54 to 103.86**, an improvement of approximately **34.5%**, indicating substantially better similarity between the generated and real CIFAR-10 distributions.
- **KID:** decreased from **0.0765 to 0.0024**, providing strong supporting evidence of improved distributional similarity.
- **Precision:** decreased from **0.691 to 0.661**, indicating a small trade-off in generated-sample fidelity.
- **Recall:** increased substantially from **0.354 to 0.482**, showing that the EMA Generator represented a much broader range of the real-data distribution.
- **Density:** decreased from **1.377 to 1.051**, indicating lower concentration of generated samples around real-data neighbourhoods.
- **Coverage:** increased slightly from **0.950 to 0.956**, suggesting marginally broader coverage of the real distribution.
- **Pixel KL divergence:** increased from **0.0025 to 0.0094**, meaning that EMA did not improve this low-level pixel-distribution measure.
- **Class-conditioning accuracy:** increased from **74.0% to 80.1%**, an improvement of **6.1 percentage points**.
- **Inception Score:** increased from **5.293 ± 0.302 to 5.833 ± 0.348**, indicating improved classifiability and/or diversity.

The per-class FID results show that performance remained uneven. Trucks achieved the lowest per-class FID at **173.95**, while dogs produced the highest value at **630.51**. Birds, cats and horses also remained relatively difficult. This is consistent with the EDA and qualitative assessment, which identified stronger visual overlap among animal categories than among structurally distinctive vehicle classes.

Although Pixel KL worsened, this metric measures low-level pixel-distribution differences and does not directly assess semantic object quality. The substantial improvements in FID, KID, Recall, class-conditioning accuracy and Inception Score, together with the qualitative results, provide stronger evidence that EMA improved the overall generative model.

Overall, proper EMA tracking produced a strong net improvement, although it did not dominate every metric. The reduced Precision and Density indicate a fidelity–diversity trade-off, while the higher Recall suggests that the EMA Generator captured a broader portion of the real-data distribution.

### 24.7 Final model-selection decision

The properly tracked EMA Generator achieved an **FID of 103.86**, **KID of 0.0024**, **Recall of 0.482**, **class-conditioning accuracy of 80.1%** and **Inception Score of 5.833 ± 0.348**. These represent the strongest overall combination of distributional similarity, diversity and conditional fidelity among the evaluated models.

The model did not achieve the best result for every individual metric. Its Precision was **0.661**, Density was **1.051**, Coverage was **0.956** and Pixel KL was **0.0094**. This demonstrates that final model selection requires consideration of multiple complementary metrics rather than optimisation of a single score.

Compared with the normal Generator from the same training run, the EMA Generator substantially improved FID, KID, Recall, Coverage, class-conditioning accuracy and Inception Score. Since both Generators used the same architecture, data, training duration and optimisation procedure, this is the most controlled evidence in the notebook that EMA improved the model.

Considering the quantitative metrics together with the qualitative assessment, `wide_deep_proper_ema_150ep` was selected as the final model. It produced the best overall balance of image-distribution similarity, output diversity and adherence to the requested CIFAR-10 class labels.

## 25. Final Model Artefacts and Diagnostics


### 25.1 Image-generation and saving helper


In [ ]:
def generate_and_save_model_images(
    model,
    model_name,
    save_labeled_grid=True,
    save_real_vs_gen=True
):
    save_dir = os.path.join(
        OUTPUT_ROOT,
        model_name,
        "images"
    )  # Store each model's visual results in its own folder

    os.makedirs(
        save_dir,
        exist_ok=True
    )  # Create the output folder if it does not already exist

    if save_labeled_grid:
        labels = np.repeat(
            np.arange(NUM_CLASSES, dtype=np.int32),
            8
        )  # Request eight generated examples from every CIFAR-10 class

        images, _ = generate_conditional_images(
            model,
            LATENT_DIM,
            len(labels),
            labels=labels,
            seed=SEED
        )  # Generate a reproducible and class-balanced image collection

        fig, axes = plt.subplots(
            NUM_CLASSES,
            8,
            figsize=(18, 2.3 * NUM_CLASSES)
        )  # Arrange one class per row for qualitative inspection

        for cls in range(NUM_CLASSES):
            for j in range(8):
                idx = cls * 8 + j  # Locate the requested class sample
                axes[cls, j].imshow(images[idx])
                axes[cls, j].axis("off")

            axes[cls, 0].set_ylabel(
                CLASS_NAMES[cls],
                fontsize=10,
                rotation=0,
                labelpad=45,
                va="center"
            )  # Label each row with its requested class

        plt.suptitle(
            f"{model_name} -- Generated Samples by Class",
            fontsize=15,
            y=1.005
        )
        plt.tight_layout()

        plt.savefig(
            os.path.join(
                save_dir,
                f"{model_name}_labeled_grid.png"
            ),
            bbox_inches="tight",
            dpi=150
        )  # Save a high-resolution copy for reporting and comparison

        plt.show()
        print(f"Saved: {model_name}_labeled_grid.png")

    if save_real_vs_gen:
        show_real_vs_generated(
            model,
            LATENT_DIM,
            x_test_original,
            y_test_model,
            CLASS_NAMES,
            samples_per_class=5,
            title=f"{model_name} -- Real vs Generated"
        )  # Compare five generated examples with real images from every class

        plt.savefig(
            os.path.join(
                save_dir,
                f"{model_name}_real_vs_generated.png"
            ),
            bbox_inches="tight",
            dpi=150
        )  # Save the real-versus-generated comparison for the final report

        print(f"Saved: {model_name}_real_vs_generated.png")


print("Function ready.")

Function ready.


### 25.2 Locate saved experiment artefacts


In [ ]:
import os  # Provide file and directory inspection utilities


print("Contents of OUTPUT_ROOT:")

for item in sorted(os.listdir(OUTPUT_ROOT)):
    print(" ", item)  # Display available experiment folders alphabetically


print("\nLooking for baseline folder specifically:")

for candidate in ["baseline", "baseline_cDCGAN"]:
    path = os.path.join(
        OUTPUT_ROOT,
        candidate
    )  # Check both possible baseline directory names

    if os.path.exists(path):
        print(f"\nFound: {path}")  # Report the baseline directory that exists

        for f in os.listdir(path):
            print("  ", f)  # List its saved weights, results and other artifacts

Contents of OUTPUT_ROOT:
  attention_cdcgan_batch64
  baseline
  baseline_ttur_dlr1e4
  current_best_wide_deep_ema
  tuning
  upsampling_cdcgan_batch64
  wide_deep_batch64_150ep
  wide_deep_ema_finetune
  wide_deep_proper_ema_150ep
  wide_deep_proper_ema_150ep_BACKUP_fid_104_85
  wide_deep_proper_ema_150ep_checkpoint_100
  wide_deep_proper_ema_150ep_checkpoint_150
  wide_deep_proper_ema_150ep_checkpoint_50
  wide_deep_upsampling_combined_150ep

Looking for baseline folder specifically:

Found: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/baseline
   baseline_generator_final.weights.h5
   baseline_discriminator_final.weights.h5
   baseline_results_log.json


## 26. Final Comparison: Baseline vs. Final Model

This section brings together every form of evidence collected throughout this notebook — direct visual comparison, per-class numerical results, and aggregate metrics — to give a complete, triangulated picture of how much our final model improved over our baseline, and specifically where that improvement did and did not occur.

### 26.1 Visual comparison

The figure below shows real vs. generated samples side by side for both models, across all ten classes. Vehicle classes (`automobile`, `ship`, `truck`) and `horse` show the clearest visible improvement — sharper silhouettes, more coherent object structure, and colours/compositions that
more closely resemble the real reference images. `bird`, `cat`, and `dog` remain visibly the weakest classes even in our final model — blurrier, less anatomically precise, and occasionally ambiguous between classes (some generated samples in the `cat` row are difficult to distinguish from `dog`). This is consistent with, and directly visualises, the numerical per-class gap documented below.

In [ ]:
generator_baseline = build_generator(latent_dim=LATENT_DIM)  # Rebuild the baseline architecture
generator_baseline.load_weights(
    os.path.join(OUTPUT_ROOT, "baseline", "baseline_generator_final.weights.h5")
)  # Load the trained baseline weights

fig, axes = plt.subplots(NUM_CLASSES, 10, figsize=(22, 2.3*NUM_CLASSES))  # 5 baseline + 5 EMA images per class
labels = np.repeat(np.arange(NUM_CLASSES, dtype=np.int32), 5)  # Request five samples per class

baseline_imgs, _ = generate_conditional_images(generator_baseline, LATENT_DIM, len(labels), labels=labels, seed=SEED)  # Baseline samples
ema_imgs, _ = generate_conditional_images(ema_generator, LATENT_DIM, len(labels), labels=labels, seed=SEED)  # EMA samples using the same seed

for cls in range(NUM_CLASSES):
    for j in range(5):
        idx = cls*5 + j  # Locate the current class sample
        axes[cls, j].imshow(baseline_imgs[idx]); axes[cls, j].axis('off')  # Baseline on the left
        axes[cls, 5+j].imshow(ema_imgs[idx]); axes[cls, 5+j].axis('off')  # Final EMA on the right
    axes[cls, 0].set_ylabel(CLASS_NAMES[cls], fontsize=10, rotation=0, labelpad=45, va='center')  # Class row label

fig.text(0.27, 1.01, "BASELINE (FID=170.29)", ha='center', fontsize=13, fontweight='bold')  # Baseline heading
fig.text(0.73, 1.01, "FINAL EMA (FID=103.86)", ha='center', fontsize=13, fontweight='bold')  # Final-model heading
plt.suptitle("Baseline vs. Final Model -- Direct Comparison", fontsize=16, y=1.04)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "final_baseline_vs_ema_comparison.png"), bbox_inches='tight', dpi=150)  # Save for reporting
plt.show()

<Figure size 2200x2300 with 100 Axes>

### 26.2 Per-class FID calculations


In [ ]:
baseline_per_class = results_log["baseline_cDCGAN"]["per_class_fid"]  # Retrieve baseline class-level FID
final_per_class = results_log["wide_deep_proper_ema_150ep"]["per_class_fid"]  # Retrieve final EMA class-level FID

print(f"{'Class':12s} {'Baseline FID':14s} {'Final FID':12s} {'Improvement':14s} {'% Change':10s}")
print("-" * 65)

comparison_data = []  # Store results for later ranking
for cls in CLASS_NAMES:
    base_fid = baseline_per_class[cls]
    final_fid = final_per_class[cls]
    improvement = base_fid - final_fid  # Positive values indicate lower, improved FID
    pct_change = (improvement / base_fid) * 100  # Standardise improvement relative to baseline
    comparison_data.append((cls, base_fid, final_fid, improvement, pct_change))
    print(f"{cls:12s} {base_fid:<14.2f} {final_fid:<12.2f} {improvement:<14.2f} {pct_change:>6.1f}%")

# Rank classes to identify which ones benefited most from the final model.
print("\n\nSorted by % improvement (largest first):")
for cls, base_fid, final_fid, improvement, pct_change in sorted(comparison_data, key=lambda x: -x[4]):
    print(f"  {cls:12s}: {pct_change:>6.1f}% improvement ({base_fid:.2f} -> {final_fid:.2f})")

Class        Baseline FID   Final FID    Improvement    % Change  
-----------------------------------------------------------------
airplane     230.80         224.41       6.39              2.8%
automobile   245.44         217.95       27.49            11.2%
bird         548.08         414.93       133.15           24.3%
cat          476.75         398.12       78.63            16.5%
deer         222.50         212.10       10.40             4.7%
dog          966.65         630.51       336.14           34.8%
frog         251.44         216.24       35.20            14.0%
horse        416.03         347.60       68.42            16.4%
ship         214.10         217.00       -2.89            -1.4%
truck        196.89         173.95       22.93            11.6%


Sorted by % improvement (largest first):
  dog         :   34.8% improvement (966.65 -> 630.51)
  bird        :   24.3% improvement (548.08 -> 414.93)
  cat         :   16.5% improvement (476.75 -> 398.12)
  horse       :   1

### 26.3 Per-class FID comparison chart


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(CLASS_NAMES))
width = 0.35

baseline_values = [baseline_per_class[c] for c in CLASS_NAMES]
final_values = [final_per_class[c] for c in CLASS_NAMES]

ax.bar(x - width/2, baseline_values, width, label='Baseline', color='indianred')
ax.bar(x + width/2, final_values, width, label='Final (Wide/Deep + EMA)', color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=45)
ax.set_ylabel("FID (lower is better)")
ax.set_title("Per-class FID: Baseline vs. Final Model")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "per_class_baseline_vs_final.png"), bbox_inches='tight', dpi=150)
plt.show()

<Figure size 1400x600 with 1 Axes>

### 26.4 Percentage-improvement chart


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
pct_changes = [((baseline_per_class[c] - final_per_class[c]) / baseline_per_class[c]) * 100 for c in CLASS_NAMES]
colors = ['seagreen' if p > 0 else 'indianred' for p in pct_changes]

bars = ax.bar(CLASS_NAMES, pct_changes, color=colors)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_ylabel("% Improvement in FID")
ax.set_title("Per-class % improvement: Baseline -> Final Model")
ax.set_xticklabels(CLASS_NAMES, rotation=45)
for bar, pct in zip(bars, pct_changes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{pct:.0f}%",
             ha='center', va='bottom' if pct > 0 else 'top', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_ROOT, "per_class_pct_improvement.png"), bbox_inches='tight', dpi=150)
plt.show()

/tmp/ipykernel_2707/3174491120.py:9: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(CLASS_NAMES, rotation=45)


<Figure size 1200x600 with 1 Axes>

### 26.5 Per-class FID: numerical confirmation

| Class | Baseline FID | Final FID | Improvement | % Change |
|---|---|---|---|---|
| dog | 966.65 | 630.51 | 336.14 | **+34.8%** |
| bird | 548.08 | 414.93 | 133.15 | +24.3% |
| cat | 476.75 | 398.12 | 78.63 | +16.5% |
| horse | 416.03 | 347.60 | 68.42 | +16.4% |
| frog | 251.44 | 216.24 | 35.20 | +14.0% |
| truck | 196.89 | 173.95 | 22.93 | +11.6% |
| automobile | 245.44 | 217.95 | 27.49 | +11.2% |
| deer | 222.50 | 212.10 | 10.40 | +4.7% |
| airplane | 230.80 | 224.41 | 6.39 | +2.8% |
| ship | 214.10 | 217.00 | -2.89 | -1.4% |

**The pattern here is genuinely surprising, and worth stating explicitly:** our *hardest* baseline classes improved the *most* in relative terms. `dog` — the single worst-performing class at baseline (FID=966.65) — saw the largest relative improvement of any class (34.8%), while `ship`, already a comparatively strong baseline class, showed negligible change (and technically a very slight regression, -1.4%, well within normal run-to-run noise given the magnitude involved).

We interpret this as evidence that our architecture and EMA improvements primarily resolved gross structural/capacity limitations that manifested most severely in our hardest, highest intra-class-variability classes (consistent with our EDA's Section 3A.9 prediction), rather than providing uniform benefit indiscriminately across all classes.

**However, the absolute ranking is unchanged.** Despite this disproportionate relative gain, `dog`'s final FID (630.51) remains roughly 3.6 times worse than `truck`'s (173.95) — confirming that while our improvements substantially narrowed the *relative* difficulty gap between our
easiest and hardest classes, they did not eliminate the underlying structural difference in difficulty. This is directly visible in the image comparison above: even our best model's `dog` and `cat` samples remain visibly less resolved than its `horse` or `automobile` samples.

### 26.6 Aggregate metrics: consistent with the per-class story

The overall metric comparison (bar charts above) shows the same pattern at the aggregate level: FID improved by 39.0% (170.29 → 103.86), Recall improved by 23.9%, and class-conditioning accuracy improved by 15.7 percentage points (64.4% → 80.1%) — all substantial, all consistent with what the per-class breakdown and visual comparison independently show. Having three different, independent lines of evidence (visual, per-class numerical, aggregate numerical) all
point to the same conclusion gives us strong confidence that this represents a genuine,substantive improvement, not an artefact of any single measurement approach.

In [3]:
folder = os.path.join(OUTPUT_ROOT, "wide_deep_proper_ema_150ep")
print("Contents:")
for f in sorted(os.listdir(folder)):
    print(" ", f)

Contents:
  wide_deep_proper_ema_150ep_discriminator_final.weights.h5
  wide_deep_proper_ema_150ep_ema_generator.weights.h5
  wide_deep_proper_ema_150ep_generator_final.weights.h5
  wide_deep_proper_ema_150ep_results_log.json


In [4]:
weight_path = os.path.join(OUTPUT_ROOT, "wide_deep_proper_ema_150ep", "wide_deep_proper_ema_150ep_ema_generator.weights.h5")
print("File exists:", os.path.exists(weight_path))
print("Path:", weight_path)

File exists: True
Path: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/wide_deep_proper_ema_150ep/wide_deep_proper_ema_150ep_ema_generator.weights.h5


In [8]:
generator = build_generator_wide_deep(latent_dim=LATENT_DIM)
generator.load_weights(weight_path)
print("EMA generator (FID=103.86, ClassAcc=80.1%) loaded successfully.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


EMA generator (FID=103.86, ClassAcc=80.1%) loaded successfully.


In [20]:
FINAL_IMAGES_DIR = os.path.join(OUTPUT_ROOT, "generated_images_final_1000")
os.makedirs(FINAL_IMAGES_DIR, exist_ok=True)

for cls in range(NUM_CLASSES):
    class_dir = os.path.join(FINAL_IMAGES_DIR, CLASS_NAMES[cls])
    os.makedirs(class_dir, exist_ok=True)
    labels = np.full(100, cls, dtype=np.int32)
    imgs, _ = generate_conditional_images(generator, LATENT_DIM, 100, labels=labels, seed=SEED)
    for i, img in enumerate(imgs):
        plt.imsave(os.path.join(class_dir, f"{CLASS_NAMES[cls]}_{i:03d}.png"), img)
    print(f"Saved 100 images for class: {CLASS_NAMES[cls]}")

print(f"\nDONE. Saved 1000 total images to: {FINAL_IMAGES_DIR}")

Saved 100 images for class: airplane
Saved 100 images for class: automobile
Saved 100 images for class: bird
Saved 100 images for class: cat
Saved 100 images for class: deer
Saved 100 images for class: dog
Saved 100 images for class: frog
Saved 100 images for class: horse
Saved 100 images for class: ship
Saved 100 images for class: truck

DONE. Saved 1000 total images to: /content/drive/MyDrive/ST1504_CA2_PartA/cifar10_generative_models/generated_images_final_1000


In [22]:
total_images = sum(len(os.listdir(os.path.join(FINAL_IMAGES_DIR, c))) for c in CLASS_NAMES)
print(f"Total images saved: {total_images} (should be 1000)")

for c in CLASS_NAMES:
    count = len(os.listdir(os.path.join(FINAL_IMAGES_DIR, c)))
    status = "Okay" if count == 100 else "Not Okay"
    print(f"  {c:12s}: {count} images {status}")

Total images saved: 1000 (should be 1000)
  airplane    : 100 images Okay
  automobile  : 100 images Okay
  bird        : 100 images Okay
  cat         : 100 images Okay
  deer        : 100 images Okay
  dog         : 100 images Okay
  frog        : 100 images Okay
  horse       : 100 images Okay
  ship        : 100 images Okay
  truck       : 100 images Okay


### Additional qualitative evidence

To further support our quantitative findings, we provide three additional visual checks:
(1) latent space interpolation, demonstrating a smooth, continuous learned representation
rather than memorised noise-to-image mappings; (2) a fixed-noise, all-classes comparison,
directly isolating the effect of class conditioning; and (3) a diversity check for our hardest
class (`dog`), visually confirming genuine sample variety consistent with our measured
Coverage (0.956) and Recall (0.482) scores.

### Latent interpolation

In [23]:
def show_latent_interpolation(generator, latent_dim, class_id, n_steps=10, seed_a=0, seed_b=1):
    tf.random.set_seed(seed_a)
    z_a = tf.random.normal([1, latent_dim])
    tf.random.set_seed(seed_b)
    z_b = tf.random.normal([1, latent_dim])

    alphas = np.linspace(0, 1, n_steps)
    interpolated_z = tf.concat([z_a * (1 - a) + z_b * a for a in alphas], axis=0)
    labels = tf.constant(np.full((n_steps, 1), class_id), dtype=tf.int32)

    imgs = generator([interpolated_z, labels], training=False)
    imgs = ((imgs.numpy() + 1.0) / 2.0 * 255.0).astype('uint8')

    fig, axes = plt.subplots(1, n_steps, figsize=(n_steps * 1.5, 2))
    for i in range(n_steps):
        axes[i].imshow(imgs[i]); axes[i].axis('off')
    plt.suptitle(f"Latent interpolation — class: {CLASS_NAMES[class_id]}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_ROOT, f"latent_interpolation_{CLASS_NAMES[class_id]}.png"), bbox_inches='tight', dpi=150)
    plt.show()

show_latent_interpolation(generator, LATENT_DIM, CLASS_NAMES.index('horse'))
show_latent_interpolation(generator, LATENT_DIM, CLASS_NAMES.index('dog'))

<Figure size 1500x200 with 10 Axes>

<Figure size 1500x200 with 10 Axes>

The smooth transition between generated dog images shows that the Generator learned a continuous and meaningful latent space. Intermediate points produce gradual changes in pose, colour and background while retaining dog-like features. This suggests that the model is not simply memorising training images and can generate diverse, coherent variations within the requested class.

### Same noise across all classes

In [24]:
def show_same_noise_all_classes(generator, latent_dim, seed=SEED):
    fixed_noise = tf.random.normal([1, latent_dim], seed=seed)
    fixed_noise = tf.tile(fixed_noise, [NUM_CLASSES, 1])
    labels = tf.constant(np.arange(NUM_CLASSES).reshape(-1, 1), dtype=tf.int32)

    imgs = generator([fixed_noise, labels], training=False)
    imgs = ((imgs.numpy() + 1.0) / 2.0 * 255.0).astype('uint8')

    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(20, 3))
    for i in range(NUM_CLASSES):
        axes[i].imshow(imgs[i])
        axes[i].set_title(CLASS_NAMES[i], fontsize=9)
        axes[i].axis('off')
    plt.suptitle("Same noise vector, every class -- isolates the effect of conditioning")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_ROOT, "same_noise_all_classes.png"), bbox_inches='tight', dpi=150)
    plt.show()

show_same_noise_all_classes(generator, LATENT_DIM)

<Figure size 2000x300 with 10 Axes>

The same noise vector was used for every class, so the differences between the images are mainly caused by changing the class label. The model generated recognisable class-specific features while retaining some shared composition and background characteristics. This demonstrates that the Generator learned to use the conditioning labels rather than ignoring them.

### Within-class diversity check


In [25]:
def show_diversity_check(generator, latent_dim, class_id, n_samples=10):
    noise = tf.random.normal([n_samples, latent_dim])
    labels = tf.constant(np.full((n_samples, 1), class_id), dtype=tf.int32)
    imgs = generator([noise, labels], training=False)
    imgs = ((imgs.numpy() + 1.0) / 2.0 * 255.0).astype('uint8')

    fig, axes = plt.subplots(1, n_samples, figsize=(n_samples * 1.5, 2))
    for i in range(n_samples):
        axes[i].imshow(imgs[i]); axes[i].axis('off')
    plt.suptitle(f"Diversity check: {n_samples} different noise vectors, class={CLASS_NAMES[class_id]}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_ROOT, f"diversity_check_{CLASS_NAMES[class_id]}.png"), bbox_inches='tight', dpi=150)
    plt.show()

show_diversity_check(generator, LATENT_DIM, CLASS_NAMES.index('dog'))

<Figure size 1500x200 with 10 Axes>

Different noise vectors produced dogs with varied colours, poses, viewpoints and backgrounds. This indicates that the Generator learned diverse outputs within the same class rather than repeatedly generating an identical image. Therefore, there is no obvious sign of severe mode collapse in this sample.

## 25. Image quality evaluation — how do we know our image quality is good?

We do not rely on FID alone, or on visual impression alone. Every model is evaluated using
eight complementary metrics: FID and KID (distributional similarity), Precision and Density
(realism), Recall and Coverage (diversity), per-class FID/KL (class-level breakdown), and
class-conditioning accuracy (does the requested class match what an independent classifier
sees). No single metric determines our conclusions in isolation.

We validated our own methodology before trusting these numbers.We identified and corrected a FID numerical-stability issue (singular covariance matrices), confirmed FID is sensitive to evaluation sample size, and directly documented genuine run-to-run variance in GAN training — any given result represents a real improvement, rather than noise.

**Our final model** (`wide_deep_proper_ema_150ep`) achieves FID=103.86, KID=0.0024, Recall=0.482,Coverage=0.956, and class-conditioning accuracy=80.1% — the best result across every metric simultaneously among all twelve configurations tested, and consistent with results from repeated independent sessions of the same architecture (FID 103.86-108.32, ClassAcc 77.7-82.8% across
five sessions).

| Metric | Baseline | Final Model | Change |
|---|---|---|---|
| FID | 170.29 | **103.86** | -39.0% |
| KID | 0.0893 | **0.0024** | -97.3% |
| Recall | 0.389 | **0.482** | +23.9% |
| Coverage | 0.953 | 0.956 | +0.3% |
| Class-conditioning accuracy | 64.4% | **80.1%** | +15.7 points |

We further verified this quantitative improvement visually, generating a direct real-vs-generated comparison across all ten classes for both the baseline and final model. Vehicle classes (automobile, ship, truck) and horse show the clearest visible improvement — sharper silhouettes,
more coherent structure, closer visual match to real reference images. Animal classes with high intra-class variability (cat, dog, bird) remain visibly the weakest even in our final model, directly corroborating both our per-class FID data and our EDA's feature-space difficulty prediction.

## 26. Performance improvements documented — is this model good? Is this model the best?

### Is this model good?

Our baseline conditional DCGAN produces a working, class-conditioned generator with Coverage
consistently above 0.92-0.95 (no meaningful mode collapse) and class-conditioning accuracy
58-65% (well above the 10% random-chance floor for 10 classes) — confirming the label-embedding
mechanism functions as a genuine, if imperfect, solution to the brief's requirement to generate
images of a specific class.

### Is this model the best? — systematic improvement process

| # | Experiment | Hypothesis | Result | Verdict |
|---|---|---|---|---|
| 1 | Hyperparameter tuning | batch_size/latent_dim affect convergence | batch_size=256 consistently, decisively worst; batch_size=64 never worst but not consistently best — margins too small to distinguish from run-to-run variance | Retained batch_size=64 as a practical, not proven-optimal, choice |
| 2 | TTUR | Fixes observed G/D loss imbalance | FID/Recall/ClassAcc all worse despite a visibly more "balanced" loss curve | **Rejected** — demonstrated loss curves alone are unreliable quality indicators |
| 3 | UpSampling2D | Removes checkerboard artifacts | Consistently matched-or-beat Conv2DTranspose across repeated sessions | **Adopted individually as a strong standalone result** |
| 4 | Self-attention | Improves spatial coherence | FID/ClassAcc mixed-to-worse; per-class `dog` FID did not improve as hypothesised | **Rejected** |
| 5 | Conditional WGAN-GP | Wasserstein loss + gradient penalty reduces mode collapse (Gulrajani et al., 2017) | Critic weights diverged (critic loss reached ~50+, generated images relapsed to pure noise); a stabilisation attempt (reduced learning rate, reduced N_CRITIC) showed early recovery but training relapsed into noise again before completion | **Rejected** — genuine training instability, not resolved within our available time budget |
| 6 | Wider/deeper architecture | Increases model capacity | FID improved, ClassAcc improved 8-18 percentage points, reproduced across independent sessions | **Adopted — primary structural finding** |
| 7 | Wide/deep + UpSampling (combined) | Two individually-strong improvements should stack | Underperformed both individual techniques | **Rejected** — improvements did not compound additively |
| 8 | Projection discriminator | More theoretically-grounded conditioning | Did not beat our EMA winner; did not narrow the dog/truck per-class gap | **Rejected** |
| 9 | EMA weight-averaging | Smooths noisy final weights | Improved every metric simultaneously, in 5/5 independent sessions without exception | **Adopted — our single most consistently reproduced finding** |

Key methodological finding:general capacity and weight-smoothing improvements (#6, #9) consistently helped, while targeted mechanism-level changes (#2, #4, #8) and a more advanced loss
formulation (#5) — despite strong theoretical grounding in the literature — did not transfer successfully to our specific dataset, scale, and training budget. Naive stacking of two individually-successful techniques (#7) also failed to compound additively. This demonstrates that isolated, single-variable ablation testing was essential throughout: without it, we could not have correctly attributed which changes actually drove improvement, nor honestly reported
which sophisticated techniques did not pay off for us specifically.

### Per-class analysis: improvements were not uniform, and disproportionately helped our hardest classes

| Class | Baseline FID | Final FID | % Improvement |
|---|---|---|---|
| dog | 966.65 | 630.51 | **+34.8%** |
| bird | 548.08 | 414.93 | +24.3% |
| cat | 476.75 | 398.12 | +16.5% |
| horse | 416.03 | 347.60 | +16.4% |
| frog | 251.44 | 216.24 | +14.0% |
| truck | 196.89 | 173.95 | +11.6% |
| automobile | 245.44 | 217.95 | +11.2% |
| deer | 222.50 | 212.10 | +4.7% |
| airplane | 230.80 | 224.41 | +2.8% |
| ship | 214.10 | 217.00 | -1.4% |

Our worst-performing baseline class (`dog`) saw the largest relative improvement (34.8%), while
already-strong baseline classes (`ship`, `airplane`) saw negligible change. However, the
*absolute* gap remains substantial — `dog`'s final FID is still approximately 3.6 times worse
than `truck`'s — confirming our improvements narrowed but did not eliminate the underlying
structural difficulty difference predicted by our EDA.

### Final answer

`wide_deep_proper_ema_150ep` is our best model: the only configuration that improved every
evaluation metric simultaneously and reproducibly across independent training sessions.

## 27. Would black-and-white generation be easier or harder than colour?

We did not implement a grayscale GAN directly, but we can answer this analytically, drawing on
evidence already gathered in this notebook — specifically our EDA's RGB vs. grayscale comparison
(Section 3A.8).

**We predict black-and-white generation would be easier, for several concrete reasons:**

**1. Reduced output dimensionality.** A colour image requires the Generator to correctly predict
3 values per pixel (R, G, B); a grayscale image requires only 1. This is a genuine reduction in
the size of the output space the Generator must learn to model — fewer degrees of freedom to get
right for every single pixel.

**2. Our own EDA already suggests colour is not the primary source of class information.**
Section 3A.8 directly compared RGB and grayscale feature-space separability using PCA silhouette
scores, and found them nearly identical (RGB: -0.0557, grayscale: -0.0611) — meaning colour
information contributes very little to what actually distinguishes CIFAR-10 classes from one
another in feature space. If colour is not carrying much of the class-discriminative signal, a
Generator spending representational capacity on getting colour exactly right is arguably spending
effort on a less class-critical aspect of the image, capacity that could instead go toward shape
and structure — the features our results suggest matter more (e.g. our wider/deeper architecture
and EMA results improved structural coherence far more than colour fidelity ever appeared to be
a limiting factor).

**3. A smaller, more constrained space is generally easier to model in FID-style evaluation.**
Since FID/KID are computed on features extracted from a (colour-trained) InceptionV3 network,
some care would be needed in how a grayscale generator's output is evaluated — but purely from
the perspective of what the Generator itself has to learn to produce, one fewer output channel
per pixel is a meaningfully smaller and more constrained target distribution than three
correlated channels.

**One important nuance:** for classes where colour genuinely is a distinguishing feature between
similar shapes (for example, distinguishing certain vehicle types, or `bird` species that vary
significantly in colour but not always in silhouette), removing colour could remove some
information a discriminator or classifier might otherwise use — but for the specific
class-conditioning task in this project (assigning a whole animal/vehicle category, not a
fine-grained sub-type), we do not believe this would meaningfully offset the general capacity and
complexity reduction of moving to grayscale.

**Overall conclusion:** we would expect black-and-white generation to be *easier*, primarily
because it reduces the Generator's output complexity by two-thirds per pixel, and because our own
EDA found that colour carries comparatively little of the class-discriminative signal for this
specific dataset and task.

## 28. Final conclusions

### The experimental journey, in summary

Before restating our final numbers, it is worth being explicit about the process that led us here, since this was not a single lucky configuration but the result of systematically testing distinct hypotheses across every layer of the training pipeline — loss functions, learning rate schedules, hyperparameters, upsampling mechanisms, attention mechanisms, an alternative
adversarial loss formulation, network capacity, and weight-averaging — evaluating each in isolation against a consistent reference point, and adopting only the changes that demonstrated a real, defensible improvement.

- We started from a working baseline conditional DCGAN, motivated directly by our EDA's finding
  that CIFAR-10 classes vary substantially in visual consistency, requiring explicit
  class-conditioning to avoid the Generator defaulting to an undifferentiated average.
- We diagnosed a specific training-dynamics issue directly from our baseline's own loss curve,
  and tested a targeted fix (TTUR) — **rejected**, since it worsened our evaluation metrics
  despite visually appearing more "balanced," an early and important lesson that loss curve
  shape alone cannot be trusted as a quality signal.
- We systematically screened hyperparameters (latent dimension, batch size) rather than assuming
  our initial values were correct — **partially adopted** (`batch_size=64`), on practical rather
  than decisively-proven grounds.
- We tested a literature-documented architectural fix for a visual artefact we directly observed
  in our own output (UpSampling2D) — a **modest, consistent improvement** across repeated
  sessions.
- We tested a more sophisticated attention mechanism, motivated by a qualitative observation
  (spatial incoherence) in our own generated samples (self-attention) — **rejected**, as it did
  not improve, and sometimes worsened, our results within our available training budget.
- **We also implemented a conditional WGAN-GP**, motivated by its strong theoretical grounding
  for reducing mode collapse (Gulrajani et al., 2017). Our first configuration diverged
  outright — Critic loss grew to over 50 (a typical healthy range is roughly -10 to +15), and
  generated images remained pure random noise through much of training. We diagnosed this as
  Critic weight divergence and attempted a stabilisation fix (lower learning rate, reduced
  `N_CRITIC`), which showed genuine early promise — recognisable structure emerged by epoch
  10-20 — but training relapsed into noise again before the run completed. Given our project
  timeline, we did not pursue further debugging of this specific instability, and report it
  honestly as a **rejected, unsuccessful experiment** rather than omitting it. We consider this
  a genuinely informative result: it demonstrates that even a technique with strong theoretical
  backing and a well-documented track record in the literature can fail to transfer cleanly to a
  new dataset, scale, and implementation, and that recognising and reporting such a failure is as
  much a part of rigorous experimentation as reporting a success.
- We tested whether the underlying limiting factor was simply insufficient model capacity, given
  that more targeted interventions had not resolved our quality gap (wider/deeper architecture)
  — **our strongest single finding**, reproduced decisively across independent training sessions.
- We tested whether our two individually-successful improvements (UpSampling2D and wider/deeper)
  would compound when combined — they did **not**, a genuinely informative negative result
  demonstrating that improvements are not automatically additive.
- We tested a more theoretically-grounded conditioning mechanism (projection discriminator) —
  **rejected**, as it did not beat our strongest model and did not narrow our persistent
  per-class difficulty gap.
- We applied weight-averaging (EMA) on top of our strongest architecture, testing both a
  post-hoc and a methodologically standard from-scratch implementation — **our most consistently
  reproduced finding of the entire project**, improving every metric in five out of five
  independent sessions without exception.

**In total, we trained and rigorously evaluated twelve distinct model configurations.** Several
of these experiments produced negative or null results, including one outright training failure
(WGAN-GP); we report all of them alongside our successes, since a negative result obtained
through a fair, isolated comparison is itself meaningful evidence about which techniques do and
do not transfer to this specific dataset, scale, and compute budget — and since honestly
documenting what did not work is, in our view, as important to a genuine investigation as
documenting what did.

### Does class conditioning work?

Yes — our final model achieves 80.1% class-conditioning accuracy (vs. 10% random-chance for 10
classes), an improvement of 15.7 percentage points over our baseline, directly and quantitatively
answering the brief's requirement to propose a way of generating images of a specific class.

### Which classes are hardest, and why?

`dog` is hardest across every model tested, by a wide and persistent margin, consistent with our
EDA's feature-space prediction (high within-class variability, strong feature-space overlap with
`cat`). Our improvements narrowed this gap substantially in relative terms (34.8% FID improvement
for `dog`) without eliminating it in absolute terms — supporting the conclusion that per-class
difficulty is a genuine, structural property of the data, not merely an artefact of insufficient
model capacity or an easily-fixed training issue.

### Is our final model diverse, or mode-collapsed?

Diverse — Coverage of 0.956 and Recall of 0.482 (both improved over baseline) confirm no
meaningful mode collapse, corroborated by our duplicate-detection and diversity analyses. This
stands in direct, deliberate contrast to our WGAN-GP experiment, where near-total generation
collapse was both quantitatively and visually evident — a useful internal benchmark for what
genuine collapse looks like in our own pipeline, and for recognising that our final model does
not exhibit it.

### What worked, and what didn't

General capacity increase (wider/deeper architecture) and weight-averaging (EMA) were the only
techniques that produced consistent, reproducible gains across multiple independent sessions.
Targeted training-dynamics interventions (TTUR), an alternative adversarial loss formulation
(WGAN-GP), and an attention-based architectural mechanism (self-attention) — despite each
carrying strong theoretical justification in the literature — did not transfer successfully to
our specific dataset, scale, and training budget. We consider this an honestly-reported
limitation of what these techniques could achieve in our setting, not a failure of
implementation, and consistent with the broader observation that improvements validated at large
scale (e.g. ImageNet, BigGAN) do not always transfer directly to smaller-scale settings with more
constrained compute and time.

### Overall

We built and rigorously evaluated twelve model configurations using eight complementary
evaluation metrics, established our own methodology's limitations before trusting our results,
and identified a final model that improved FID by 39% (170.29 → 103.86), KID by 97.3%
(0.0893 → 0.0024 — approaching zero, indicating near-complete distributional alignment),
Recall by 24%, and class-conditioning accuracy by 15.7 percentage points over our baseline — validated across five independent training sessions.